# Severe AKI dynamic prediction: sectioned runnable workflow

This notebook is the cell-by-cell version of the full Python workflow for **self-supervised ICU trajectory pretraining + discrete-time competing-risk survival modeling** of severe AKI/RRT in MIMIC-IV and eICU.

The notebook keeps the research proposal logic intact:

- adult first ICU stay cohort;
- no severe AKI/RRT by the first 6-hour landmark;
- KDIGO stage 2/3 phenotype using serum creatinine and urine-output tracks;
- 6-hour landmark updating with a 6-hour actionability gap;
- future 12h, 24h, 48h, 72h, and 7d cumulative incidence risk;
- death and ICU discharge as competing events;
- strict MIMIC->eICU and eICU->MIMIC validation;
- pooled-unlabeled self-supervised adaptation as a **separate** experiment, not as strict external validation;
- PSM sensitivity analysis and SHAP visualizations.

Run the notebook from top to bottom in Colab after placing the two PhysioNet ZIP files in `/content/drive/MyDrive` with exactly these names:

```text
/content/drive/MyDrive/mimic-iv-3.1.zip
/content/drive/MyDrive/eicu-collaborative-research-database-2.0.zip
```


## Strict Healthcare Review Assumptions

This adjusted copy treats the workflow as retrospective clinical prediction research plus a blocked deployment-readiness evidence gate. It does **not** turn MIMIC/eICU retrospective performance into bedside authorization. The edits favor temporal validity, conservative AKI phenotyping, explicit uncertainty, and auditability over a larger but clinically weaker cohort or a cherry-picked headline result.

Key strict-mode changes in this notebook:

- baseline creatinine fallback is limited to data available by the first 6-hour landmark instead of using late post-admission values;
- MIMIC in-ICU labs and prescriptions are joined by both `subject_id` and `hadm_id` to reduce cross-admission contamination;
- creatinine >=4.0 mg/dL requires an acute rise over baseline before being treated as severe AKI;
- urine-output KDIGO labeling requires denser observation coverage by default;
- baseline kidney function and baseline-source indicators are carried into model metadata;
- split leakage checks, subgroup performance tables, clustered uncertainty, and clinical audit files are written during reruns.

v4 D-drive/publication additions:

- the notebook auto-discovers `D:/mimic-iv-3.1.zip` and `D:/eicu-collaborative-research-database-2.0.zip`, while still honoring `AKI_MIMIC_ZIP`, `AKI_EICU_ZIP`, and `AKI_OUTPUT_ROOT` environment overrides;
- outputs default to the current AKI project folder on Windows instead of a Colab-only `/content/drive` path;
- deployment evidence templates are now treated as incomplete placeholders until filled with real prospective/governance evidence;
- a transparent `model_selection_and_claims_report` records every candidate metric file and the best external AUROC at the prespecified horizon, so publication claims cannot silently cherry-pick;
- SHAP and PSM outputs remain explicitly exploratory/non-causal, and the deployment gate remains blocked without local prospective shadow validation and governance artifacts.


## 0. Colab setup

This cell mounts Google Drive and installs the packages used by the workflow. It does not download or create any toy data. The workflow reads the credentialed MIMIC-IV and eICU archives directly from Drive.


In [ ]:

# Install runtime dependencies used by the full workflow.
# PyTorch is normally already available in Colab; this cell installs the tabular,
# parquet, progress-bar, and explanation packages used below.
%pip install -q pyarrow tqdm scikit-learn matplotlib shap

# Mount Google Drive in Colab. Outside Colab, this block is skipped gracefully.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")


## 1. Imports and runtime dependencies

Core libraries are loaded once here. The workflow uses local Python, pandas, parquet, scikit-learn, SHAP, and PyTorch. PostgreSQL is not required.


In [ ]:
from __future__ import annotations

import argparse
import gc
import gzip
import hashlib
import json
import logging
import math
import os
import platform
import pickle
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from functools import lru_cache
from importlib import metadata as importlib_metadata
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception as exc:  # pragma: no cover
    raise RuntimeError("pyarrow is required. In Colab run: !pip -q install pyarrow") from exc

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    def tqdm(x, **kwargs):
        return x

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
except Exception as exc:  # pragma: no cover
    raise RuntimeError("PyTorch is required. Colab normally includes it by default.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)


## 2. Required dataset locations

These paths are intentionally fixed to the user-specified Google Drive locations. The rest of the notebook uses these constants through the `Config` object.


In [ ]:
# -----------------------------------------------------------------------------
# Data and output location discovery
# -----------------------------------------------------------------------------
# Environment overrides, when present, always win:
#   AKI_MIMIC_ZIP   -> full path to mimic-iv-3.1.zip
#   AKI_EICU_ZIP    -> full path to eicu-collaborative-research-database-2.0.zip
#   AKI_OUTPUT_ROOT -> directory where notebook outputs should be written
#   AKI_WORK_DIR    -> scratch/work directory
PROJECT_ROOT = Path(os.environ.get('AKI_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
COLAB_DRIVE_ROOT = Path('/content/drive/MyDrive')
WINDOWS_D_ROOT = Path('D:/')


def _dedupe_existing_dirs(paths: Sequence[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for p in paths:
        try:
            pp = Path(p).expanduser()
            key = str(pp).lower()
        except Exception:
            continue
        if key in seen:
            continue
        seen.add(key)
        if pp.exists() and pp.is_dir():
            out.append(pp)
    return out


DATA_SEARCH_ROOTS = _dedupe_existing_dirs([
    WINDOWS_D_ROOT,
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    Path.home() / 'Downloads',
    COLAB_DRIVE_ROOT,
])


def _resolve_zip(env_var: str, filename: str) -> Path:
    env_value = os.environ.get(env_var, '').strip()
    checked: List[Path] = []
    if env_value:
        p = Path(env_value).expanduser()
        checked.append(p)
        if p.exists():
            return p.resolve()
    for root in DATA_SEARCH_ROOTS:
        p = root / filename
        checked.append(p)
        if p.exists():
            return p.resolve()
    # Return the highest-priority candidate so require_files can raise a helpful error.
    return checked[0] if checked else Path(filename)


MIMIC_ZIP = _resolve_zip('AKI_MIMIC_ZIP', 'mimic-iv-3.1.zip')
EICU_ZIP = _resolve_zip('AKI_EICU_ZIP', 'eicu-collaborative-research-database-2.0.zip')

if os.environ.get('AKI_OUTPUT_ROOT'):
    DRIVE_ROOT = Path(os.environ['AKI_OUTPUT_ROOT']).expanduser().resolve()
elif os.name == 'nt':
    DRIVE_ROOT = PROJECT_ROOT
elif COLAB_DRIVE_ROOT.exists():
    DRIVE_ROOT = COLAB_DRIVE_ROOT
else:
    DRIVE_ROOT = PROJECT_ROOT

if os.environ.get('AKI_WORK_DIR'):
    WORK_ROOT = Path(os.environ['AKI_WORK_DIR']).expanduser().resolve()
elif os.name == 'nt':
    WORK_ROOT = PROJECT_ROOT / 'aki_ssl_survival_work'
else:
    WORK_ROOT = Path('/content/aki_ssl_survival_work')

print('AKI project root:', PROJECT_ROOT)
print('Data search roots:', [str(p) for p in DATA_SEARCH_ROOTS])
print('MIMIC ZIP candidate:', MIMIC_ZIP, 'exists=', Path(MIMIC_ZIP).exists())
print('eICU ZIP candidate:', EICU_ZIP, 'exists=', Path(EICU_ZIP).exists())
print('Output root:', DRIVE_ROOT)
print('Work root:', WORK_ROOT)


## 3. Global configuration

`Config` controls the landmark design, grid resolution, model size, optimization settings, SHAP sampling, PSM caliper, output directories, and device selection. The defaults run the full 7-day dynamic survival task with 2-hour trajectory grids, 6-hour landmark updates, and a 6-hour actionability gap.


In [ ]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Config:
    drive_root: str = str(DRIVE_ROOT)
    mimic_zip: str = str(MIMIC_ZIP)
    eicu_zip: str = str(EICU_ZIP)
    out_dir: str = str(DRIVE_ROOT / 'aki_ssl_survival_output_clinical_deployment_gated_v4')
    work_dir: str = str(WORK_ROOT)

    # AKI dynamic landmark design
    grid_hours: int = 2
    landmark_start_hours: int = 6
    landmark_step_hours: int = 6
    action_gap_hours: int = 6
    horizon_hours: int = 168
    history_hours: int = 168
    max_icu_hours: int = 336
    min_los_hours: int = 24
    # Strict phenotype defaults: prefer fewer false AKI labels over a larger noisier cohort.
    urine_min_observed_fraction: float = 0.80
    baseline_lookback_days: int = 365
    baseline_admission_fallback_hours: int = 6
    require_acute_rise_for_scr_4: bool = True
    min_split_events_warn: int = 25
    bootstrap_repeats: int = 100
    bootstrap_max_rows: int = 50_000
    bootstrap_ci_alpha: float = 0.05
    min_subgroup_n: int = 100
    risk_thresholds: Tuple[float, ...] = (0.05, 0.10, 0.20, 0.30)

    # Clinical deployment gate. These do not make the model deployable by themselves;
    # they force the notebook to produce a blocked/pass readiness decision.
    clinical_deployment_gate: bool = True
    target_operating_horizon_hours: int = 72
    deployment_min_sensitivity: float = 0.70
    deployment_max_alert_rate: float = 0.25
    deployment_max_ece: float = 0.05
    deployment_required_min_n: int = 1000
    measurement_bias_auc_warn: float = 0.70
    treatment_ablation_auc_delta_warn: float = 0.03
    baseline_source_calibration_gap_warn: float = 0.05
    primary_claim_horizon_hours: int = 72
    minimum_external_auc_for_strong_claim: float = 0.80
    minimum_events_for_performance_claim: int = 25
    model_selection_metric: str = 'auc'

    # Extraction and modeling
    # Keep chunks conservative for eICU vitals/labs on local CPU/RAM runs.
    chunksize: int = 200_000
    seed: int = 42
    deterministic: bool = True
    n_jobs: int = max(os.cpu_count() or 2, 2)

    # Deep model
    ssl_epochs: int = 8
    finetune_epochs: int = 12
    batch_size: int = 128
    dataloader_workers: int = 0  # 0 is safest in Colab notebooks; increase after the first successful run.
    lr: float = 2e-4
    weight_decay: float = 1e-4
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.15
    mask_ratio: float = 0.25
    early_stop_patience: int = 4
    mixed_precision: bool = True

    # Baseline + SHAP + PSM
    shap_horizon_hours: int = 72
    shap_sample_size: int = 3000
    rf_n_estimators: int = 300
    psm_caliper_sd: float = 0.20
    min_psm_pairs: int = 100

    # Runtime behavior
    overwrite: bool = False
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    @property
    def n_history_steps(self) -> int:
        return int(math.ceil(self.history_hours / self.grid_hours))

    @property
    def n_surv_bins(self) -> int:
        return int(math.ceil(self.horizon_hours / self.landmark_step_hours))

    @property
    def max_grid_steps(self) -> int:
        return int(math.ceil(self.max_icu_hours / self.grid_hours))

    @property
    def horizons(self) -> List[int]:
        hs = [12, 24, 48, 72, 168]
        return [h for h in hs if h <= self.horizon_hours]


CLINICAL_PIPELINE_VERSION = '2026-06-19-d-drive-publication-gated-v4'


## 4. Clinical variable ontology and general helpers

This section defines the harmonized variable list used across MIMIC and eICU: kidney markers, hemodynamics, perfusion, acid-base variables, electrolytes, liver/inflammation markers, treatment exposures, RRT, ventilation, comorbidities, exclusion patterns, and value-range checks. It also defines reproducibility, logging, directory, and file-existence helpers.


In [ ]:
# -----------------------------------------------------------------------------
# Canonical variables and clinical concept mappings
# -----------------------------------------------------------------------------
CONTINUOUS_VARS = [
    'creatinine', 'bun', 'map', 'heart_rate', 'resp_rate', 'spo2', 'temperature',
    'lactate', 'ph', 'pao2', 'paco2', 'sodium', 'potassium', 'chloride',
    'bicarbonate', 'albumin', 'hemoglobin', 'platelets', 'wbc', 'bilirubin',
    'ast', 'alt', 'glucose', 'inr', 'weight_kg'
]
SUM_VARS = ['urine_output', 'fluid_input', 'net_fluid']
BINARY_VARS = ['vasopressor', 'diuretic', 'nephrotoxin', 'antibiotic', 'mechanical_ventilation', 'rrt']
MATRIX_VARS = CONTINUOUS_VARS + SUM_VARS + BINARY_VARS
VAR_INDEX = {v: i for i, v in enumerate(MATRIX_VARS)}
N_VARS = len(MATRIX_VARS)

STATIC_COLS = [
    'age', 'sex_female', 'baseline_scr', 'baseline_source_prior',
    'baseline_source_admission_fallback', 'weight_missing', 'ckd', 'diabetes',
    'chf', 'liver_disease', 'sepsis_dx', 'medical_icu', 'surgical_icu'
]

VALUE_RANGES = {
    'creatinine': (0.05, 30.0),
    'baseline_scr': (0.05, 30.0),
    'bun': (1.0, 350.0),
    'map': (20.0, 200.0),
    'heart_rate': (20.0, 260.0),
    'resp_rate': (2.0, 80.0),
    'spo2': (40.0, 100.0),
    'temperature': (25.0, 45.0),
    'lactate': (0.1, 30.0),
    'ph': (6.5, 8.2),
    'pao2': (10.0, 700.0),
    'paco2': (5.0, 200.0),
    'sodium': (90.0, 190.0),
    'potassium': (1.0, 10.0),
    'chloride': (60.0, 150.0),
    'bicarbonate': (2.0, 60.0),
    'albumin': (0.5, 8.0),
    'hemoglobin': (3.0, 25.0),
    'platelets': (1.0, 2000.0),
    'wbc': (0.1, 500.0),
    'bilirubin': (0.05, 80.0),
    'ast': (1.0, 50000.0),
    'alt': (1.0, 50000.0),
    'glucose': (10.0, 1500.0),
    'inr': (0.5, 20.0),
    'weight_kg': (20.0, 350.0),
    'urine_output': (0.0, 10000.0),
    'fluid_input': (0.0, 50000.0),
    'net_fluid': (-20000.0, 50000.0),
}

LAB_PATTERNS = {
    'creatinine': [r'\bcreatinine\b'],
    'bun': [r'urea nitrogen', r'\bbun\b'],
    'lactate': [r'\blactate\b', r'lactic acid'],
    'ph': [r'^ph$', r'\bpH\b'],
    'pao2': [r'po2', r'oxygen pressure', r'partial pressure.*oxygen'],
    'paco2': [r'pco2', r'carbon dioxide pressure', r'partial pressure.*co2'],
    'sodium': [r'\bsodium\b'],
    'potassium': [r'\bpotassium\b'],
    'chloride': [r'\bchloride\b'],
    'bicarbonate': [r'bicarbonate', r'total co2'],
    'albumin': [r'\balbumin\b'],
    'hemoglobin': [r'\bhemoglobin\b'],
    'platelets': [r'platelet'],
    'wbc': [r'white blood', r'\bwbc\b'],
    'bilirubin': [r'bilirubin.*total', r'total bilirubin'],
    'ast': [r'\bast\b', r'aspartate'],
    'alt': [r'\balt\b', r'alanine'],
    'glucose': [r'\bglucose\b'],
    'inr': [r'\binr\b'],
}

CHART_PATTERNS = {
    'map': [r'mean.*bp', r'mean arterial', r'arterial bp mean', r'non invasive.*mean', r'nbp mean'],
    'heart_rate': [r'heart rate'],
    'resp_rate': [r'respiratory rate', r'resp rate'],
    'spo2': [r'spo2', r'o2 saturation', r'oxygen saturation'],
    'temperature': [r'temperature'],
    'weight_kg': [r'weight', r'admission weight', r'daily weight'],
}

URINE_OUTPUT_PATTERNS = [r'urine', r'foley', r'void', r'nephrostomy', r'urostomy', r'straight cath']
RRT_PATTERNS = [r'\brrt\b', r'dialysis', r'hemodialysis', r'haemodialysis', r'crrt', r'cvvh', r'cvvhd', r'cvvhdf', r'hemofiltration']
MECHVENT_PATTERNS = [r'mechanical ventilation', r'ventilator', r'intubat', r'ett', r'endotracheal']
VASOPRESSOR_PATTERNS = [r'norepinephrine', r'noradrenaline', r'epinephrine', r'adrenaline', r'vasopressin', r'phenylephrine', r'dopamine', r'dobutamine', r'angiotensin']
DIURETIC_PATTERNS = [r'furosemide', r'lasix', r'bumetanide', r'bumex', r'torsemide', r'chlorothiazide', r'metolazone']
ANTIBIOTIC_PATTERNS = [
    r'vancomycin', r'piperacillin', r'tazobactam', r'cefepime', r'ceftriaxone', r'ceftazidime',
    r'meropenem', r'imipenem', r'ertapenem', r'ciprofloxacin', r'levofloxacin', r'azithromycin',
    r'gentamicin', r'tobramycin', r'amikacin', r'linezolid', r'daptomycin', r'metronidazole'
]
NEPHROTOXIN_PATTERNS = [
    r'vancomycin', r'gentamicin', r'tobramycin', r'amikacin', r'amphotericin', r'acyclovir',
    r'tacrolimus', r'cyclosporine', r'cisplatin', r'ibuprofen', r'ketorolac', r'naproxen',
    r'indomethacin', r'contrast', r'iodixanol', r'iohexol', r'iopamidol'
]

COMORBIDITY_PATTERNS = {
    'ckd': [r'chronic kidney', r'ckd', r'renal insufficiency', r'585\b', r'N18'],
    'diabetes': [r'diabetes', r'\b250', r'E10', r'E11'],
    'chf': [r'congestive heart', r'heart failure', r'\b428', r'I50'],
    'liver_disease': [r'cirrhosis', r'hepatic failure', r'liver disease', r'K70', r'K72', r'K74'],
    'sepsis_dx': [r'sepsis', r'septic shock', r'99591', r'99592', r'A40', r'A41', r'R652'],
}
EXCLUSION_PATTERNS = [
    r'end stage renal', r'esrd', r'chronic dialysis', r'long.?term dialysis', r'hemodialysis dependence',
    r'peritoneal dialysis', r'kidney transplant', r'renal transplant', r'5856', r'N186', r'V451', r'Z992', r'V420', r'Z940'
]


def seed_all(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    try:
        torch.use_deterministic_algorithms(deterministic, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(deterministic)
    except Exception as exc:
        logging.warning("Could not set deterministic torch algorithms: %s", exc)


def setup_logging(out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / 'workflow.log'
    handlers = [logging.StreamHandler(sys.stdout), logging.FileHandler(log_path, mode='a')]
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s | %(levelname)s | %(message)s',
        handlers=handlers,
        force=True,
    )


def ensure_dirs(cfg: Config) -> Dict[str, Path]:
    out = Path(cfg.out_dir)
    work = Path(cfg.work_dir)
    dirs = {
        'out': out,
        'work': work,
        'cache': out / 'cache',
        'cohort': out / 'cohort',
        'events': out / 'events',
        'grids': out / 'grids',
        'landmarks': out / 'landmarks',
        'models': out / 'models',
        'predictions': out / 'predictions',
        'metrics': out / 'metrics',
        'figures': out / 'figures',
        'features': out / 'features',
        'psm': out / 'psm',
        'reports': out / 'reports',
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return dirs


def data_location_summary() -> str:
    roots = globals().get('DATA_SEARCH_ROOTS', [])
    pieces = [f"search_roots={[str(p) for p in roots]}"]
    pieces.append(f"AKI_MIMIC_ZIP={os.environ.get('AKI_MIMIC_ZIP', '<unset>')}")
    pieces.append(f"AKI_EICU_ZIP={os.environ.get('AKI_EICU_ZIP', '<unset>')}")
    pieces.append(f"resolved_mimic={globals().get('MIMIC_ZIP', '<unset>')}")
    pieces.append(f"resolved_eicu={globals().get('EICU_ZIP', '<unset>')}")
    return '; '.join(pieces)


def require_files(cfg: Config) -> None:
    mimic = Path(cfg.mimic_zip)
    eicu = Path(cfg.eicu_zip)
    missing = []
    if not mimic.exists():
        missing.append(f"MIMIC ZIP not found: {mimic}")
    if not eicu.exists():
        missing.append(f"eICU ZIP not found: {eicu}")
    if missing:
        raise FileNotFoundError(
            '\n'.join(missing) +
            "\nChecked configured locations: " + data_location_summary() +
            "\nSet AKI_MIMIC_ZIP and AKI_EICU_ZIP if the archives live elsewhere."
        )
    for label, path in [('MIMIC', mimic), ('eICU', eicu)]:
        try:
            with zipfile.ZipFile(path) as zf:
                n_members = len(zf.namelist())
            logging.info('%s ZIP OK: %s members at %s', label, n_members, path)
        except zipfile.BadZipFile as exc:
            raise RuntimeError(f'{label} archive exists but is not a valid ZIP: {path}') from exc



def write_environment_manifest(cfg: Config, dirs: Dict[str, Path]) -> None:
    """Write lightweight runtime metadata for reproducibility review."""
    packages = {}
    for dist_name, label in [
        ('numpy', 'numpy'), ('pandas', 'pandas'), ('pyarrow', 'pyarrow'),
        ('scikit-learn', 'scikit_learn'), ('torch', 'torch'),
        ('matplotlib', 'matplotlib'), ('shap', 'shap'), ('tqdm', 'tqdm'),
    ]:
        try:
            packages[label] = importlib_metadata.version(dist_name)
        except Exception:
            packages[label] = 'not-installed-or-unknown'
    payload = {
        'python': sys.version,
        'platform': platform.platform(),
        'device': cfg.device,
        'config': asdict(cfg),
        'packages': packages,
    }
    path = dirs['out'] / 'environment_manifest.json'
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2)



def write_clinical_model_card(cfg: Config, dirs: Dict[str, Path]) -> None:
    """Write a plain-language clinical research model card and checklist scaffold."""
    reports = dirs['reports']
    reports.mkdir(parents=True, exist_ok=True)
    model_card = f"""# Severe AKI Dynamic Prediction - Clinical Research Model Card

Pipeline version: `{CLINICAL_PIPELINE_VERSION}`

## Intended Use
Retrospective research workflow for dynamic prediction of KDIGO stage 2/3 AKI or RRT after ICU admission using MIMIC-IV and eICU data. This notebook produces a deployment-gated research artifact. It must not be used for clinical care unless the deployment gate passes with institution-specific prospective shadow validation, locked thresholds, monitoring, incident response, regulatory review, and local governance approval.

## Outcome Definition
The notebook targets severe AKI/RRT after a 6-hour landmark and 6-hour actionability gap. KDIGO-compatible severe AKI is operationalized as creatinine >=2x baseline, creatinine >=4.0 mg/dL with acute rise when configured, urine output <0.5 mL/kg/h over 12h, urine output <0.3 mL/kg/h over 24h, or RRT initiation. Death and ICU discharge are handled as competing events.

## Major Limitations
- Retrospective EHR data can encode measurement frequency, treatment decisions, documentation practice, and site effects rather than kidney biology alone.
- Baseline creatinine is imperfect; admission fallback is flagged and audited.
- Race/ethnicity is coarse, inconsistently captured, and included only for audit/subgroup summaries, not as a predictor.
- Repeated landmark rows are correlated; reported CIs use cluster bootstrap but remain approximate.
- SHAP and PSM outputs are exploratory and must not be interpreted causally.

## Required Review Before Any Clinical Translation
1. TRIPOD+AI-complete manuscript/reporting package.
2. PROBAST+AI risk-of-bias assessment by independent reviewers.
3. External validation on the target institution with frozen preprocessing and thresholds.
4. Calibration-in-the-large, calibration slope, decision-curve and threshold utility review.
5. Subgroup review across age, sex, CKD, sepsis, ICU type, baseline-source, and race/ethnicity where available.
6. Prospective silent deployment with alert burden, workflow impact, and harm monitoring.

## Reference Anchors
- KDIGO 2012 AKI guideline: https://kdigo.org/guidelines/acute-kidney-injury/
- TRIPOD+AI: https://www.tripod-statement.org/
- EQUATOR TRIPOD+AI record: https://www.equator-network.org/reporting-guidelines/tripod-statement/
- PROBAST+AI publication: https://www.bmj.com/content/388/bmj-2024-082505
"""
    (reports / 'clinical_model_card.md').write_text(model_card, encoding='utf-8')
    checklist = pd.DataFrame([
        {'domain': 'participants_data_sources', 'item': 'Cohort eligibility and exclusions audited', 'status': 'partial', 'artifact': 'metrics/clinical_audit/*_clinical_phenotype_audit.json'},
        {'domain': 'predictors', 'item': 'Predictor timing precedes landmark and action gap', 'status': 'partial', 'artifact': 'landmarks/*_landmarks.parquet'},
        {'domain': 'outcome', 'item': 'KDIGO phenotype and competing events explicitly defined', 'status': 'partial', 'artifact': 'reports/clinical_model_card.md'},
        {'domain': 'analysis', 'item': 'Internal/external metrics include calibration and clustered uncertainty', 'status': 'partial', 'artifact': 'metrics/*_metrics.csv'},
        {'domain': 'analysis', 'item': 'Threshold operating characteristics written for clinical review', 'status': 'partial', 'artifact': 'metrics/*_threshold_metrics.csv'},
        {'domain': 'applicability', 'item': 'Subgroup performance reviewed across available clinical strata', 'status': 'partial', 'artifact': 'metrics/*_subgroup_metrics.csv'},
        {'domain': 'governance', 'item': 'Prospective validation, workflow impact, and harm monitoring completed', 'status': 'not_done', 'artifact': 'external_to_notebook'},
    ])
    checklist.to_csv(reports / 'reporting_readiness_checklist.csv', index=False)



def deployment_evidence_files(dirs: Dict[str, Path]) -> Dict[str, Path]:
    evidence = dirs['reports'] / 'deployment_evidence'
    evidence.mkdir(parents=True, exist_ok=True)
    return {
        'prospective_shadow_validation_metrics': evidence / 'prospective_shadow_validation_metrics.csv',
        'locked_thresholds': evidence / 'locked_thresholds.csv',
        'intended_use_and_workflow': evidence / 'intended_use_and_workflow.md',
        'clinical_governance_approval': evidence / 'clinical_governance_approval.md',
        'regulatory_assessment': evidence / 'regulatory_assessment.md',
        'monitoring_and_drift_plan': evidence / 'monitoring_and_drift_plan.md',
        'incident_response_plan': evidence / 'incident_response_plan.md',
        'human_factors_signoff': evidence / 'human_factors_signoff.md',
    }


def write_deployment_evidence_templates(cfg: Config, dirs: Dict[str, Path]) -> None:
    '''Write a draft hospital deployment evidence packet without claiming approval.

    These documents are intentionally detailed but remain drafts. The deployment
    gate treats the draft marker as incomplete until a local hospital supplies
    prospective shadow-mode metrics, locks thresholds, and signs governance,
    regulatory, human-factors, monitoring, and incident-response artifacts.
    '''
    files = deployment_evidence_files(dirs)
    marker = 'DRAFT_NOT_APPROVED_FOR_CLINICAL_USE'
    evidence_dir = files['intended_use_and_workflow'].parent

    csv_templates = {
        'prospective_shadow_validation_metrics': 'horizon_hours,n,events,threshold,sensitivity,specificity,ppv,npv,alert_rate,ece,calibration_intercept,calibration_slope,review_status\n',
        'locked_thresholds': 'horizon_hours,threshold,locked_by,locked_date,clinical_rationale,review_status\n',
    }

    draft_docs = {
        'intended_use_and_workflow': f'''# Intended Use and Workflow

{marker}

## Intended Use
This artifact is a draft workflow specification for prospective shadow validation of the severe AKI/RRT dynamic prediction model. It is not an order set, treatment recommendation, autonomous diagnostic system, or clinically approved alert. It may only be used in silent mode until the deployment gate passes with local evidence.

## Eligible Population
Adult ICU patients after the first 6 hours of an ICU stay, excluding patients with ESRD/chronic dialysis, kidney transplant status where captured, ICU stays shorter than 24 hours, severe AKI/RRT already present by the 6-hour landmark, and stays with insufficient urine-output observability under the configured KDIGO urine-output coverage rule.

## Prediction Timing
The primary operating horizon is {getattr(cfg, 'target_operating_horizon_hours', 72)} hours after each eligible landmark, with a {getattr(cfg, 'action_gap_hours', 6)}-hour actionability gap. Repeated landmark rows must be interpreted as serial assessments within the same stay, not independent patients.

## Recipients and Actions
During shadow validation, predictions are logged only for evaluation. After governance approval, any alert must be routed to the locally named ICU/nephrology clinical owner and paired with a written clinical pathway covering medication review, hemodynamic assessment, fluid balance review, repeat creatinine/urine-output assessment, and nephrology escalation criteria.

## Required Local Customization
The hospital must specify the alert recipient, escalation window, override workflow, audit owner, downtime procedure, model version, EHR integration owner, and patient-safety contact. No bedside display or interruptive alert is allowed until those fields are locally approved.
''',
        'clinical_governance_approval': f'''# Clinical Governance Approval

{marker}

## Required Decision
A local clinical AI, patient-safety, ICU, nephrology, informatics, privacy, and quality-governance body must make an explicit go/no-go decision before any non-silent use. Retrospective MIMIC/eICU performance does not satisfy this approval.

## Minimum Evidence for Review
- Prospective silent-mode validation at the target hospital with at least {getattr(cfg, 'deployment_required_min_n', 1000)} eligible prediction rows and enough outcome events for stable calibration review.
- Locked threshold row for the {getattr(cfg, 'target_operating_horizon_hours', 72)}-hour horizon.
- Calibration-in-the-large, calibration slope, ECE, Brier score, AUROC, AUPRC, PPV, sensitivity, specificity, NPV, and alert burden.
- Subgroup review by age, sex, CKD, sepsis, ICU type, baseline creatinine source, race/ethnicity where legally and ethically available, and missingness/measurement-frequency strata.
- Human-factors simulation showing that clinicians understand that SHAP/PSM outputs are exploratory and non-causal.

## Local Signoff Fields
Committee name: TBD.
Clinical owner: TBD.
Technical owner: TBD.
Patient-safety owner: TBD.
Decision date: TBD.
Decision: not approved.
Review cadence: TBD.
''',
        'regulatory_assessment': f'''# Regulatory Assessment

{marker}

## Preliminary Classification Note
This is a retrospective research model and prospective shadow-validation package, not a cleared clinical device. Any move from silent evaluation to clinician-facing use requires legal and regulatory review. The review should consider FDA Clinical Decision Support guidance, whether the user can independently review the basis for a recommendation, SaMD implications, institutional AI policy, privacy/security obligations, ONC HTI-1 decision-support transparency requirements where applicable, and local procurement rules.

## Current Regulatory Position
- No clinical deployment approval is documented.
- No locked intended-use statement has been approved by the institution.
- No local threshold has been locked.
- No post-deployment monitoring owner has been assigned.
- SHAP and PSM are exploratory diagnostics only and must not be represented as causal justification for clinical actions.

## Required Local Review Outputs
Counsel/regulatory reviewer: TBD.
FDA CDS/SaMD determination: TBD.
ONC HTI-1/DSI applicability: TBD.
Privacy/security assessment: TBD.
Approved labeling and user-facing language: TBD.
''',
        'monitoring_and_drift_plan': f'''# Monitoring and Drift Plan

{marker}

## Silent-Mode Monitoring Before Go-Live
Monitor case mix, variable missingness, urine-output coverage, baseline creatinine source, serum creatinine measurement frequency, treatment/documentation features, calibration, discrimination, threshold operating characteristics, subgroup performance, and alert burden. The prespecified primary horizon is {getattr(cfg, 'target_operating_horizon_hours', 72)} hours.

## Prospective Performance Guardrails
The local review should require sensitivity >= {getattr(cfg, 'deployment_min_sensitivity', 0.70):.2f}, alert rate <= {getattr(cfg, 'deployment_max_alert_rate', 0.25):.2f}, ECE <= {getattr(cfg, 'deployment_max_ece', 0.05):.2f}, and clinically acceptable calibration slope/intercept at the locked threshold. These are minimum guardrails, not automatic approval rules.

## Drift Triggers
Pause or investigate if baseline-source mix changes materially, urine-output capture deteriorates, serum creatinine ordering frequency shifts, any subgroup shows clinically meaningful calibration drift, alert burden exceeds the locked threshold plan, the EHR mapping changes, RRT documentation changes, or observed event prevalence moves outside the validation envelope.

## Cadence
Daily technical feed checks during shadow validation; weekly clinical safety review during the first 8 weeks of any controlled deployment; monthly calibration and subgroup review thereafter; immediate review after EHR upgrades, ICU workflow changes, laboratory interface changes, or safety events.
''',
        'incident_response_plan': f'''# Incident Response Plan

{marker}

## Scope
This plan covers silent-mode data failures, erroneous predictions, suspected patient-safety events, excessive alert burden, calibration drift, EHR feed interruptions, and any clinician-facing misunderstanding if the model later enters controlled deployment.

## Immediate Actions
1. Disable clinician-facing alerts or hold go-live if patient safety may be affected.
2. Preserve model inputs, prediction logs, threshold version, mapping version, and relevant EHR audit logs.
3. Notify the clinical owner, technical owner, patient-safety owner, and governance chair.
4. Triage whether the issue is data mapping, model drift, threshold misuse, interface design, or clinical workflow mismatch.

## Review and Rollback
A rollback decision must be available at all times. Reinstatement requires documented root-cause analysis, regression testing on recent local data, calibration/threshold review, and governance signoff. Serious events require institutional patient-safety reporting according to local policy.
''',
        'human_factors_signoff': f'''# Human Factors Signoff

{marker}

## Required Testing
Before any clinician-facing use, complete simulated ICU workflow testing with intended users. Include cases with missing urine output, admission-fallback baseline creatinine, CKD, sepsis, diuretic exposure, nephrotoxin exposure, RRT documentation edge cases, and repeated landmark predictions from the same stay.

## Comprehension Requirements
Users must understand that the score is a risk estimate, not a diagnosis or treatment instruction; repeated landmarks are correlated; retrospective external validation does not prove local transportability; and SHAP/PSM outputs are non-causal exploratory diagnostics.

## Alert-Burden Review
The local locked threshold must be reviewed against predicted daily alert volume, ICU staffing, escalation capacity, false-positive burden, and potential automation bias. If users cannot explain the action pathway and limitations, the model must remain silent.

## Signoff Fields
Simulation date: TBD.
Number and roles of participants: TBD.
Observed workflow failures: TBD.
Training materials approved: TBD.
Decision: not approved.
''',
    }

    supporting_docs = {
        'prospective_shadow_validation_protocol.md': f'''# Prospective Shadow Validation Protocol

{marker}

## Objective
Estimate local, prospective, silent-mode performance before any clinical use. The model must run with frozen preprocessing, frozen feature definitions, no clinician-visible alerts, and no retraining on the target hospital data during the evaluation window.

## Primary Endpoint
Severe AKI/RRT within {getattr(cfg, 'target_operating_horizon_hours', 72)} hours after an eligible landmark, with the configured actionability gap and exclusion rules. Death and ICU discharge should be reported as competing outcomes rather than ignored.

## Required Analyses
Report AUROC, AUPRC, Brier score, ECE, calibration intercept, calibration slope, threshold sensitivity/specificity/PPV/NPV, alert rate, decision-curve or net-benefit analysis if possible, first-landmark-per-stay sensitivity, and subgroup performance. Use patient/stay-level clustered confidence intervals because repeated landmark rows are correlated.

## Pass/Review Criteria
Minimum statistical review requires n >= {getattr(cfg, 'deployment_required_min_n', 1000)}, sensitivity >= {getattr(cfg, 'deployment_min_sensitivity', 0.70):.2f}, alert rate <= {getattr(cfg, 'deployment_max_alert_rate', 0.25):.2f}, ECE <= {getattr(cfg, 'deployment_max_ece', 0.05):.2f}, no unacceptable subgroup failure, and governance acceptance. These criteria do not replace clinician judgment or regulatory review.
''',
        'threshold_locking_protocol.md': f'''# Threshold Locking Protocol

{marker}

## Current Position
No clinical threshold is locked. The retrospective 72-hour external model did not meet the aspirational AUROC >= {getattr(cfg, 'minimum_external_auc_for_strong_claim', 0.80):.2f} guardrail, and the evaluated retrospective thresholds did not establish a deployable alert policy.

## Locking Process
Threshold selection must occur after local prospective shadow validation. The lock decision should consider sensitivity, PPV, alert rate, calibration, subgroup performance, staffing capacity, intended action pathway, and patient-safety review. The selected threshold must be entered in locked_thresholds.csv with review_status=approved only after governance approval.

## Change Control
Any threshold change after approval requires versioning, justification, retrospective replay on recent local data, prospective silent recheck if material, and governance signoff.
''',
        'local_data_mapping_specification.md': f'''# Local Data Mapping Specification

{marker}

## Required Mappings
Map local EHR data to the notebook variable ontology before prospective validation: serum creatinine, urine output, RRT, weight, ICU admission/discharge times, death time, creatinine baseline source, vasopressors, diuretics, nephrotoxins, antibiotics, mechanical ventilation, vitals, blood gases, chemistry, hematology, comorbidities, ICU type, and demographics needed for audit.

## Unit and Timing Checks
Verify units for creatinine, urine output, weight, temperature, blood gases, and medication timestamps. All predictors must be available before the prediction landmark plus actionability-gap logic. Outcome labels must not leak post-landmark information into features.

## Measurement Bias Checks
Before validation, summarize serum creatinine ordering frequency, urine-output coverage, missingness, medication documentation, and RRT documentation by service line and ICU type. These summaries must be reviewed as possible measurement-process bias, not merely data quality noise.
''',
        'deployment_evidence_packet_index.md': f'''# Deployment Evidence Packet Index

{marker}

## Retrospective Research Artifacts
- clinical_model_card.md
- clinical_bias_transportability_audit.md
- model_selection_and_claims_report.md/json
- manuscript_results_summary.json
- external_metric_ranking_primary_horizon.csv
- all_candidate_metrics_for_selection.csv

## Local Evidence Required Before Clinical Use
- prospective_shadow_validation_metrics.csv with approved local rows
- locked_thresholds.csv with approved local threshold
- intended_use_and_workflow.md completed by local clinical owner
- clinical_governance_approval.md signed by governance body
- regulatory_assessment.md completed by legal/regulatory reviewer
- monitoring_and_drift_plan.md completed by operations owner
- incident_response_plan.md completed by patient-safety and technical owners
- human_factors_signoff.md completed after simulation testing
''',
    }

    def should_write(path: Path) -> bool:
        if not path.exists() or path.stat().st_size == 0:
            return True
        text = path.read_text(encoding='utf-8', errors='ignore')
        return 'PLACEHOLDER_TEMPLATE_NOT_COMPLETE' in text or len(text.strip()) < 250

    for key, text in csv_templates.items():
        path = files[key]
        if not path.exists() or path.stat().st_size == 0:
            path.write_text(text, encoding='utf-8')

    for key, text in draft_docs.items():
        path = files[key]
        if should_write(path):
            path.write_text(text, encoding='utf-8')

    for name, text in supporting_docs.items():
        path = evidence_dir / name
        if should_write(path):
            path.write_text(text, encoding='utf-8')


def _read_local_validation_metrics(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as exc:
        logging.warning('Could not read local validation metrics %s: %s', path, exc)
        return pd.DataFrame()


def _deployment_evidence_completion(name: str, path: Path, cfg: Config) -> Tuple[bool, str]:
    marker = 'PLACEHOLDER_TEMPLATE_NOT_COMPLETE'
    if not path.exists():
        return False, 'missing'
    if path.stat().st_size == 0:
        return False, 'empty'
    if path.suffix.lower() == '.csv':
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            return False, f'unreadable CSV: {exc}'
        if df.empty:
            return False, 'CSV has no completed evidence rows'
        if 'review_status' not in df.columns:
            return False, 'missing review_status column'
        status = df['review_status'].fillna('').astype(str).str.lower().str.strip()
        if not status.isin({'approved', 'accepted', 'complete', 'pass', 'passed'}).any():
            return False, 'no approved/complete review_status row'
        if name in {'prospective_shadow_validation_metrics', 'locked_thresholds'} and 'horizon_hours' in df.columns:
            target_h = int(getattr(cfg, 'target_operating_horizon_hours', 72))
            h = pd.to_numeric(df['horizon_hours'], errors='coerce')
            if not h.eq(target_h).any():
                return False, f'no row for target horizon {target_h}h'
        return True, 'complete'
    text = path.read_text(encoding='utf-8', errors='ignore')
    if marker in text:
        return False, 'placeholder template was not replaced'
    draft_markers = {'DRAFT_NOT_APPROVED_FOR_CLINICAL_USE'}
    if any(m in text for m in draft_markers):
        return False, 'draft document requires local completion and approval'
    if len(text.strip()) < 500:
        return False, 'document is too short for governance evidence'
    return True, 'complete'


def _collect_retrospective_clinical_warnings(cfg: Config, dirs: Dict[str, Path]) -> List[str]:
    warnings_out: List[str] = []
    metrics_dir = dirs.get('metrics') if isinstance(dirs, dict) else None
    if metrics_dir is None or not Path(metrics_dir).exists():
        return warnings_out
    metrics_dir = Path(metrics_dir)
    for path in sorted(metrics_dir.glob('*_measurement_process_bias_audit.csv')):
        try:
            df = pd.read_csv(path)
            for _, row in df.iterrows():
                auc = pd.to_numeric(pd.Series([row.get('auc')]), errors='coerce').iloc[0]
                if np.isfinite(auc) and auc >= float(getattr(cfg, 'measurement_bias_auc_warn', 0.70)):
                    warnings_out.append(f'{path.name}: measurement-process-only AUROC={auc:.3f}; retrospective EHR measurement bias is material.')
        except Exception as exc:
            warnings_out.append(f'{path.name}: could not inspect measurement audit ({exc}).')
    for path in sorted(metrics_dir.glob('*_treatment_feature_ablation.csv')):
        try:
            df = pd.read_csv(path)
            d = df.loc[df.get('model', pd.Series(dtype=str)).astype(str).eq('delta_all_minus_reduced')]
            if not d.empty:
                delta = pd.to_numeric(d.iloc[0].get('auc'), errors='coerce')
                if np.isfinite(delta) and abs(float(delta)) >= float(getattr(cfg, 'treatment_ablation_auc_delta_warn', 0.03)):
                    warnings_out.append(f'{path.name}: AUROC changed by {float(delta):.3f} after removing treatment/documentation features; confounding sensitivity is material.')
        except Exception as exc:
            warnings_out.append(f'{path.name}: could not inspect treatment ablation ({exc}).')
    for path in sorted(metrics_dir.glob('*_baseline_source_sensitivity.csv')):
        try:
            df = pd.read_csv(path)
            if {'horizon_hours', 'ece', 'baseline_source'}.issubset(df.columns):
                for h, g in df.groupby('horizon_hours'):
                    ece = pd.to_numeric(g['ece'], errors='coerce').dropna()
                    if len(ece) >= 2:
                        gap = float(ece.max() - ece.min())
                        if gap >= float(getattr(cfg, 'baseline_source_calibration_gap_warn', 0.05)):
                            warnings_out.append(f'{path.name}: ECE gap by baseline creatinine source is {gap:.3f} at {h}h; baseline-source transportability is fragile.')
        except Exception as exc:
            warnings_out.append(f'{path.name}: could not inspect baseline-source sensitivity ({exc}).')
    return warnings_out


def write_clinical_deployment_gate_report(cfg: Config, dirs: Dict[str, Path], experiment: str = 'unknown') -> Dict[str, Any]:
    """Create a hard go/no-go deployment report.

    The gate intentionally cannot be satisfied by MIMIC/eICU retrospective metrics alone.
    It requires local prospective shadow-mode evidence and completed governance artifacts.
    """
    reports = dirs['reports']
    reports.mkdir(parents=True, exist_ok=True)
    write_deployment_evidence_templates(cfg, dirs)
    files = deployment_evidence_files(dirs)
    blockers: List[str] = []
    warnings_list = _collect_retrospective_clinical_warnings(cfg, dirs)
    completion = {name: _deployment_evidence_completion(name, path, cfg) for name, path in files.items()}
    incomplete = [f'{name} ({reason})' for name, (ok, reason) in completion.items() if not ok]
    if incomplete:
        blockers.append('Incomplete required deployment evidence: ' + '; '.join(incomplete))
    local = _read_local_validation_metrics(files['prospective_shadow_validation_metrics'])
    target_h = int(getattr(cfg, 'target_operating_horizon_hours', 72))
    if local.empty:
        blockers.append('No completed local prospective shadow-mode validation metrics were provided.')
    else:
        local_h = local.loc[pd.to_numeric(local.get('horizon_hours', pd.Series(dtype=float)), errors='coerce').eq(target_h)].copy()
        if local_h.empty:
            blockers.append(f'No local prospective metrics for target horizon {target_h}h.')
        else:
            approved = local_h.get('review_status', pd.Series('', index=local_h.index)).fillna('').astype(str).str.lower().isin({'approved', 'accepted', 'complete', 'pass', 'passed'})
            if not approved.any():
                blockers.append(f'No approved prospective validation row for {target_h}h horizon.')
            row = local_h.loc[approved].iloc[0] if approved.any() else local_h.iloc[0]
            n = float(row.get('n', 0) or 0)
            sensitivity = float(row.get('sensitivity', np.nan))
            alert_rate = float(row.get('alert_rate', np.nan))
            ece = float(row.get('ece', np.nan))
            if n < float(getattr(cfg, 'deployment_required_min_n', 1000)):
                blockers.append(f'Prospective validation n={n:.0f} is below required minimum {cfg.deployment_required_min_n}.')
            if not np.isfinite(sensitivity) or sensitivity < float(cfg.deployment_min_sensitivity):
                blockers.append(f'Sensitivity {sensitivity} is below required minimum {cfg.deployment_min_sensitivity}.')
            if not np.isfinite(alert_rate) or alert_rate > float(cfg.deployment_max_alert_rate):
                blockers.append(f'Alert rate {alert_rate} exceeds maximum {cfg.deployment_max_alert_rate}.')
            if not np.isfinite(ece) or ece > float(cfg.deployment_max_ece):
                blockers.append(f'ECE {ece} exceeds maximum {cfg.deployment_max_ece}.')
    retrospective_metric_files = sorted(Path(dirs['metrics']).glob('*_external_metrics.csv')) if dirs.get('metrics') else []
    status = 'PASS_FOR_LOCAL_CONTROLLED_DEPLOYMENT_REVIEW' if not blockers else 'BLOCKED_NOT_CLINICALLY_DEPLOYABLE'
    payload = {
        'pipeline_version': CLINICAL_PIPELINE_VERSION,
        'experiment': experiment,
        'status': status,
        'blockers': blockers,
        'warnings': warnings_list,
        'required_evidence_files': {k: str(v) for k, v in files.items()},
        'evidence_completion': {k: {'complete': bool(ok), 'reason': reason} for k, (ok, reason) in completion.items()},
        'retrospective_external_metric_files': [str(p) for p in retrospective_metric_files],
        'supporting_deployment_documents': [str(p) for p in sorted((reports / 'deployment_evidence').glob('*.md'))],
        'regulatory_note': 'This report is not legal advice. FDA CDS/SaMD and ONC HTI-1 applicability require institutional review.',
    }
    with open(reports / 'clinical_deployment_gate_report.json', 'w') as f:
        json.dump(payload, f, indent=2)
    md = [
        '# Clinical Deployment Gate Report',
        '',
        f'Pipeline version: `{CLINICAL_PIPELINE_VERSION}`',
        f'Status: **{status}**',
        '',
        '## Blockers',
    ]
    md += [f'- {b}' for b in blockers] if blockers else ['- None. Proceed only through local controlled deployment review.']
    md += ['', '## Retrospective Clinical Warnings']
    md += [f'- {w}' for w in warnings_list] if warnings_list else ['- No retrospective warning thresholds were crossed, or audits have not been run yet.']
    md += ['', '## Required Evidence Files']
    md += [f'- `{name}`: `{path}`' for name, path in payload['required_evidence_files'].items()]
    md += ['', '## Supporting Draft Deployment Documents']
    md += [f'- `{path}`' for path in payload['supporting_deployment_documents']]
    md += ['', '## Non-Causal Interpretation Warning', '- SHAP and PSM outputs are exploratory diagnostics. They must not be used as causal evidence for treatment decisions.']
    (reports / 'clinical_deployment_gate_report.md').write_text('\n'.join(md) + '\n', encoding='utf-8')
    return payload


## 5. Direct ZIP CSV reader and parquet utilities

`ZipCSVStore` lets the workflow read `csv` and `csv.gz` tables directly inside the two large ZIP archives. This is the key reason no PostgreSQL server is needed. `ParquetSink` writes chunked extraction results incrementally to compressed parquet.


In [ ]:
# -----------------------------------------------------------------------------
# ZIP CSV reader
# -----------------------------------------------------------------------------
class ZipCSVStore:
    def __init__(self, zip_path: Path):
        self.zip_path = Path(zip_path)
        self._zip = zipfile.ZipFile(self.zip_path, 'r')
        self.members = [m for m in self._zip.namelist() if not m.endswith('/')]
        self.lower_members = {m.lower(): m for m in self.members}
        self._member_cache: Dict[str, str] = {}

    def close(self) -> None:
        self._zip.close()

    def find(self, table: str, required: bool = True) -> Optional[str]:
        key = table.lower().strip('/').replace('\\', '/')
        if key in self._member_cache:
            return self._member_cache[key]
        candidates = []
        stems = {key, key + '.csv', key + '.csv.gz'}
        base_key = key.split('/')[-1]
        base_stems = {base_key, base_key + '.csv', base_key + '.csv.gz'}
        for m in self.members:
            ml = m.lower()
            b = ml.split('/')[-1]
            if ml.endswith(tuple(stems)) or b in base_stems:
                candidates.append(m)
        if not candidates:
            if required:
                raise FileNotFoundError(f"Table {table!r} not found in {self.zip_path}")
            return None
        # Prefer exact basename csv.gz/csv and shorter paths.
        candidates = sorted(candidates, key=lambda x: (len(x), x))
        member = candidates[0]
        self._member_cache[key] = member
        return member

    def open_binary(self, member: str):
        raw = self._zip.open(member, 'r')
        if member.lower().endswith('.gz'):
            gz = gzip.GzipFile(fileobj=raw)
            # GzipFile.close() does NOT close the underlying fileobj when opened
            # with fileobj=. Wrap so that close() cleans up both layers; a leaked
            # ZipExtFile holds a _SharedFile position that, on FUSE mounts like
            # Google Drive, causes the next open of the same member to read EOF.
            gz._aki_underlying_raw = raw
            _orig_close = gz.close
            def _patched_close(_oc=_orig_close, _r=raw):
                _oc()
                try:
                    _r.close()
                except Exception:
                    pass
            gz.close = _patched_close
            return gz
        return raw

    def columns(self, table: str, required: bool = True) -> List[str]:
        member = self.find(table, required=required)
        if member is None:
            return []
        with self.open_binary(member) as f:
            df = pd.read_csv(f, nrows=0)
        return list(df.columns)

    def read_csv(
        self,
        table: str,
        usecols: Optional[Sequence[str]] = None,
        parse_dates: Optional[Sequence[str]] = None,
        dtype: Optional[Dict[str, Any]] = None,
        chunksize: Optional[int] = None,
        required: bool = True,
    ):
        member = self.find(table, required=required)
        if member is None:
            return None
        cols = self.columns(table, required=required)
        if usecols is not None:
            usecols_present = [c for c in usecols if c in cols]
            missing = [c for c in usecols if c not in cols]
            if missing:
                logging.warning("%s missing columns ignored: %s", table, missing[:10])
            usecols = usecols_present
        if parse_dates is not None:
            parse_dates = [c for c in parse_dates if usecols is None or c in usecols]
        f = self.open_binary(member)
        return pd.read_csv(
            f,
            usecols=usecols,
            parse_dates=parse_dates,
            dtype=dtype,
            chunksize=chunksize,
            low_memory=False if chunksize is None else True,
        )


def normalize_cols_lower(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def get_col(df: pd.DataFrame, options: Sequence[str], default: Optional[str] = None) -> Optional[str]:
    colmap = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in colmap:
            return colmap[opt.lower()]
    return default


def series_or_default(df: pd.DataFrame, col: str, default: Any = np.nan) -> pd.Series:
    """Return df[col] as a Series, or a same-length default Series when the column is absent.

    Several MIMIC/eICU tables differ slightly across releases or exported subsets. Using
    DataFrame.get(..., scalar) followed by .fillna() can fail because scalars do not have
    pandas Series methods. This helper keeps optional-column handling safe and explicit.
    """
    actual = get_col(df, [col])
    if actual is not None and actual in df.columns:
        return df[actual]
    return pd.Series(default, index=df.index)


def normalize_id_series(s: Any, fill_missing: str = '') -> pd.Series:
    """Return stable string identifiers for MIMIC/eICU joins.

    Pandas reads identifier columns with any missing value as float. A MIMIC hadm_id
    can therefore become the string ``"12345678.0"`` in labevents while the same ID is
    ``"12345678"`` in icustays/admissions. That exact mismatch was the cause of the
    MIMIC landmark failure where all 49,958 stays were excluded as ``missing_scr_or_uo``:
    the creatinine labs were present in the ZIP, but the ICU stay merge dropped them.
    This helper canonicalizes numeric-looking IDs by removing trailing decimals while
    preserving nonnumeric IDs such as eICU uniquePid values.
    """
    if isinstance(s, pd.Series):
        ser = s.copy()
        idx = ser.index
    else:
        ser = pd.Series(s)
        idx = ser.index
    out = ser.astype('string').str.strip()
    num = pd.to_numeric(out, errors='coerce')
    m = num.notna() & np.isfinite(num.astype(float))
    if m.any():
        rounded = np.rint(num.loc[m].astype(float).to_numpy()).astype(np.int64).astype(str)
        out.loc[m] = rounded
    out = out.str.replace(r'\.0$', '', regex=True)
    bad = out.str.lower().isin(['', 'nan', 'none', '<na>', 'nat'])
    out = out.mask(bad, pd.NA)
    if fill_missing is not None:
        out = out.fillna(fill_missing)
    out.index = idx
    return out.astype('string')


def normalize_id_columns(df: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    """Normalize selected ID columns in-place when they are present and return df."""
    for c in cols:
        if c in df.columns:
            df[c] = normalize_id_series(df[c])
    return df


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors='coerce')


def str_contains_any(s: pd.Series, patterns: Sequence[str]) -> pd.Series:
    if len(patterns) == 0:
        return pd.Series(False, index=s.index)
    text = s.fillna('').astype(str).str.lower()
    regex = '|'.join(f'(?:{p})' for p in patterns)
    return text.str.contains(regex, regex=True, case=False, na=False)


def clip_values(var: str, values: pd.Series) -> pd.Series:
    values = to_num(values)
    if var in VALUE_RANGES:
        lo, hi = VALUE_RANGES[var]
        values = values.where((values >= lo) & (values <= hi))
    return values


def parse_eicu_age(x: Any) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.startswith('>'):
        return 91.0
    try:
        return float(s)
    except Exception:
        return np.nan


def sex_female_value(x: Any) -> float:
    s = str(x).strip().lower()
    if s.startswith('f'):
        return 1.0
    if s.startswith('m'):
        return 0.0
    return np.nan




def normalize_race_ethnicity_value(x: Any) -> str:
    """Coarse harmonization for audit/subgroup reporting only, not model input."""
    if pd.isna(x):
        return 'unknown'
    s = str(x).strip().lower()
    if s in {'', 'nan', 'none', '<na>', 'unknown', 'unable to obtain', 'patient declined'}:
        return 'unknown'
    if 'hispanic' in s or 'latino' in s or 'latina' in s:
        return 'hispanic_latino'
    if 'black' in s or 'african' in s:
        return 'black'
    if 'asian' in s:
        return 'asian'
    if 'white' in s or 'caucasian' in s:
        return 'white'
    if 'american indian' in s or 'alaska' in s or 'native american' in s:
        return 'american_indian_alaska_native'
    if 'pacific' in s or 'hawaiian' in s:
        return 'native_hawaiian_pacific_islander'
    if 'multi' in s or 'multiple' in s:
        return 'multiple'
    if 'other' in s:
        return 'other'
    return 'other'


def safe_datetime(df: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


def read_parquet(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)


class ParquetSink:
    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists():
            self.path.unlink()
        self.writer: Optional[pq.ParquetWriter] = None
        self.n_rows = 0

    def write(self, df: pd.DataFrame) -> None:
        if df is None or df.empty:
            return
        df = df.reset_index(drop=True)
        table = pa.Table.from_pandas(df, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(self.path, table.schema, compression='zstd')
        else:
            table = table.cast(self.writer.schema, safe=False)
        self.writer.write_table(table)
        self.n_rows += len(df)

    def close(self) -> None:
        if self.writer is not None:
            self.writer.close()
            self.writer = None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()


## 6. Clinical concept mapping

This layer maps source-specific item labels, lab names, drug names, treatment strings, and diagnosis text into the harmonized AKI modeling variables. It also expands medication/treatment strings into vasopressor, diuretic, nephrotoxin, antibiotic, mechanical-ventilation, and RRT exposures.


In [ ]:
# -----------------------------------------------------------------------------
# Item/concept matching
# -----------------------------------------------------------------------------
def match_by_patterns(
    df: pd.DataFrame,
    label_col: str,
    patterns_by_var: Dict[str, Sequence[str]],
    exclude_patterns: Optional[Sequence[str]] = None,
    extra_filter: Optional[pd.Series] = None,
) -> Dict[int, str]:
    out: Dict[int, str] = {}
    if df.empty:
        return out
    tmp = df.copy()
    tmp[label_col] = tmp[label_col].fillna('').astype(str)
    if extra_filter is not None:
        tmp = tmp.loc[extra_filter.reindex(tmp.index).fillna(False)]
    if exclude_patterns:
        tmp = tmp.loc[~str_contains_any(tmp[label_col], exclude_patterns)]
    for var, patterns in patterns_by_var.items():
        mask = str_contains_any(tmp[label_col], patterns)
        ids = tmp.loc[mask, 'itemid'].dropna().astype(int).unique().tolist()
        for itemid in ids:
            out[itemid] = var
    return out


def map_name_to_var(name: Any, pattern_groups: Dict[str, Sequence[str]]) -> Optional[str]:
    s = str(name).lower() if not pd.isna(name) else ''
    for var, pats in pattern_groups.items():
        for p in pats:
            if re.search(p, s, flags=re.IGNORECASE):
                return var
    return None


def map_drug_to_exposures(name: Any) -> List[str]:
    s = str(name).lower() if not pd.isna(name) else ''
    out = []
    if any(re.search(p, s, re.I) for p in VASOPRESSOR_PATTERNS):
        out.append('vasopressor')
    if any(re.search(p, s, re.I) for p in DIURETIC_PATTERNS):
        out.append('diuretic')
    if any(re.search(p, s, re.I) for p in ANTIBIOTIC_PATTERNS):
        out.append('antibiotic')
    if any(re.search(p, s, re.I) for p in NEPHROTOXIN_PATTERNS):
        out.append('nephrotoxin')
    return sorted(set(out))


def map_treatment_to_exposures(name: Any) -> List[str]:
    s = str(name).lower() if not pd.isna(name) else ''
    out = []
    if any(re.search(p, s, re.I) for p in RRT_PATTERNS):
        out.append('rrt')
    if any(re.search(p, s, re.I) for p in MECHVENT_PATTERNS):
        out.append('mechanical_ventilation')
    return sorted(set(out))


## 7. Time-bin aggregation

Raw measurements are converted into regular ICU time bins. Continuous variables are averaged, flow/sum variables such as urine output and fluid input are summed, and binary exposures are max-pooled within each bin. The model still receives masks, observation counts, and time-gap channels, so the informative observation process is not erased.


In [ ]:
# -----------------------------------------------------------------------------
# Binned event aggregation
# -----------------------------------------------------------------------------
def empty_binned_events() -> pd.DataFrame:
    return pd.DataFrame(columns=['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind'])


def make_binned_events(
    df: pd.DataFrame,
    cfg: Config,
    stay_col: str = 'stay_id',
    hour_col: str = 'hour',
    var_col: str = 'variable',
    value_col: str = 'value',
) -> pd.DataFrame:
    """Aggregate raw events into fixed time bins with a low-memory path.

    eICU vital/lab chunks can expand into more than one million long-format rows.
    Avoid copying object-heavy columns wholesale; construct compact typed columns
    and then group by variable family. This changes runtime memory behavior only,
    not the phenotype, timing, or aggregation semantics.
    """
    if df is None or df.empty:
        return empty_binned_events()
    required = [stay_col, hour_col, var_col, value_col]
    if any(c not in df.columns for c in required):
        return empty_binned_events()

    hour = to_num(df[hour_col])
    value = to_num(df[value_col])
    variable = df[var_col].astype('string')
    stay = normalize_id_series(df[stay_col])
    valid = (
        stay.notna()
        & hour.notna()
        & variable.notna()
        & value.notna()
        & (hour >= 0)
        & (hour <= cfg.max_icu_hours)
    )
    if not bool(valid.any()):
        return empty_binned_events()

    hour_valid = hour.loc[valid].to_numpy(dtype=np.float32, copy=False)
    bins = np.floor(hour_valid / float(cfg.grid_hours)).astype(np.int32)
    keep = (bins >= 0) & (bins < cfg.max_grid_steps)
    if not bool(np.any(keep)):
        return empty_binned_events()

    # Materialize only the typed columns needed for grouping. The previous
    # df[[...]].copy() path duplicated large object blocks and failed on eICU.
    stay_values = stay.loc[valid].astype(str).to_numpy()[keep]
    var_values = variable.loc[valid].astype(str).to_numpy()[keep]
    val_values = value.loc[valid].to_numpy(dtype=np.float32, copy=False)[keep]
    bin_values = bins[keep]
    x = pd.DataFrame(
        {
            'stay_id': stay_values,
            'bin': bin_values,
            'variable': var_values,
            'value': val_values,
        },
        copy=False,
    )
    del stay_values, var_values, val_values, bin_values, bins, keep, hour_valid
    if x.empty:
        return empty_binned_events()

    pieces = []
    variable_series = x['variable']
    for kind, vars_ in [('continuous', CONTINUOUS_VARS), ('sum', ['urine_output', 'fluid_input']), ('binary', BINARY_VARS)]:
        m = variable_series.isin(vars_)
        if not bool(m.any()):
            continue
        if kind == 'binary':
            y = pd.DataFrame(
                {
                    'stay_id': x.loc[m, 'stay_id'].to_numpy(copy=False),
                    'bin': x.loc[m, 'bin'].to_numpy(copy=False),
                    'variable': x.loc[m, 'variable'].to_numpy(copy=False),
                    'value': (x.loc[m, 'value'].fillna(0).to_numpy(dtype=np.float32, copy=False) > 0).astype(np.float32),
                },
                copy=False,
            )
            g = y.groupby(['stay_id', 'bin', 'variable'], observed=True)['value'].agg(['max', 'count']).reset_index()
            g = g.rename(columns={'max': 'value', 'count': 'n_obs'})
        else:
            y = x.loc[m, ['stay_id', 'bin', 'variable', 'value']]
            if kind == 'continuous':
                g = y.groupby(['stay_id', 'bin', 'variable'], observed=True)['value'].agg(['mean', 'count']).reset_index()
                g = g.rename(columns={'mean': 'value', 'count': 'n_obs'})
            else:
                g = y.groupby(['stay_id', 'bin', 'variable'], observed=True)['value'].agg(['sum', 'count']).reset_index()
                g = g.rename(columns={'sum': 'value', 'count': 'n_obs'})
        g['kind'] = kind
        pieces.append(g[['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind']])
        del y, g

    if not pieces:
        return empty_binned_events()
    out = pd.concat(pieces, ignore_index=True)
    out['stay_id'] = out['stay_id'].astype(str)
    out['variable'] = out['variable'].astype(str)
    out['kind'] = out['kind'].astype(str)
    out['bin'] = out['bin'].astype(np.int32)
    out['value'] = out['value'].astype(np.float32)
    out['n_obs'] = out['n_obs'].clip(0, 32767).astype(np.int16)
    return out

def finalize_binned_events(in_path: Path, out_path: Path) -> pd.DataFrame:
    logging.info("Finalizing binned events: %s", in_path)
    if not in_path.exists() or in_path.stat().st_size == 0:
        df = pd.DataFrame(columns=['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind'])
        save_parquet(df, out_path)
        return df
    df = read_parquet(in_path)
    if df.empty:
        save_parquet(df, out_path)
        return df
    df['kind'] = df['variable'].map(lambda v: 'binary' if v in BINARY_VARS else ('sum' if v in ['urine_output', 'fluid_input'] else 'continuous'))
    parts = []
    cont = df.loc[df['kind'] == 'continuous'].copy()
    if not cont.empty:
        cont['weighted'] = cont['value'] * cont['n_obs'].clip(lower=1)
        g = cont.groupby(['stay_id', 'bin', 'variable'], observed=True).agg(
            weighted=('weighted', 'sum'), n_obs=('n_obs', 'sum')
        ).reset_index()
        g['value'] = g['weighted'] / g['n_obs'].clip(lower=1)
        g['kind'] = 'continuous'
        parts.append(g[['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind']])
    sm = df.loc[df['kind'] == 'sum'].copy()
    if not sm.empty:
        g = sm.groupby(['stay_id', 'bin', 'variable'], observed=True).agg(value=('value', 'sum'), n_obs=('n_obs', 'sum')).reset_index()
        g['kind'] = 'sum'
        parts.append(g[['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind']])
    bi = df.loc[df['kind'] == 'binary'].copy()
    if not bi.empty:
        g = bi.groupby(['stay_id', 'bin', 'variable'], observed=True).agg(value=('value', 'max'), n_obs=('n_obs', 'sum')).reset_index()
        g['kind'] = 'binary'
        parts.append(g[['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind']])
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['stay_id', 'bin', 'variable', 'value', 'n_obs', 'kind'])
    out['bin'] = out['bin'].astype(np.int32)
    out['value'] = out['value'].astype(np.float32)
    out['n_obs'] = out['n_obs'].clip(0, 32767).astype(np.int16)
    save_parquet(out, out_path)
    logging.info("Finalized %s rows to %s", len(out), out_path)
    return out


## 8. MIMIC-IV extraction: cohort and item dictionaries

This section builds the MIMIC adult first-ICU-stay base cohort, identifies comorbidity/exclusion flags, and creates item maps from `d_labitems` and `d_items`. It uses the fixed Drive ZIP path and reads directly from the archive.


In [ ]:

# -----------------------------------------------------------------------------
# MIMIC-IV extraction: cohort and item dictionaries
# -----------------------------------------------------------------------------
MIMIC_ITEM_MAP_VERSION = '2026-06-14-id-normalized-blood-labs-v4'


def build_mimic_cohort(cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    """Build the adult first-ICU-stay MIMIC cohort.

    This function intentionally performs all identifier joins after canonicalizing
    ``subject_id``, ``hadm_id`` and ``stay_id`` to stable strings. MIMIC hospital tables
    sometimes contain nullable identifiers; without canonicalization, pandas can turn an
    ID into ``123.0`` in one table and ``123`` in another, silently destroying joins.
    """
    out_path = dirs['cohort'] / 'mimic_cohort_base.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)

    logging.info("Building MIMIC cohort from %s", cfg.mimic_zip)
    store = ZipCSVStore(Path(cfg.mimic_zip))
    try:
        patients = store.read_csv(
            'patients',
            usecols=['subject_id', 'gender', 'anchor_age', 'anchor_year_group', 'dod'],
            parse_dates=['dod'],
        )
        admissions = store.read_csv(
            'admissions',
            usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
                     'admission_type', 'race', 'hospital_expire_flag'],
            parse_dates=['admittime', 'dischtime', 'deathtime'],
        )
        icu = store.read_csv(
            'icustays',
            usecols=['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
                     'intime', 'outtime', 'los'],
            parse_dates=['intime', 'outtime'],
        )

        patients = normalize_id_columns(patients, ['subject_id'])
        admissions = normalize_id_columns(admissions, ['subject_id', 'hadm_id'])
        icu = normalize_id_columns(icu, ['subject_id', 'hadm_id', 'stay_id'])

        icu = icu.sort_values(['subject_id', 'intime']).groupby('subject_id', as_index=False).head(1).copy()
        icu['los_hours'] = (icu['outtime'] - icu['intime']).dt.total_seconds() / 3600.0
        icu['los_hours'] = icu['los_hours'].fillna(series_or_default(icu, 'los', np.nan) * 24.0)
        cohort = icu.merge(patients, on='subject_id', how='left').merge(admissions, on=['subject_id', 'hadm_id'], how='left')

        cohort['age'] = to_num(cohort['anchor_age'])
        cohort['sex_female'] = cohort['gender'].map(sex_female_value)
        cohort['race_ethnicity'] = series_or_default(cohort, 'race', 'unknown').map(normalize_race_ethnicity_value)
        cohort['death_hours'] = np.nan
        m = cohort['deathtime'].notna()
        cohort.loc[m, 'death_hours'] = (cohort.loc[m, 'deathtime'] - cohort.loc[m, 'intime']).dt.total_seconds() / 3600.0
        cohort.loc[(cohort['death_hours'] < 0) | (cohort['death_hours'] > cohort['los_hours'] + 1), 'death_hours'] = np.nan
        cohort['discharge_hours'] = cohort['los_hours']
        cohort['hospital_id'] = 'MIMIC_BIDMC'
        cohort['dataset'] = 'mimic'
        cohort['weight_kg'] = np.nan
        cohort['weight_missing'] = 1.0
        care = cohort['first_careunit'].fillna('').astype(str).str.lower()
        cohort['medical_icu'] = care.str.contains('medical|micu', regex=True).astype(float)
        cohort['surgical_icu'] = care.str.contains('surgical|sicu|trauma|neuro|cardiac', regex=True).astype(float)

        # ICD comorbidities and renal-exclusion flags.
        dx = store.read_csv('diagnoses_icd', usecols=['subject_id', 'hadm_id', 'icd_code', 'icd_version'], required=False)
        if dx is not None and not dx.empty:
            dx = normalize_id_columns(dx, ['subject_id', 'hadm_id'])
            dx['dx_text'] = dx['icd_code'].fillna('').astype(str)
            dx = dx.loc[dx['hadm_id'].isin(set(cohort['hadm_id']))]
            for name, pats in COMORBIDITY_PATTERNS.items():
                flag = dx.loc[str_contains_any(dx['dx_text'], pats)].groupby('hadm_id').size().rename(name)
                cohort = cohort.merge(flag, left_on='hadm_id', right_index=True, how='left')
            excl = dx.loc[str_contains_any(dx['dx_text'], EXCLUSION_PATTERNS)].groupby('hadm_id').size().rename('renal_exclusion')
            cohort = cohort.merge(excl, left_on='hadm_id', right_index=True, how='left')
        for col in list(COMORBIDITY_PATTERNS.keys()) + ['renal_exclusion']:
            if col not in cohort.columns:
                cohort[col] = 0
            cohort[col] = (cohort[col].fillna(0).astype(float) > 0).astype(float)

        cohort = cohort.loc[
            (cohort['age'] >= 18) &
            (cohort['los_hours'] >= cfg.min_los_hours) &
            (cohort['renal_exclusion'] < 0.5)
        ].copy()

        keep = ['dataset', 'stay_id', 'subject_id', 'hadm_id', 'hospital_id', 'intime', 'outtime',
                'los_hours', 'discharge_hours', 'death_hours', 'age', 'sex_female', 'race_ethnicity', 'anchor_year_group',
                'first_careunit', 'medical_icu', 'surgical_icu', 'weight_kg', 'weight_missing'] + list(COMORBIDITY_PATTERNS.keys())
        cohort = cohort[keep].reset_index(drop=True)
        save_parquet(cohort, out_path)
        logging.info("MIMIC base cohort: %s stays", len(cohort))
        return cohort
    finally:
        store.close()


def build_mimic_item_maps(cfg: Config, dirs: Dict[str, Path]) -> Dict[str, Dict[int, str]]:
    """Map MIMIC item IDs to the canonical variables used by the AKI pipeline.

    The creatinine map is deliberately restricted to blood/whole-blood/serum/plasma
    fluids. Earlier notebook versions allowed CSF, stool and joint-fluid creatinine
    labels through the fluid filter; this did not usually create the empty-landmark
    failure, but it was clinically incorrect and is fixed here.
    """
    out_path = dirs['cache'] / 'mimic_item_maps.json'
    version_path = dirs['cache'] / 'mimic_item_maps.version'
    if out_path.exists() and version_path.exists() and version_path.read_text().strip() == MIMIC_ITEM_MAP_VERSION and not cfg.overwrite:
        with open(out_path, 'r') as f:
            raw = json.load(f)
        return {k: {int(kk): vv for kk, vv in v.items()} for k, v in raw.items()}

    store = ZipCSVStore(Path(cfg.mimic_zip))
    try:
        d_lab = store.read_csv('d_labitems', usecols=['itemid', 'label', 'fluid', 'category'])
        d_items = store.read_csv('d_items', usecols=['itemid', 'label', 'category', 'unitname', 'linksto'])
        d_lab['itemid'] = to_num(d_lab['itemid']).astype('Int64')
        d_items['itemid'] = to_num(d_items['itemid']).astype('Int64')
        d_lab = d_lab.dropna(subset=['itemid']).copy()
        d_items = d_items.dropna(subset=['itemid']).copy()
        d_lab['itemid'] = d_lab['itemid'].astype(int)
        d_items['itemid'] = d_items['itemid'].astype(int)

        fluid = series_or_default(d_lab, 'fluid', '').fillna('').astype(str).str.lower()
        # Keep blood-derived labs; drop urine/body-fluid/stool/CSF/joint/pleural/ascites specimens.
        blood_like = fluid.str.contains(r'\bblood\b|whole blood|serum|plasma', regex=True, na=False) | fluid.eq('')
        nonblood_bad = fluid.str.contains(r'urine|body fluid|ascites|pleural|joint|cerebrospinal|csf|stool|dialysate', regex=True, na=False)
        fluid_ok = blood_like & ~nonblood_bad
        lab_map = match_by_patterns(d_lab, 'label', LAB_PATTERNS, exclude_patterns=[r'urine', r'body fluid'], extra_filter=fluid_ok)

        creatinine_mapped_ids = sorted([itemid for itemid, var_name in lab_map.items() if var_name == 'creatinine'])
        logging.info("MIMIC blood creatinine item IDs: %s", creatinine_mapped_ids)
        if not creatinine_mapped_ids:
            # Hard fail early rather than spending hours extracting a dataset that cannot label AKI.
            potential = d_lab.loc[str_contains_any(d_lab['label'], LAB_PATTERNS['creatinine']), ['itemid', 'label', 'fluid', 'category']]
            raise RuntimeError("No blood creatinine item IDs were mapped from d_labitems. Potential rows:\n" + potential.head(30).to_string(index=False))

        chart_map = match_by_patterns(d_items, 'label', CHART_PATTERNS)
        linksto = series_or_default(d_items, 'linksto', '').fillna('').astype(str).str.lower()
        output_scope = linksto.eq('outputevents') | linksto.eq('')
        output_candidates = d_items.loc[output_scope & str_contains_any(d_items['label'], URINE_OUTPUT_PATTERNS)].copy()
        output_map = {int(i): 'urine_output' for i in output_candidates['itemid'].tolist()}
        logging.info("MIMIC urine-output item IDs: %s", sorted(output_map.keys())[:50])
        if not output_map:
            raise RuntimeError("No urine-output item IDs were mapped from d_items/outputevents.")

        input_map: Dict[int, str] = {}
        for itemid, label in zip(d_items['itemid'], d_items['label'].fillna('')):
            exps = map_drug_to_exposures(label)
            if exps:
                input_map[int(itemid)] = '|'.join(exps)

        proc_map: Dict[int, str] = {}
        for itemid, label in zip(d_items['itemid'], d_items['label'].fillna('')):
            exps = map_treatment_to_exposures(label)
            if exps:
                proc_map[int(itemid)] = '|'.join(exps)

        maps = {'lab': lab_map, 'chart': chart_map, 'output': output_map, 'input': input_map, 'procedure': proc_map}
        with open(out_path, 'w') as f:
            json.dump({k: {str(kk): vv for kk, vv in v.items()} for k, v in maps.items()}, f, indent=2)
        version_path.write_text(MIMIC_ITEM_MAP_VERSION)
        logging.info("MIMIC item maps: %s", {k: len(v) for k, v in maps.items()})
        return maps
    finally:
        store.close()


## 9. MIMIC-IV extraction: baseline creatinine, weight, events, medications, RRT, ventilation

This section extracts baseline serum creatinine and early weight, then reads MIMIC lab, chart, output, input, prescription, and procedure tables to construct binned ICU trajectories. The RRT and medication exposure logic supports the AKI/RRT primary outcome and the PSM/nephrotoxin sensitivity analysis.


In [ ]:
def extract_mimic_baseline_and_weight(cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame, item_maps: Dict[str, Dict[int, str]]) -> pd.DataFrame:
    out_path = dirs['cohort'] / 'mimic_baseline_weight.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)

    logging.info("Extracting MIMIC baseline creatinine and admission weight")
    store = ZipCSVStore(Path(cfg.mimic_zip))
    try:
        cohort_small = cohort[['stay_id', 'subject_id', 'hadm_id', 'intime']].copy()
        cohort_small = normalize_id_columns(cohort_small, ['stay_id', 'subject_id', 'hadm_id'])
        creat_ids = [i for i, v in item_maps['lab'].items() if v == 'creatinine']
        scr_rows = []
        if creat_ids:
            usecols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum']
            reader = store.read_csv('labevents', usecols=usecols, parse_dates=['charttime'], chunksize=cfg.chunksize)
            for chunk in tqdm(reader, desc='MIMIC baseline sCr'):
                chunk = chunk.loc[chunk['itemid'].isin(creat_ids)].copy()
                if chunk.empty:
                    continue
                chunk = normalize_id_columns(chunk, ['subject_id', 'hadm_id'])
                chunk['valuenum'] = clip_values('creatinine', chunk['valuenum'])
                chunk = chunk.dropna(subset=['valuenum', 'charttime'])
                merged = chunk.merge(cohort_small[['stay_id', 'subject_id', 'hadm_id', 'intime']], on='subject_id', how='inner', suffixes=('', '_cohort'))
                if merged.empty:
                    continue
                merged['chart_hour'] = (merged['charttime'] - merged['intime']).dt.total_seconds() / 3600.0
                merged['same_hadm'] = merged['hadm_id'].astype(str).eq(merged['hadm_id_cohort'].astype(str))
                merged = merged.loc[((merged['chart_hour'] < 0) | merged['same_hadm']) & (merged['chart_hour'] >= -cfg.baseline_lookback_days * 24) & (merged['chart_hour'] <= cfg.baseline_admission_fallback_hours)]
                if not merged.empty:
                    scr_rows.append(merged[['stay_id', 'chart_hour', 'valuenum']])
        if scr_rows:
            scr = pd.concat(scr_rows, ignore_index=True)
            scr = scr.sort_values(['stay_id', 'chart_hour'])
            prior = scr.loc[scr['chart_hour'] < 0].groupby('stay_id', as_index=False).tail(1).set_index('stay_id')['valuenum']
            admission_fallback_h = int(cfg.baseline_admission_fallback_hours)
            early = scr.loc[(scr['chart_hour'] >= 0) & (scr['chart_hour'] <= admission_fallback_h)].groupby('stay_id')['valuenum'].min()
            first = scr.loc[(scr['chart_hour'] >= 0) & (scr['chart_hour'] <= admission_fallback_h)].groupby('stay_id', as_index=False).head(1).set_index('stay_id')['valuenum']
            baseline = pd.DataFrame(index=cohort['stay_id'])
            baseline['prior_scr'] = prior
            baseline['early_min_scr'] = early
            baseline['first_scr'] = first
            baseline['baseline_scr'] = baseline['prior_scr'].combine_first(baseline['early_min_scr']).combine_first(baseline['first_scr'])
            admission_source = f'early_min_0_{admission_fallback_h}h'
            first_source = f'first_observed_0_{admission_fallback_h}h'
            baseline['baseline_source'] = np.select(
                [baseline['prior_scr'].notna(), baseline['early_min_scr'].notna(), baseline['first_scr'].notna()],
                ['prior', admission_source, first_source], default='missing'
            )
            baseline = baseline.reset_index().rename(columns={'index': 'stay_id'})
        else:
            baseline = pd.DataFrame({'stay_id': cohort['stay_id'], 'baseline_scr': np.nan, 'baseline_source': 'missing'})

        # Admission / early weight from chartevents item labels.
        weight_ids = [i for i, v in item_maps['chart'].items() if v == 'weight_kg']
        w_rows = []
        if weight_ids:
            usecols = ['stay_id', 'itemid', 'charttime', 'valuenum']
            reader = store.read_csv('chartevents', usecols=usecols, parse_dates=['charttime'], chunksize=cfg.chunksize)
            time_ref = cohort[['stay_id', 'intime']].copy()
            time_ref = normalize_id_columns(time_ref, ['stay_id'])
            for chunk in tqdm(reader, desc='MIMIC weight'):
                chunk = chunk.loc[chunk['itemid'].isin(weight_ids)].copy()
                if chunk.empty:
                    continue
                chunk = normalize_id_columns(chunk, ['stay_id'])
                chunk['valuenum'] = clip_values('weight_kg', chunk['valuenum'])
                chunk = chunk.dropna(subset=['valuenum', 'charttime'])
                merged = chunk.merge(time_ref, on='stay_id', how='inner')
                merged['chart_hour'] = (merged['charttime'] - merged['intime']).dt.total_seconds() / 3600.0
                merged = merged.loc[(merged['chart_hour'] >= -6) & (merged['chart_hour'] <= 24)]
                if not merged.empty:
                    w_rows.append(merged[['stay_id', 'chart_hour', 'valuenum']])
        if w_rows:
            weight = pd.concat(w_rows, ignore_index=True).sort_values(['stay_id', 'chart_hour'])
            weight = weight.groupby('stay_id', as_index=False).head(1).rename(columns={'valuenum': 'weight_kg'})[['stay_id', 'weight_kg']]
        else:
            weight = pd.DataFrame({'stay_id': cohort['stay_id'], 'weight_kg': np.nan})

        out = cohort[['stay_id']].merge(baseline[['stay_id', 'baseline_scr', 'baseline_source']], on='stay_id', how='left').merge(weight, on='stay_id', how='left')
        save_parquet(out, out_path)
        return out
    finally:
        store.close()


def expand_mimic_exposure_rows(df: pd.DataFrame, item_map: Dict[int, str], item_col: str = 'itemid') -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=list(df.columns) + ['variable'])
    rows = []
    for itemid, sub in df.groupby(item_col):
        labels = item_map.get(int(itemid), '')
        if not labels:
            continue
        for var in labels.split('|'):
            z = sub.copy()
            z['variable'] = var
            rows.append(z)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=list(df.columns) + ['variable'])


def extract_mimic_events(cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame, item_maps: Dict[str, Dict[int, str]]) -> Path:
    raw_path = dirs['events'] / 'mimic_binned_events_raw.parquet'
    final_path = dirs['events'] / 'mimic_binned_events.parquet'
    if final_path.exists() and not cfg.overwrite:
        # Do not reuse a large but incompatible event cache from older notebook
        # revisions. In the reported failure this file existed with ~17M rows,
        # but it contained no usable creatinine stream after ID joins.
        if '_event_file_status' in globals():
            ok, reason = _event_file_status('mimic', dirs)
            if ok:
                return final_path
            logging.warning("Rebuilding cached MIMIC event table because it is invalid: %s", reason)
            _unlink_quietly(final_path) if '_unlink_quietly' in globals() else final_path.unlink(missing_ok=True)
            _unlink_quietly(raw_path) if '_unlink_quietly' in globals() else raw_path.unlink(missing_ok=True)
        else:
            return final_path

    logging.info("Extracting MIMIC binned events")
    store = ZipCSVStore(Path(cfg.mimic_zip))
    time_by_stay = cohort[['stay_id', 'subject_id', 'hadm_id', 'intime', 'outtime', 'los_hours']].copy()
    time_by_stay = normalize_id_columns(time_by_stay, ['stay_id', 'subject_id', 'hadm_id'])
    stay_ids = set(time_by_stay['stay_id'])
    hadm_ref = time_by_stay[['stay_id', 'subject_id', 'hadm_id', 'intime', 'los_hours']]
    stay_ref = time_by_stay[['stay_id', 'intime', 'los_hours']]

    with ParquetSink(raw_path) as sink:
        try:
            # Labs.
            lab_ids = set(item_maps['lab'].keys())
            if lab_ids:
                reader = store.read_csv('labevents', usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'], parse_dates=['charttime'], chunksize=cfg.chunksize)
                for chunk in tqdm(reader, desc='MIMIC labs'):
                    chunk = chunk.loc[chunk['itemid'].isin(lab_ids)].copy()
                    if chunk.empty:
                        continue
                    chunk = normalize_id_columns(chunk, ['subject_id', 'hadm_id'])
                    chunk['variable'] = chunk['itemid'].map(item_maps['lab'])
                    chunk['value'] = to_num(chunk['valuenum'])
                    for var in chunk['variable'].dropna().unique():
                        m = chunk['variable'] == var
                        chunk.loc[m, 'value'] = clip_values(str(var), chunk.loc[m, 'value'])
                    chunk = chunk.dropna(subset=['variable', 'value', 'charttime'])
                    merged = chunk.merge(hadm_ref[['stay_id', 'subject_id', 'hadm_id', 'intime', 'los_hours']], on=['subject_id', 'hadm_id'], how='inner')
                    if merged.empty:
                        continue
                    merged['hour'] = (merged['charttime'] - merged['intime']).dt.total_seconds() / 3600.0
                    merged = merged.loc[(merged['hour'] >= 0) & (merged['hour'] <= merged['los_hours'].clip(upper=cfg.max_icu_hours))]
                    sink.write(make_binned_events(merged[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk, merged
                    gc.collect()

            # Charted vitals.
            chart_ids = set(item_maps['chart'].keys())
            if chart_ids:
                reader = store.read_csv('chartevents', usecols=['stay_id', 'itemid', 'charttime', 'valuenum'], parse_dates=['charttime'], chunksize=cfg.chunksize)
                for chunk in tqdm(reader, desc='MIMIC chartevents'):
                    chunk = chunk.loc[chunk['itemid'].isin(chart_ids)].copy()
                    if chunk.empty:
                        continue
                    chunk = normalize_id_columns(chunk, ['stay_id'])
                    chunk = chunk.loc[chunk['stay_id'].isin(stay_ids)]
                    chunk['variable'] = chunk['itemid'].map(item_maps['chart'])
                    chunk['value'] = to_num(chunk['valuenum'])
                    for var in chunk['variable'].dropna().unique():
                        m = chunk['variable'] == var
                        chunk.loc[m, 'value'] = clip_values(str(var), chunk.loc[m, 'value'])
                    chunk = chunk.dropna(subset=['variable', 'value', 'charttime'])
                    merged = chunk.merge(stay_ref, on='stay_id', how='inner')
                    merged['hour'] = (merged['charttime'] - merged['intime']).dt.total_seconds() / 3600.0
                    merged = merged.loc[(merged['hour'] >= 0) & (merged['hour'] <= merged['los_hours'].clip(upper=cfg.max_icu_hours))]
                    sink.write(make_binned_events(merged[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk, merged
                    gc.collect()

            # Urine output.
            output_ids = set(item_maps['output'].keys())
            if output_ids:
                reader = store.read_csv('outputevents', usecols=['stay_id', 'itemid', 'charttime', 'value', 'valueuom'], parse_dates=['charttime'], chunksize=cfg.chunksize)
                for chunk in tqdm(reader, desc='MIMIC outputevents'):
                    chunk = chunk.loc[chunk['itemid'].isin(output_ids)].copy()
                    if chunk.empty:
                        continue
                    chunk = normalize_id_columns(chunk, ['stay_id'])
                    chunk = chunk.loc[chunk['stay_id'].isin(stay_ids)]
                    chunk['variable'] = 'urine_output'
                    chunk['value'] = clip_values('urine_output', chunk['value'])
                    chunk = chunk.dropna(subset=['value', 'charttime'])
                    merged = chunk.merge(stay_ref, on='stay_id', how='inner')
                    merged['hour'] = (merged['charttime'] - merged['intime']).dt.total_seconds() / 3600.0
                    merged = merged.loc[(merged['hour'] >= 0) & (merged['hour'] <= merged['los_hours'].clip(upper=cfg.max_icu_hours))]
                    sink.write(make_binned_events(merged[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk, merged
                    gc.collect()

            # Fluid inputs and infusion/procedure exposures.
            input_map = item_maps['input']
            reader = store.read_csv('inputevents', usecols=['stay_id', 'itemid', 'starttime', 'endtime', 'amount', 'amountuom', 'rate', 'rateuom'], parse_dates=['starttime', 'endtime'], chunksize=cfg.chunksize, required=False)
            if reader is not None:
                for chunk in tqdm(reader, desc='MIMIC inputevents'):
                    chunk = normalize_id_columns(chunk, ['stay_id'])
                    chunk = chunk.loc[chunk['stay_id'].isin(stay_ids)].copy()
                    if chunk.empty:
                        continue
                    merged = chunk.merge(stay_ref, on='stay_id', how='inner')
                    # Fluid input as amount in mL at endtime.
                    fluid = merged.copy()
                    fluid['uom'] = series_or_default(fluid, 'amountuom', '').fillna('').astype(str).str.lower()
                    fluid['value'] = clip_values('fluid_input', series_or_default(fluid, 'amount', np.nan))
                    fluid = fluid.loc[fluid['value'].notna() & fluid['uom'].str.contains('ml|milliliter', regex=True, na=False)]
                    if not fluid.empty:
                        fluid['variable'] = 'fluid_input'
                        fluid['hour'] = (fluid['endtime'].fillna(fluid['starttime']) - fluid['intime']).dt.total_seconds() / 3600.0
                        fluid = fluid.loc[(fluid['hour'] >= 0) & (fluid['hour'] <= fluid['los_hours'].clip(upper=cfg.max_icu_hours))]
                        sink.write(make_binned_events(fluid[['stay_id', 'hour', 'variable', 'value']], cfg))
                    # Binary exposures from itemid; expand intervals to grid bins.
                    exp = merged.loc[merged['itemid'].isin(set(input_map.keys()))].copy()
                    if not exp.empty:
                        exp = expand_mimic_exposure_rows(exp, input_map)
                        if not exp.empty:
                            exp['start_hour'] = (exp['starttime'] - exp['intime']).dt.total_seconds() / 3600.0
                            exp['end_hour'] = (exp['endtime'].fillna(exp['starttime']) - exp['intime']).dt.total_seconds() / 3600.0
                            exp['end_hour'] = exp[['start_hour', 'end_hour']].max(axis=1) + cfg.grid_hours
                            rows = []
                            for sid, sh, eh, var in exp[['stay_id', 'start_hour', 'end_hour', 'variable']].itertuples(index=False):
                                if pd.isna(sh):
                                    continue
                                sh = max(0.0, float(sh)); eh = min(float(eh), cfg.max_icu_hours)
                                if eh < 0 or sh > cfg.max_icu_hours:
                                    continue
                                bins = range(int(math.floor(sh / cfg.grid_hours)), int(math.ceil(eh / cfg.grid_hours)) + 1)
                                for b in bins:
                                    if 0 <= b < cfg.max_grid_steps:
                                        rows.append((sid, b * cfg.grid_hours, var, 1.0))
                            if rows:
                                exp_ev = pd.DataFrame(rows, columns=['stay_id', 'hour', 'variable', 'value'])
                                sink.write(make_binned_events(exp_ev, cfg))
                    del chunk, merged
                    gc.collect()

            # Prescriptions for nephrotoxins / antibiotics / diuretics.
            presc = store.read_csv('prescriptions', usecols=['subject_id', 'hadm_id', 'drug', 'starttime', 'stoptime'], parse_dates=['starttime', 'stoptime'], chunksize=cfg.chunksize, required=False)
            if presc is not None:
                for chunk in tqdm(presc, desc='MIMIC prescriptions'):
                    chunk = normalize_id_columns(chunk, ['subject_id', 'hadm_id'])
                    chunk['exposures'] = chunk['drug'].map(map_drug_to_exposures)
                    chunk = chunk.loc[chunk['exposures'].map(len) > 0].copy()
                    if chunk.empty:
                        continue
                    merged = chunk.merge(hadm_ref[['stay_id', 'subject_id', 'hadm_id', 'intime', 'los_hours']], on=['subject_id', 'hadm_id'], how='inner')
                    if merged.empty:
                        continue
                    rows = []
                    for r in merged.itertuples(index=False):
                        start = getattr(r, 'starttime')
                        stop = getattr(r, 'stoptime')
                        if pd.isna(start):
                            continue
                        if pd.isna(stop):
                            stop = start + pd.Timedelta(hours=cfg.grid_hours)
                        sh = (start - r.intime).total_seconds() / 3600.0
                        eh = (stop - r.intime).total_seconds() / 3600.0
                        eh = max(eh, sh + cfg.grid_hours)
                        for var in r.exposures:
                            for b in range(int(math.floor(max(0, sh) / cfg.grid_hours)), int(math.ceil(min(cfg.max_icu_hours, eh) / cfg.grid_hours)) + 1):
                                if 0 <= b < cfg.max_grid_steps:
                                    rows.append((r.stay_id, b * cfg.grid_hours, var, 1.0))
                    if rows:
                        sink.write(make_binned_events(pd.DataFrame(rows, columns=['stay_id', 'hour', 'variable', 'value']), cfg))
                    del chunk, merged
                    gc.collect()

            # Procedures for RRT / mechanical ventilation.
            proc_map = item_maps['procedure']
            proc = store.read_csv('procedureevents', usecols=['stay_id', 'itemid', 'starttime', 'endtime'], parse_dates=['starttime', 'endtime'], chunksize=cfg.chunksize, required=False)
            if proc is not None and proc_map:
                for chunk in tqdm(proc, desc='MIMIC procedureevents'):
                    chunk = chunk.loc[chunk['itemid'].isin(set(proc_map.keys()))].copy()
                    if chunk.empty:
                        continue
                    chunk = normalize_id_columns(chunk, ['stay_id'])
                    chunk = chunk.loc[chunk['stay_id'].isin(stay_ids)]
                    merged = chunk.merge(stay_ref, on='stay_id', how='inner')
                    exp = expand_mimic_exposure_rows(merged, proc_map)
                    rows = []
                    for r in exp.itertuples(index=False):
                        start = r.starttime
                        stop = r.endtime if not pd.isna(r.endtime) else r.starttime + pd.Timedelta(hours=cfg.grid_hours)
                        if pd.isna(start):
                            continue
                        sh = (start - r.intime).total_seconds() / 3600.0
                        eh = max((stop - r.intime).total_seconds() / 3600.0, sh + cfg.grid_hours)
                        for b in range(int(math.floor(max(0, sh) / cfg.grid_hours)), int(math.ceil(min(cfg.max_icu_hours, eh) / cfg.grid_hours)) + 1):
                            if 0 <= b < cfg.max_grid_steps:
                                rows.append((r.stay_id, b * cfg.grid_hours, r.variable, 1.0))
                    if rows:
                        sink.write(make_binned_events(pd.DataFrame(rows, columns=['stay_id', 'hour', 'variable', 'value']), cfg))
                    del chunk, merged, exp
                    gc.collect()
        finally:
            store.close()
    final_df = finalize_binned_events(raw_path, final_path)
    _validate_event_critical_coverage('mimic', final_df, cohort, cfg, require_overlap=False)
    return final_path


## 10. eICU extraction: cohort and text flags

This section builds the eICU adult first-ICU-stay base cohort, keeps the `hospital_id` structure needed for clustered/external validation, and extracts comorbidity and renal-exclusion flags from diagnosis, past-history, and treatment text.


In [ ]:
# -----------------------------------------------------------------------------
# eICU extraction
# -----------------------------------------------------------------------------
def build_eicu_text_flags(cfg: Config, store: ZipCSVStore, ids: set) -> pd.DataFrame:
    flags = pd.DataFrame({'stay_id': list(ids)})
    flags['stay_id'] = flags['stay_id'].astype(str)
    for col in list(COMORBIDITY_PATTERNS.keys()) + ['renal_exclusion']:
        flags[col] = 0.0

    def process_table(table: str, id_col: str, text_cols: List[str], chunksize: int = 400_000) -> None:
        nonlocal flags
        if store.find(table, required=False) is None:
            return
        usecols = [id_col] + text_cols
        reader = store.read_csv(table, usecols=usecols, chunksize=chunksize, required=False)
        found = defaultdict(set)
        for chunk in tqdm(reader, desc=f'eICU flags {table}'):
            chunk = normalize_cols_lower(chunk)
            if id_col.lower() not in chunk.columns:
                continue
            chunk['stay_id'] = chunk[id_col.lower()].astype(str)
            chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
            if chunk.empty:
                continue
            text = pd.Series('', index=chunk.index, dtype='object')
            for c in text_cols:
                cc = c.lower()
                if cc in chunk.columns:
                    text = text + ' ' + chunk[cc].fillna('').astype(str)
            for name, pats in COMORBIDITY_PATTERNS.items():
                for sid in chunk.loc[str_contains_any(text, pats), 'stay_id'].unique():
                    found[name].add(sid)
            for sid in chunk.loc[str_contains_any(text, EXCLUSION_PATTERNS), 'stay_id'].unique():
                found['renal_exclusion'].add(sid)
        for name, sids in found.items():
            flags.loc[flags['stay_id'].isin(sids), name] = 1.0

    process_table('diagnosis', 'patientunitstayid', ['diagnosisstring', 'icd9code'])
    process_table('pastHistory', 'patientunitstayid', ['pasthistorypath', 'pasthistoryvalue'])
    process_table('treatment', 'patientunitstayid', ['treatmentstring'])
    return flags


def build_eicu_cohort(cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    out_path = dirs['cohort'] / 'eicu_cohort_base.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)
    logging.info("Building eICU cohort from %s", cfg.eicu_zip)
    store = ZipCSVStore(Path(cfg.eicu_zip))
    try:
        patient = store.read_csv('patient')
        patient = normalize_cols_lower(patient)
        id_col = get_col(patient, ['patientunitstayid'])
        patient['stay_id'] = patient[id_col].astype(str)
        patient['subject_id'] = patient.get('uniquepid', patient['stay_id']).astype(str)
        patient['hadm_id'] = patient.get('patienthealthsystemstayid', patient['stay_id']).astype(str)
        patient['hospital_id'] = patient.get('hospitalid', 'missing').astype(str)
        patient['age'] = patient.get('age', np.nan).map(parse_eicu_age)
        patient['sex_female'] = patient.get('gender', np.nan).map(sex_female_value)
        patient['race_ethnicity'] = series_or_default(patient, 'ethnicity', 'unknown').map(normalize_race_ethnicity_value)
        los_col = get_col(patient, ['unitdischargeoffset'])
        if los_col is None:
            raise KeyError('eICU patient table must contain unitdischargeoffset.')
        patient['los_hours'] = to_num(patient[los_col]) / 60.0
        patient['discharge_hours'] = patient['los_hours']
        status = series_or_default(patient, 'unitdischargestatus', '').fillna('').astype(str).str.lower()
        patient['death_hours'] = np.where(status.str.contains('expired|death|dead', regex=True), patient['los_hours'], np.nan)
        patient['weight_kg'] = clip_values('weight_kg', series_or_default(patient, 'admissionweight', np.nan))
        patient['weight_missing'] = patient['weight_kg'].isna().astype(float)
        unit = series_or_default(patient, 'unittype', '').fillna('').astype(str).str.lower()
        adx = series_or_default(patient, 'apacheadmissiondx', '').fillna('').astype(str).str.lower()
        patient['medical_icu'] = (unit.str.contains('med|micu', regex=True) | adx.str.contains('medical', regex=False)).astype(float)
        patient['surgical_icu'] = (unit.str.contains('surg|sicu|cardiac|neuro|trauma', regex=True) | adx.str.contains('surg', regex=True)).astype(float)
        # First ICU stay per unique patient if unitvisitnumber is available.
        if 'unitvisitnumber' in patient.columns:
            patient['unitvisitnumber_num'] = to_num(patient['unitvisitnumber'])
            patient = patient.sort_values(['subject_id', 'unitvisitnumber_num', 'stay_id']).groupby('subject_id', as_index=False).head(1)
        else:
            patient = patient.sort_values(['subject_id', 'stay_id']).groupby('subject_id', as_index=False).head(1)

        ids = set(patient['stay_id'])
        flags = build_eicu_text_flags(cfg, store, ids)
        patient = patient.merge(flags, on='stay_id', how='left')
        for col in list(COMORBIDITY_PATTERNS.keys()) + ['renal_exclusion']:
            patient[col] = patient[col].fillna(0).astype(float)

        patient['dataset'] = 'eicu'
        patient['anchor_year_group'] = '2014-2015'
        patient = patient.loc[
            (patient['age'] >= 18) &
            (patient['los_hours'] >= cfg.min_los_hours) &
            (patient['renal_exclusion'] < 0.5)
        ].copy()
        keep = ['dataset', 'stay_id', 'subject_id', 'hadm_id', 'hospital_id', 'los_hours', 'discharge_hours',
                'death_hours', 'age', 'sex_female', 'race_ethnicity', 'anchor_year_group', 'medical_icu', 'surgical_icu',
                'weight_kg', 'weight_missing'] + list(COMORBIDITY_PATTERNS.keys())
        cohort = patient[keep].reset_index(drop=True)
        save_parquet(cohort, out_path)
        logging.info("eICU base cohort: %s stays", len(cohort))
        return cohort
    finally:
        store.close()


## 11. eICU extraction: baseline creatinine, vitals, labs, intake/output, medications, RRT, ventilation

This section harmonizes eICU tables into the same binned-event format used for MIMIC. It reads `lab`, `vitalPeriodic`, `vitalAperiodic`, `intakeOutput`, `medication`, `infusionDrug`, and `treatment` when present.


In [ ]:
def extract_eicu_baseline(cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame) -> pd.DataFrame:
    out_path = dirs['cohort'] / 'eicu_baseline.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)
    logging.info("Extracting eICU baseline creatinine")
    store = ZipCSVStore(Path(cfg.eicu_zip))
    try:
        ids = set(cohort['stay_id'].astype(str))
        rows = []
        reader = store.read_csv('lab', usecols=['patientunitstayid', 'labresultoffset', 'labname', 'labresult'], chunksize=cfg.chunksize)
        for chunk in tqdm(reader, desc='eICU baseline sCr'):
            chunk = normalize_cols_lower(chunk)
            chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
            chunk = chunk.loc[chunk['stay_id'].isin(ids)]
            if chunk.empty:
                continue
            m = str_contains_any(chunk['labname'], LAB_PATTERNS['creatinine']) & ~str_contains_any(chunk['labname'], [r'urine'])
            chunk = chunk.loc[m].copy()
            if chunk.empty:
                continue
            chunk['chart_hour'] = to_num(chunk['labresultoffset']) / 60.0
            chunk['valuenum'] = clip_values('creatinine', chunk['labresult'])
            chunk = chunk.dropna(subset=['chart_hour', 'valuenum'])
            chunk = chunk.loc[(chunk['chart_hour'] >= -cfg.baseline_lookback_days * 24) & (chunk['chart_hour'] <= cfg.baseline_admission_fallback_hours)]
            if not chunk.empty:
                rows.append(chunk[['stay_id', 'chart_hour', 'valuenum']])
        if rows:
            scr = pd.concat(rows, ignore_index=True).sort_values(['stay_id', 'chart_hour'])
            prior = scr.loc[scr['chart_hour'] < 0].groupby('stay_id', as_index=False).tail(1).set_index('stay_id')['valuenum']
            admission_fallback_h = int(cfg.baseline_admission_fallback_hours)
            early = scr.loc[(scr['chart_hour'] >= 0) & (scr['chart_hour'] <= admission_fallback_h)].groupby('stay_id')['valuenum'].min()
            first = scr.loc[(scr['chart_hour'] >= 0) & (scr['chart_hour'] <= admission_fallback_h)].groupby('stay_id', as_index=False).head(1).set_index('stay_id')['valuenum']
            baseline = pd.DataFrame(index=cohort['stay_id'])
            baseline['prior_scr'] = prior
            baseline['early_min_scr'] = early
            baseline['first_scr'] = first
            baseline['baseline_scr'] = baseline['prior_scr'].combine_first(baseline['early_min_scr']).combine_first(baseline['first_scr'])
            admission_source = f'early_min_0_{admission_fallback_h}h'
            first_source = f'first_observed_0_{admission_fallback_h}h'
            baseline['baseline_source'] = np.select(
                [baseline['prior_scr'].notna(), baseline['early_min_scr'].notna(), baseline['first_scr'].notna()],
                ['prior', admission_source, first_source], default='missing'
            )
            baseline = baseline.reset_index().rename(columns={'index': 'stay_id'})
        else:
            baseline = pd.DataFrame({'stay_id': cohort['stay_id'], 'baseline_scr': np.nan, 'baseline_source': 'missing'})
        save_parquet(baseline[['stay_id', 'baseline_scr', 'baseline_source']], out_path)
        return baseline[['stay_id', 'baseline_scr', 'baseline_source']]
    finally:
        store.close()


def melt_eicu_vitals(chunk: pd.DataFrame, cfg: Config, ids: set, table_name: str) -> pd.DataFrame:
    chunk = normalize_cols_lower(chunk)
    if 'patientunitstayid' not in chunk.columns:
        return pd.DataFrame(columns=['stay_id', 'hour', 'variable', 'value'])
    chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
    chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
    if chunk.empty:
        return pd.DataFrame(columns=['stay_id', 'hour', 'variable', 'value'])
    offset_col = 'observationoffset' if 'observationoffset' in chunk.columns else None
    if offset_col is None:
        return pd.DataFrame(columns=['stay_id', 'hour', 'variable', 'value'])
    var_cols = {
        'heartrate': 'heart_rate',
        'heart_rate': 'heart_rate',
        'respiration': 'resp_rate',
        'respiratoryrate': 'resp_rate',
        'sao2': 'spo2',
        'spo2': 'spo2',
        'temperature': 'temperature',
        'systemicmean': 'map',
        'systemic_mean': 'map',
        'noninvasivemean': 'map',
        'noninvasive_mean': 'map',
        'nibpmean': 'map',
    }
    present = [(c, v) for c, v in var_cols.items() if c in chunk.columns]
    rows = []
    for col, var in present:
        z = chunk[['stay_id', offset_col, col]].copy()
        z['variable'] = var
        z['hour'] = to_num(z[offset_col]) / 60.0
        z['value'] = clip_values(var, z[col])
        z = z.dropna(subset=['hour', 'value'])
        rows.append(z[['stay_id', 'hour', 'variable', 'value']])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['stay_id', 'hour', 'variable', 'value'])


def extract_eicu_events(cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame) -> Path:
    raw_path = dirs['events'] / 'eicu_binned_events_raw.parquet'
    final_path = dirs['events'] / 'eicu_binned_events.parquet'
    if final_path.exists() and not cfg.overwrite:
        return final_path
    logging.info("Extracting eICU binned events")
    store = ZipCSVStore(Path(cfg.eicu_zip))
    ids = set(cohort['stay_id'].astype(str))
    los_map = cohort.set_index('stay_id')['los_hours'].to_dict()

    with ParquetSink(raw_path) as sink:
        try:
            # Labs.
            reader = store.read_csv('lab', usecols=['patientunitstayid', 'labresultoffset', 'labname', 'labresult'], chunksize=cfg.chunksize)
            for chunk in tqdm(reader, desc='eICU labs'):
                chunk = normalize_cols_lower(chunk)
                chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
                chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
                if chunk.empty:
                    continue
                chunk['variable'] = chunk['labname'].map(lambda x: map_name_to_var(x, LAB_PATTERNS))
                chunk = chunk.loc[chunk['variable'].notna()].copy()
                chunk = chunk.loc[~str_contains_any(chunk['labname'], [r'urine', r'body fluid'])]
                if chunk.empty:
                    continue
                chunk['hour'] = to_num(chunk['labresultoffset']) / 60.0
                chunk['value'] = to_num(chunk['labresult'])
                for var in chunk['variable'].dropna().unique():
                    m = chunk['variable'] == var
                    chunk.loc[m, 'value'] = clip_values(str(var), chunk.loc[m, 'value'])
                chunk['los_hours'] = chunk['stay_id'].map(los_map)
                chunk = chunk.loc[(chunk['hour'] >= 0) & (chunk['hour'] <= chunk['los_hours'].clip(upper=cfg.max_icu_hours))]
                sink.write(make_binned_events(chunk[['stay_id', 'hour', 'variable', 'value']], cfg))
                del chunk
                gc.collect()

            # Vitals.
            for table in ['vitalPeriodic', 'vitalAperiodic']:
                if store.find(table, required=False) is None:
                    continue
                reader = store.read_csv(table, chunksize=cfg.chunksize, required=False)
                for chunk in tqdm(reader, desc=f'eICU {table}'):
                    ev = melt_eicu_vitals(chunk, cfg, ids, table)
                    if ev.empty:
                        continue
                    ev['los_hours'] = ev['stay_id'].map(los_map)
                    ev = ev.loc[(ev['hour'] >= 0) & (ev['hour'] <= ev['los_hours'].clip(upper=cfg.max_icu_hours))]
                    sink.write(make_binned_events(ev[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del ev, chunk
                    gc.collect()

            # Intake/output.
            if store.find('intakeOutput', required=False) is not None:
                reader = store.read_csv('intakeOutput', chunksize=cfg.chunksize, required=False)
                for chunk in tqdm(reader, desc='eICU intakeOutput'):
                    chunk = normalize_cols_lower(chunk)
                    if 'patientunitstayid' not in chunk.columns:
                        continue
                    chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
                    chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
                    if chunk.empty:
                        continue
                    offset_col = get_col(chunk, ['intakeoutputoffset', 'intakeoutputoffset']) or 'intakeoutputoffset'
                    value_col = get_col(chunk, ['cellvaluenumeric', 'cellvalue', 'cellvalue_numeric'])
                    if value_col is None or offset_col not in chunk.columns:
                        continue
                    text = series_or_default(chunk, 'celllabel', '').fillna('').astype(str) + ' ' + series_or_default(chunk, 'cellpath', '').fillna('').astype(str)
                    chunk['hour'] = to_num(chunk[offset_col]) / 60.0
                    chunk['value_raw'] = to_num(chunk[value_col])
                    rows = []
                    uo = chunk.loc[str_contains_any(text, URINE_OUTPUT_PATTERNS)].copy()
                    if not uo.empty:
                        uo['variable'] = 'urine_output'
                        uo['value'] = clip_values('urine_output', uo['value_raw'])
                        rows.append(uo[['stay_id', 'hour', 'variable', 'value']])
                    intake = chunk.loc[str_contains_any(text, [r'intake', r'infusion', r'fluid', r'iv']) & ~str_contains_any(text, [r'output', r'urine'])].copy()
                    if not intake.empty:
                        intake['variable'] = 'fluid_input'
                        intake['value'] = clip_values('fluid_input', intake['value_raw'])
                        rows.append(intake[['stay_id', 'hour', 'variable', 'value']])
                    if rows:
                        ev = pd.concat(rows, ignore_index=True)
                        ev['los_hours'] = ev['stay_id'].map(los_map)
                        ev = ev.loc[(ev['hour'] >= 0) & (ev['hour'] <= ev['los_hours'].clip(upper=cfg.max_icu_hours))]
                        sink.write(make_binned_events(ev[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk
                    gc.collect()

            # Medication and infusion drugs.
            for table, offset_candidates, name_candidates in [
                ('medication', ['drugstartoffset', 'drugorderoffset'], ['drugname']),
                ('infusionDrug', ['infusionoffset'], ['drugname']),
            ]:
                if store.find(table, required=False) is None:
                    continue
                reader = store.read_csv(table, chunksize=cfg.chunksize, required=False)
                for chunk in tqdm(reader, desc=f'eICU {table}'):
                    chunk = normalize_cols_lower(chunk)
                    if 'patientunitstayid' not in chunk.columns:
                        continue
                    chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
                    chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
                    if chunk.empty:
                        continue
                    offset_col = get_col(chunk, offset_candidates)
                    name_col = get_col(chunk, name_candidates)
                    if offset_col is None or name_col is None:
                        continue
                    chunk['exposures'] = chunk[name_col].map(map_drug_to_exposures)
                    chunk = chunk.loc[chunk['exposures'].map(len) > 0].copy()
                    if chunk.empty:
                        continue
                    chunk['hour'] = to_num(chunk[offset_col]) / 60.0
                    rows = []
                    for sid, hour, exps in chunk[['stay_id', 'hour', 'exposures']].itertuples(index=False):
                        if pd.isna(hour):
                            continue
                        for var in exps:
                            rows.append((sid, hour, var, 1.0))
                    if rows:
                        ev = pd.DataFrame(rows, columns=['stay_id', 'hour', 'variable', 'value'])
                        ev['los_hours'] = ev['stay_id'].map(los_map)
                        ev = ev.loc[(ev['hour'] >= 0) & (ev['hour'] <= ev['los_hours'].clip(upper=cfg.max_icu_hours))]
                        sink.write(make_binned_events(ev[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk
                    gc.collect()

            # Treatments for RRT and ventilation.
            if store.find('treatment', required=False) is not None:
                reader = store.read_csv('treatment', chunksize=cfg.chunksize, required=False)
                for chunk in tqdm(reader, desc='eICU treatment'):
                    chunk = normalize_cols_lower(chunk)
                    if 'patientunitstayid' not in chunk.columns:
                        continue
                    chunk['stay_id'] = chunk['patientunitstayid'].astype(str)
                    chunk = chunk.loc[chunk['stay_id'].isin(ids)].copy()
                    if chunk.empty:
                        continue
                    offset_col = get_col(chunk, ['treatmentoffset'])
                    txt_col = get_col(chunk, ['treatmentstring'])
                    if offset_col is None or txt_col is None:
                        continue
                    chunk['exposures'] = chunk[txt_col].map(map_treatment_to_exposures)
                    chunk = chunk.loc[chunk['exposures'].map(len) > 0].copy()
                    chunk['hour'] = to_num(chunk[offset_col]) / 60.0
                    rows = []
                    for sid, hour, exps in chunk[['stay_id', 'hour', 'exposures']].itertuples(index=False):
                        if pd.isna(hour):
                            continue
                        for var in exps:
                            rows.append((sid, hour, var, 1.0))
                    if rows:
                        ev = pd.DataFrame(rows, columns=['stay_id', 'hour', 'variable', 'value'])
                        ev['los_hours'] = ev['stay_id'].map(los_map)
                        ev = ev.loc[(ev['hour'] >= 0) & (ev['hour'] <= ev['los_hours'].clip(upper=cfg.max_icu_hours))]
                        sink.write(make_binned_events(ev[['stay_id', 'hour', 'variable', 'value']], cfg))
                    del chunk
                    gc.collect()
        finally:
            store.close()
    finalize_binned_events(raw_path, final_path)
    return final_path


## 12. Final cohort, KDIGO phenotype, severe AKI/RRT event times, and landmark labels

This section finalizes cohort-level baseline information, builds sparse stay-level matrices, computes severe AKI event times from serum creatinine and urine-output tracks, identifies new RRT, and generates dynamic landmark labels. Events during the actionability gap are excluded from that landmark to reduce leakage.


In [ ]:
# -----------------------------------------------------------------------------
# Cohort finalization and gridding/phenotype labeling
# -----------------------------------------------------------------------------
def finalize_cohort(dataset: str, cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame, baseline: pd.DataFrame) -> pd.DataFrame:
    out_path = dirs['cohort'] / f'{dataset}_cohort_final.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)
    c = cohort.copy()
    c['stay_id'] = c['stay_id'].astype(str)
    baseline['stay_id'] = baseline['stay_id'].astype(str)
    c = c.merge(baseline, on='stay_id', how='left', suffixes=('', '_base'))
    if 'weight_kg_base' in c.columns:
        c['weight_kg'] = c['weight_kg'].combine_first(c['weight_kg_base'])
    c['baseline_scr'] = clip_values('creatinine', c['baseline_scr'])
    baseline_source = series_or_default(c, 'baseline_source', 'missing').fillna('missing').astype(str)
    c['baseline_source_prior'] = baseline_source.eq('prior').astype(float)
    admission_fallback_source = baseline_source.str.startswith('early_min_0_') | baseline_source.str.startswith('first_observed_0_')
    c['baseline_source_admission_fallback'] = admission_fallback_source.astype(float)
    c['weight_missing'] = c['weight_kg'].isna().astype(float)
    # Conservative imputation for urine-output KDIGO denominator; indicator is retained as static input.
    sex_median = c.groupby('sex_female')['weight_kg'].transform('median')
    c['weight_kg'] = c['weight_kg'].fillna(sex_median).fillna(c['weight_kg'].median()).fillna(75.0)
    for col in STATIC_COLS:
        if col not in c.columns:
            c[col] = 0.0
        c[col] = to_num(c[col]).fillna(0.0)
    c = c.loc[c['baseline_scr'].notna() & (c['baseline_scr'] > 0)].copy()
    save_parquet(c, out_path)
    logging.info("%s final cohort after baseline availability: %s", dataset, len(c))
    return c



def _validate_event_critical_coverage(dataset: str, events: pd.DataFrame, cohort: pd.DataFrame, cfg: Config, require_overlap: bool = True) -> Dict[str, Any]:
    """Log and optionally validate critical AKI variables in a binned-event table."""
    info: Dict[str, Any] = {'dataset': dataset, 'n_rows': int(len(events)) if events is not None else 0}
    if events is None or events.empty:
        logging.warning("%s event table is empty", dataset)
        info.update({'creatinine_stays': 0, 'urine_output_stays': 0, 'overlap_stays': 0})
        return info
    ev = events[['stay_id', 'variable']].copy()
    ev['stay_id'] = normalize_id_series(ev['stay_id'])
    scr_stays = set(ev.loc[ev['variable'].eq('creatinine'), 'stay_id'].dropna().astype(str).unique())
    uo_stays = set(ev.loc[ev['variable'].eq('urine_output'), 'stay_id'].dropna().astype(str).unique())
    cohort_stays = set(normalize_id_series(cohort['stay_id']).astype(str).tolist()) if cohort is not None and 'stay_id' in cohort.columns else set()
    overlap = scr_stays & uo_stays & cohort_stays
    info.update({
        'creatinine_stays': len(scr_stays & cohort_stays),
        'urine_output_stays': len(uo_stays & cohort_stays),
        'overlap_stays': len(overlap),
        'variables': sorted(ev['variable'].dropna().astype(str).unique().tolist()),
    })
    logging.info("%s critical event coverage: creatinine stays=%s; urine-output stays=%s; overlap=%s",
                 dataset, info['creatinine_stays'], info['urine_output_stays'], info['overlap_stays'])
    if require_overlap and info['overlap_stays'] == 0:
        logging.warning(
            "%s has zero stays with both creatinine and urine output in the binned event table. "
            "Baseline creatinine will be seeded as an observed t=0 value, but urine output must still be present.",
            dataset,
        )
    return info

def build_matrix_for_stay(events: pd.DataFrame, stay: pd.Series, cfg: Config) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_steps = int(min(cfg.max_grid_steps, math.ceil(float(stay['los_hours']) / cfg.grid_hours)))
    n_steps = max(n_steps, int(math.ceil(cfg.min_los_hours / cfg.grid_hours)))
    x = np.full((n_steps, N_VARS), np.nan, dtype=np.float32)
    mask = np.zeros((n_steps, N_VARS), dtype=np.float32)
    count = np.zeros((n_steps, N_VARS), dtype=np.float32)
    # Binary exposures default to known zero; fluid_input default to known zero.
    for var in BINARY_VARS + ['fluid_input']:
        j = VAR_INDEX[var]
        x[:, j] = 0.0
        mask[:, j] = 1.0
    if events is not None and not events.empty:
        for r in events.itertuples(index=False):
            b = int(r.bin)
            var = str(r.variable)
            if var not in VAR_INDEX or b < 0 or b >= n_steps:
                continue
            j = VAR_INDEX[var]
            x[b, j] = float(r.value)
            mask[b, j] = 1.0
            count[b, j] = float(getattr(r, 'n_obs', 1))
    # If the within-ICU creatinine stream is missing because of a source-table gap,
    # seed the known baseline sCr at t=0. This keeps the cohort runnable and gives
    # the model the registered baseline kidney-function context; future AKI from sCr
    # still requires later observed creatinine values.
    j_scr = VAR_INDEX['creatinine']
    baseline_scr = pd.to_numeric(pd.Series([stay.get('baseline_scr', np.nan)]), errors='coerce').iloc[0]
    if np.isfinite(baseline_scr) and mask[:, j_scr].sum() == 0:
        x[0, j_scr] = float(baseline_scr)
        mask[0, j_scr] = 1.0
        count[0, j_scr] = 1.0
    # Static weight as repeated measured value.
    j_w = VAR_INDEX['weight_kg']
    x[:, j_w] = float(stay.get('weight_kg', 75.0))
    mask[:, j_w] = 1.0
    count[:, j_w] = 1.0
    # Net fluid only where urine output is observed; fluid input is already zero if absent.
    j_net = VAR_INDEX['net_fluid']; j_in = VAR_INDEX['fluid_input']; j_uo = VAR_INDEX['urine_output']
    uo_obs = mask[:, j_uo] > 0
    x[uo_obs, j_net] = x[uo_obs, j_in] - x[uo_obs, j_uo]
    mask[uo_obs, j_net] = 1.0
    count[uo_obs, j_net] = 1.0
    # Clip all numeric variables.
    for var, j in VAR_INDEX.items():
        if var in VALUE_RANGES:
            lo, hi = VALUE_RANGES[var]
            vals = x[:, j]
            bad = np.isfinite(vals) & ((vals < lo) | (vals > hi))
            vals[bad] = np.nan
            x[:, j] = vals
            mask[np.isnan(x[:, j]), j] = 0.0
    # Re-assert binary defaults.
    for var in BINARY_VARS + ['fluid_input']:
        j = VAR_INDEX[var]
        x[:, j] = np.nan_to_num(x[:, j], nan=0.0)
        mask[:, j] = 1.0
    return x, mask, count


def first_true_time_hours(flag: np.ndarray, cfg: Config) -> float:
    idx = np.where(flag)[0]
    if len(idx) == 0:
        return np.inf
    return float(idx[0] * cfg.grid_hours)


def compute_aki_event_times(x: np.ndarray, mask: np.ndarray, stay: pd.Series, cfg: Config) -> Dict[str, float]:
    baseline_scr = float(stay['baseline_scr'])
    weight = float(stay.get('weight_kg', 75.0))
    j_scr = VAR_INDEX['creatinine']; j_uo = VAR_INDEX['urine_output']; j_rrt = VAR_INDEX['rrt']
    scr_obs = (mask[:, j_scr] > 0) & np.isfinite(x[:, j_scr])
    twofold_rise = x[:, j_scr] >= 2.0 * baseline_scr
    absolute_stage3 = x[:, j_scr] >= 4.0
    if getattr(cfg, 'require_acute_rise_for_scr_4', True):
        absolute_stage3 = absolute_stage3 & (x[:, j_scr] >= baseline_scr + 0.3)
    severe_scr = scr_obs & (twofold_rise | absolute_stage3)
    scr_time = first_true_time_hours(severe_scr, cfg)

    uo = np.where((mask[:, j_uo] > 0) & np.isfinite(x[:, j_uo]), x[:, j_uo], 0.0)
    uo_obs = (mask[:, j_uo] > 0) & np.isfinite(x[:, j_uo])
    stage2_flag = np.zeros(len(uo), dtype=bool)
    stage3_flag = np.zeros(len(uo), dtype=bool)
    for window_h, threshold, out_flag in [(12, 0.5, stage2_flag), (24, 0.3, stage3_flag)]:
        w = max(1, int(math.ceil(window_h / cfg.grid_hours)))
        for i in range(w - 1, len(uo)):
            obs_n = int(uo_obs[i - w + 1:i + 1].sum())
            if obs_n < math.ceil(w * cfg.urine_min_observed_fraction):
                continue
            total = float(uo[i - w + 1:i + 1].sum())
            rate = total / max(weight * window_h, 1e-6)
            if rate < threshold:
                out_flag[i] = True
    uo_time = first_true_time_hours(stage2_flag | stage3_flag, cfg)
    rrt = (x[:, j_rrt] > 0.5) & (mask[:, j_rrt] > 0)
    rrt_time = first_true_time_hours(rrt, cfg)
    event_interest_time = min(scr_time, uo_time, rrt_time)
    return {
        'scr_stage2plus_time': scr_time,
        'uo_stage2plus_time': uo_time,
        'rrt_time': rrt_time,
        'event_interest_time': event_interest_time,
    }


def determine_landmark_label(lm: float, times: Dict[str, float], stay: pd.Series, cfg: Config) -> Optional[Tuple[int, int, float, float]]:
    origin = lm + cfg.action_gap_hours
    horizon_end = origin + cfg.horizon_hours
    event_time = times.get('event_interest_time', np.inf)
    death_time = float(stay['death_hours']) if pd.notna(stay.get('death_hours', np.nan)) else np.inf
    discharge_time = float(stay['discharge_hours']) if pd.notna(stay.get('discharge_hours', np.nan)) else float(stay['los_hours'])
    # Landmark is not valid if outcome has already occurred at/before landmark.
    if min(event_time, death_time, discharge_time) <= lm:
        return None
    # Skip non-actionable landmarks with any terminal event during the action gap.
    if min(event_time, death_time, discharge_time) <= origin:
        return None
    candidates = []
    if event_time <= horizon_end:
        candidates.append((event_time, 1))
    if death_time <= horizon_end:
        candidates.append((death_time, 2))
    if discharge_time <= horizon_end:
        candidates.append((discharge_time, 3))
    if candidates:
        t_abs, etype = min(candidates, key=lambda x: (x[0], 0 if x[1] == 1 else x[1]))
        t_rel = max(0.0, t_abs - origin)
        bin_idx = int(min(cfg.n_surv_bins - 1, max(0, math.ceil(t_rel / cfg.landmark_step_hours) - 1)))
        return etype, bin_idx, t_rel, min(t_rel, cfg.horizon_hours)
    # Censored after the complete horizon.
    return 0, cfg.n_surv_bins - 1, cfg.horizon_hours, cfg.horizon_hours



def write_clinical_phenotype_audit(
    dataset: str,
    cohort: pd.DataFrame,
    landmarks: pd.DataFrame,
    summaries: pd.DataFrame,
    excluded: Dict[str, int],
    cfg: Config,
    dirs: Dict[str, Path],
) -> None:
    """Persist compact phenotype/cohort audit files for clinical review."""
    audit_dir = dirs['metrics'] / 'clinical_audit'
    audit_dir.mkdir(parents=True, exist_ok=True)
    lm = landmarks if landmarks is not None else pd.DataFrame()
    sm = summaries if summaries is not None else pd.DataFrame()
    baseline_counts = {}
    if cohort is not None and 'baseline_source' in cohort.columns:
        baseline_counts = {str(k): int(v) for k, v in cohort['baseline_source'].fillna('missing').astype(str).value_counts().items()}
    race_ethnicity_counts = {}
    if cohort is not None and 'race_ethnicity' in cohort.columns:
        race_ethnicity_counts = {str(k): int(v) for k, v in cohort['race_ethnicity'].fillna('unknown').astype(str).value_counts().items()}
    event_type_counts = {}
    if not lm.empty and 'event_type' in lm.columns:
        event_type_counts = {str(int(k)): int(v) for k, v in lm['event_type'].value_counts().sort_index().items()}
    stay_event_counts = {}
    if not sm.empty and 'event_interest_time' in sm.columns:
        finite_event = np.isfinite(pd.to_numeric(sm['event_interest_time'], errors='coerce'))
        stay_event_counts['severe_aki_or_rrt_stays'] = int(finite_event.sum())
        for col in ['scr_stage2plus_time', 'uo_stage2plus_time', 'rrt_time']:
            if col in sm.columns:
                stay_event_counts[col.replace('_time', '_stays')] = int(np.isfinite(pd.to_numeric(sm[col], errors='coerce')).sum())
    rows_per_stay = lm.groupby('stay_id').size() if not lm.empty and 'stay_id' in lm.columns else pd.Series(dtype=float)
    payload = {
        'dataset': dataset,
        'cohort_stays_after_baseline_filter': int(len(cohort)) if cohort is not None else 0,
        'landmark_rows': int(len(lm)),
        'landmark_stays': int(lm['stay_id'].nunique()) if not lm.empty and 'stay_id' in lm.columns else 0,
        'median_landmarks_per_stay': float(rows_per_stay.median()) if len(rows_per_stay) else 0.0,
        'baseline_source_counts': baseline_counts,
        'race_ethnicity_counts': race_ethnicity_counts,
        'landmark_event_type_counts': event_type_counts,
        'stay_event_counts': stay_event_counts,
        'exclusions': {str(k): int(v) for k, v in dict(excluded).items()},
        'phenotype_settings': {
            'grid_hours': cfg.grid_hours,
            'landmark_start_hours': cfg.landmark_start_hours,
            'action_gap_hours': cfg.action_gap_hours,
            'horizon_hours': cfg.horizon_hours,
            'baseline_lookback_days': cfg.baseline_lookback_days,
            'baseline_admission_fallback_hours': cfg.baseline_admission_fallback_hours,
            'urine_min_observed_fraction': cfg.urine_min_observed_fraction,
            'require_acute_rise_for_scr_4': cfg.require_acute_rise_for_scr_4,
        },
    }
    with open(audit_dir / f'{dataset}_clinical_phenotype_audit.json', 'w') as f:
        json.dump(payload, f, indent=2)
    version_path = dirs['cache'] / f'{dataset}_clinical_pipeline.version'
    version_path.write_text(CLINICAL_PIPELINE_VERSION)
    flat = {
        'dataset': payload['dataset'],
        'cohort_stays_after_baseline_filter': payload['cohort_stays_after_baseline_filter'],
        'landmark_rows': payload['landmark_rows'],
        'landmark_stays': payload['landmark_stays'],
        'median_landmarks_per_stay': payload['median_landmarks_per_stay'],
    }
    pd.DataFrame([flat]).to_csv(audit_dir / f'{dataset}_clinical_phenotype_audit_summary.csv', index=False)


def build_grids_and_landmarks(dataset: str, cfg: Config, dirs: Dict[str, Path], cohort: pd.DataFrame, events_path: Path) -> pd.DataFrame:
    out_path = dirs['landmarks'] / f'{dataset}_landmarks.parquet'
    summary_path = dirs['landmarks'] / f'{dataset}_stay_summary.parquet'
    grid_dir = dirs['grids'] / dataset
    grid_dir.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and summary_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)

    logging.info("Building %s grids and dynamic AKI landmarks", dataset)
    events = read_parquet(events_path)
    events['stay_id'] = normalize_id_series(events['stay_id'])
    cohort = cohort.copy()
    cohort['stay_id'] = normalize_id_series(cohort['stay_id'])
    _validate_event_critical_coverage(dataset, events, cohort, cfg, require_overlap=True)
    cohort = cohort.loc[cohort['stay_id'].isin(set(cohort['stay_id']))].reset_index(drop=True)
    events_by_stay = {sid: g for sid, g in events.groupby('stay_id', sort=False)}
    landmarks = []
    summaries = []
    excluded = defaultdict(int)
    for stay in tqdm(cohort.itertuples(index=False), total=len(cohort), desc=f'{dataset} grids'):
        stay_s = pd.Series(stay._asdict())
        sid = str(stay_s['stay_id'])
        ev = events_by_stay.get(sid, pd.DataFrame(columns=events.columns))
        x, mask, count = build_matrix_for_stay(ev, stay_s, cfg)
        # Inclusion: at least one creatinine and calculable urine output.
        has_scr = bool(mask[:, VAR_INDEX['creatinine']].sum() > 0)
        has_uo = bool(mask[:, VAR_INDEX['urine_output']].sum() > 0)
        if not has_scr and not has_uo:
            excluded['missing_scr_and_uo'] += 1
            continue
        if not has_scr:
            excluded['missing_scr'] += 1
            continue
        if not has_uo:
            excluded['missing_uo'] += 1
            continue
        times = compute_aki_event_times(x, mask, stay_s, cfg)
        if times['event_interest_time'] <= cfg.landmark_start_hours:
            excluded['already_severe_aki_or_rrt_by_6h'] += 1
            continue
        # New RRT by 6h exclusion explicitly retained.
        if times['rrt_time'] <= cfg.landmark_start_hours:
            excluded['rrt_by_6h'] += 1
            continue
        grid_path = grid_dir / f'{sid}.npz'
        np.savez_compressed(grid_path, x=x, mask=mask, count=count)
        base = stay_s.to_dict()
        base.update(times)
        summaries.append({k: base.get(k) for k in ['dataset', 'stay_id', 'hospital_id', 'subject_id', 'hadm_id', 'race_ethnicity', 'los_hours', 'death_hours', 'discharge_hours', 'baseline_scr', 'baseline_source', 'weight_kg', 'weight_missing', 'scr_stage2plus_time', 'uo_stage2plus_time', 'rrt_time', 'event_interest_time'] if k in base})
        last_lm = min(float(stay_s['los_hours']), float(cfg.max_icu_hours))
        lm_values = np.arange(cfg.landmark_start_hours, last_lm, cfg.landmark_step_hours, dtype=float)
        for lm in lm_values:
            label = determine_landmark_label(float(lm), times, stay_s, cfg)
            if label is None:
                continue
            etype, tbin, t_rel, obs_h = label
            row = {
                'dataset': dataset,
                'stay_id': sid,
                'subject_id': str(stay_s.get('subject_id', sid)),
                'hadm_id': str(stay_s.get('hadm_id', sid)),
                'hospital_id': str(stay_s.get('hospital_id', 'unknown')),
                'race_ethnicity': str(stay_s.get('race_ethnicity', 'unknown')),
                'anchor_year_group': str(stay_s.get('anchor_year_group', 'unknown')),
                'landmark_hour': float(lm),
                'origin_hour': float(lm + cfg.action_gap_hours),
                'event_type': int(etype),
                'time_bin': int(tbin),
                'time_to_event_hours': float(t_rel),
                'observed_horizon_hours': float(obs_h),
                'grid_path': str(grid_path),
                'los_hours': float(stay_s.get('los_hours', np.nan)),
                'death_hours': float(stay_s.get('death_hours', np.nan)) if pd.notna(stay_s.get('death_hours', np.nan)) else np.nan,
                'discharge_hours': float(stay_s.get('discharge_hours', np.nan)),
                'event_interest_time': float(times['event_interest_time']) if np.isfinite(times['event_interest_time']) else np.inf,
                'baseline_scr': float(stay_s.get('baseline_scr', np.nan)),
                'baseline_source': str(stay_s.get('baseline_source', 'missing')),
                'weight_kg': float(stay_s.get('weight_kg', np.nan)),
                'weight_missing': float(stay_s.get('weight_missing', 0.0)),
            }
            for col in STATIC_COLS:
                row[col] = float(stay_s.get(col, 0.0)) if pd.notna(stay_s.get(col, 0.0)) else 0.0
            landmarks.append(row)
    lm_df = pd.DataFrame(landmarks)
    sum_df = pd.DataFrame(summaries)
    save_parquet(lm_df, out_path)
    save_parquet(sum_df, summary_path)
    with open(dirs['landmarks'] / f'{dataset}_exclusions.json', 'w') as f:
        json.dump(dict(excluded), f, indent=2)
    write_clinical_phenotype_audit(dataset, cohort, lm_df, sum_df, dict(excluded), cfg, dirs)
    logging.info("%s landmarks: %s rows from %s stays. Exclusions: %s", dataset, len(lm_df), lm_df['stay_id'].nunique() if not lm_df.empty else 0, dict(excluded))
    return lm_df


## 13. Sequence dataset, masks, time gaps, and standardization

The model input is `value + mask + time-gap + observation-count`. This section builds history slices ending at each landmark, computes time since last measurement, standardizes values using source-training landmarks only, and exposes a PyTorch `Dataset` for the SSL and survival models.


In [ ]:
# -----------------------------------------------------------------------------
# Sequence dataset and standardization
# -----------------------------------------------------------------------------
def load_npz_cached(path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    data = np.load(path)
    return data['x'].astype(np.float32), data['mask'].astype(np.float32), data['count'].astype(np.float32)


@lru_cache(maxsize=512)
def load_npz_lru(path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    return load_npz_cached(path)


def get_history_slice(row: pd.Series, cfg: Config) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x, mask, count = load_npz_lru(str(row['grid_path']))
    end_bin = int(math.ceil(float(row['landmark_hour']) / cfg.grid_hours))
    start_bin = end_bin - cfg.n_history_steps
    pad_left = max(0, -start_bin)
    start = max(0, start_bin)
    seq_x = x[start:end_bin]
    seq_m = mask[start:end_bin]
    seq_c = count[start:end_bin]
    if pad_left > 0:
        seq_x = np.vstack([np.full((pad_left, N_VARS), np.nan, dtype=np.float32), seq_x])
        seq_m = np.vstack([np.zeros((pad_left, N_VARS), dtype=np.float32), seq_m])
        seq_c = np.vstack([np.zeros((pad_left, N_VARS), dtype=np.float32), seq_c])
    if seq_x.shape[0] < cfg.n_history_steps:
        pad = cfg.n_history_steps - seq_x.shape[0]
        seq_x = np.vstack([np.full((pad, N_VARS), np.nan, dtype=np.float32), seq_x])
        seq_m = np.vstack([np.zeros((pad, N_VARS), dtype=np.float32), seq_m])
        seq_c = np.vstack([np.zeros((pad, N_VARS), dtype=np.float32), seq_c])
    elif seq_x.shape[0] > cfg.n_history_steps:
        seq_x = seq_x[-cfg.n_history_steps:]
        seq_m = seq_m[-cfg.n_history_steps:]
        seq_c = seq_c[-cfg.n_history_steps:]
    return seq_x.astype(np.float32), seq_m.astype(np.float32), seq_c.astype(np.float32)


def compute_scaler(landmarks: pd.DataFrame, cfg: Config, out_path: Path) -> Dict[str, Any]:
    if out_path.exists() and not cfg.overwrite:
        with open(out_path, 'r') as f:
            return json.load(f)
    sums = np.zeros(N_VARS, dtype=np.float64)
    sums2 = np.zeros(N_VARS, dtype=np.float64)
    counts = np.zeros(N_VARS, dtype=np.float64)
    static_sums = np.zeros(len(STATIC_COLS), dtype=np.float64)
    static_sums2 = np.zeros(len(STATIC_COLS), dtype=np.float64)
    static_counts = np.zeros(len(STATIC_COLS), dtype=np.float64)
    for _, row in tqdm(landmarks.iterrows(), total=len(landmarks), desc='compute scaler'):
        x, m, _ = get_history_slice(row, cfg)
        obs = (m > 0) & np.isfinite(x)
        vals = np.where(obs, x, 0.0)
        sums += vals.sum(axis=0)
        sums2 += (vals ** 2).sum(axis=0)
        counts += obs.sum(axis=0)
        sv = np.array([row.get(c, 0.0) for c in STATIC_COLS], dtype=np.float64)
        ok = np.isfinite(sv)
        static_sums += np.where(ok, sv, 0.0)
        static_sums2 += np.where(ok, sv ** 2, 0.0)
        static_counts += ok.astype(float)
    means = sums / np.maximum(counts, 1.0)
    vars_ = sums2 / np.maximum(counts, 1.0) - means ** 2
    stds = np.sqrt(np.maximum(vars_, 1e-6))
    for var in BINARY_VARS:
        j = VAR_INDEX[var]
        means[j] = 0.0; stds[j] = 1.0
    for var in ['fluid_input']:
        # Keep fluid in natural scaled units but standardized with observed distribution.
        pass
    static_means = static_sums / np.maximum(static_counts, 1.0)
    static_vars = static_sums2 / np.maximum(static_counts, 1.0) - static_means ** 2
    static_stds = np.sqrt(np.maximum(static_vars, 1e-6))
    payload = {
        'variables': MATRIX_VARS,
        'means': means.tolist(),
        'stds': stds.tolist(),
        'static_cols': STATIC_COLS,
        'static_means': static_means.tolist(),
        'static_stds': static_stds.tolist(),
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(payload, f, indent=2)
    return payload


def compute_delta(mask: np.ndarray, cfg: Config) -> np.ndarray:
    L, V = mask.shape
    delta = np.zeros((L, V), dtype=np.float32)
    last = np.full(V, -np.inf, dtype=np.float32)
    for t in range(L):
        obs = mask[t] > 0
        last[obs] = t
        gap = (t - last) * cfg.grid_hours
        gap[~np.isfinite(gap)] = cfg.history_hours
        delta[t] = np.minimum(gap, cfg.history_hours)
    return delta / max(cfg.history_hours, 1)


class AKILandmarkDataset(Dataset):
    def __init__(self, landmarks: pd.DataFrame, cfg: Config, scaler: Dict[str, Any], augment: bool = False):
        self.landmarks = landmarks.reset_index(drop=True).copy()
        self.cfg = cfg
        self.scaler = scaler
        self.augment = augment
        self.means = np.array(scaler['means'], dtype=np.float32)
        self.stds = np.array(scaler['stds'], dtype=np.float32)
        self.static_means = np.array(scaler['static_means'], dtype=np.float32)
        self.static_stds = np.array(scaler['static_stds'], dtype=np.float32)

    def __len__(self) -> int:
        return len(self.landmarks)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.landmarks.iloc[idx]
        x, mask, count = get_history_slice(row, self.cfg)
        z = (x - self.means) / self.stds
        z = np.where((mask > 0) & np.isfinite(z), z, 0.0).astype(np.float32)
        delta = compute_delta(mask, self.cfg)
        count_scaled = np.log1p(count).astype(np.float32) / 5.0
        inp = np.concatenate([z, mask.astype(np.float32), delta, count_scaled], axis=1).astype(np.float32)
        static = np.array([row.get(c, 0.0) for c in STATIC_COLS], dtype=np.float32)
        static = np.where(np.isfinite(static), static, 0.0)
        static = (static - self.static_means) / self.static_stds
        return {
            'x_in': torch.from_numpy(inp),
            'values': torch.from_numpy(z),
            'obs_mask': torch.from_numpy(mask.astype(np.float32)),
            'static': torch.from_numpy(static.astype(np.float32)),
            'event_type': torch.tensor(int(row['event_type']), dtype=torch.long),
            'time_bin': torch.tensor(int(row['time_bin']), dtype=torch.long),
            'index': torch.tensor(idx, dtype=torch.long),
        }


## 14. Model architecture: ICU encoder, SSL heads, and competing-risk survival head

The encoder is a transformer over ICU history windows plus static context. The SSL model reconstructs masked values, predicts observation masks, and performs time-order discrimination. The survival model outputs discrete hazards/CIF components for AKI/RRT, death, and ICU discharge.


In [ ]:
# -----------------------------------------------------------------------------
# Self-supervised encoder and survival transformer
# -----------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]


class ICUEncoder(nn.Module):
    def __init__(self, input_dim: int, static_dim: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.input_proj = nn.Linear(input_dim, cfg.d_model)
        self.static_proj = nn.Sequential(nn.Linear(static_dim, cfg.d_model), nn.GELU(), nn.Dropout(cfg.dropout))
        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
        self.pos = PositionalEncoding(cfg.d_model, max_len=cfg.n_history_steps + 1)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_model * 4,
            dropout=cfg.dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.norm = nn.LayerNorm(cfg.d_model)
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_in: torch.Tensor, static: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.input_proj(x_in)
        cls = self.cls.expand(h.size(0), -1, -1) + self.static_proj(static).unsqueeze(1)
        h = torch.cat([cls, h], dim=1)
        h = self.pos(h)
        h = self.encoder(h)
        h = self.norm(h)
        return h[:, 0], h[:, 1:]


class SSLPretrainModel(nn.Module):
    def __init__(self, input_dim: int, static_dim: int, cfg: Config):
        super().__init__()
        self.encoder = ICUEncoder(input_dim, static_dim, cfg)
        self.recon_head = nn.Linear(cfg.d_model, N_VARS)
        self.obs_head = nn.Linear(cfg.d_model, N_VARS)
        self.order_head = nn.Linear(cfg.d_model, 1)

    def forward(self, x_in: torch.Tensor, static: torch.Tensor) -> Dict[str, torch.Tensor]:
        pooled, seq = self.encoder(x_in, static)
        return {
            'pooled': pooled,
            'recon': self.recon_head(seq),
            'obs_logits': self.obs_head(seq),
            'order_logit': self.order_head(pooled).squeeze(-1),
        }


class SurvivalTransformer(nn.Module):
    def __init__(self, encoder: ICUEncoder, cfg: Config, n_causes: int = 3):
        super().__init__()
        self.encoder = encoder
        self.cfg = cfg
        self.n_causes = n_causes
        self.head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, cfg.n_surv_bins * (n_causes + 1)),
        )

    def forward(self, x_in: torch.Tensor, static: torch.Tensor) -> torch.Tensor:
        pooled, _ = self.encoder(x_in, static)
        logits = self.head(pooled).view(-1, self.cfg.n_surv_bins, self.n_causes + 1)
        return logits


## 15. Self-supervised pretraining, survival fine-tuning, and CIF prediction

Training uses masked trajectory reconstruction plus observation-process learning during pretraining, then fine-tunes a discrete-time competing-risk likelihood. Prediction converts per-bin hazards into cumulative incidence for the AKI/RRT cause of interest.


In [ ]:
def ssl_batch_loss(model: SSLPretrainModel, batch: Dict[str, torch.Tensor], cfg: Config, device: str) -> torch.Tensor:
    x_in = batch['x_in'].to(device)
    static = batch['static'].to(device)
    values = batch['values'].to(device)
    obs = batch['obs_mask'].to(device)
    V = N_VARS
    # Extra mask on observed values only; mask and delta channels remain visible so the model learns sparse sampling.
    rand = torch.rand_like(obs)
    extra_mask = (obs > 0.5) & (rand < cfg.mask_ratio)
    x_aug = x_in.clone()
    x_aug[..., :V] = torch.where(extra_mask, torch.zeros_like(x_aug[..., :V]), x_aug[..., :V])
    out = model(x_aug, static)
    if extra_mask.sum() > 0:
        mse = ((out['recon'] - values) ** 2)[extra_mask].mean()
    else:
        mse = ((out['recon'] - values) ** 2)[obs > 0.5].mean()
    bce = F.binary_cross_entropy_with_logits(out['obs_logits'], obs, reduction='mean')
    # Time-order discrimination: half batches are reversed in time and classified.
    flip = torch.rand(x_in.size(0), device=device) < 0.5
    x_order = x_in.clone()
    x_order[flip] = torch.flip(x_order[flip], dims=[1])
    out_order = model(x_order, static)
    order_target = (~flip).float()
    order_loss = F.binary_cross_entropy_with_logits(out_order['order_logit'], order_target)
    return mse + 0.2 * bce + 0.1 * order_loss


def discrete_survival_nll(logits: torch.Tensor, event_type: torch.Tensor, time_bin: torch.Tensor) -> torch.Tensor:
    eps = 1e-8
    probs = torch.softmax(logits, dim=-1)
    p0 = probs[:, :, 0].clamp_min(eps)
    logp0 = torch.log(p0)
    cum_log_surv_incl = torch.cumsum(logp0, dim=1)
    cum_log_surv_excl = cum_log_surv_incl - logp0
    B = logits.size(0)
    idx = torch.arange(B, device=logits.device)
    tb = time_bin.clamp(0, logits.size(1) - 1)
    et = event_type
    event_mask = et > 0
    losses = []
    if event_mask.any():
        i = idx[event_mask]
        b = tb[event_mask]
        e = et[event_mask]
        log_event = cum_log_surv_excl[i, b] + torch.log(probs[i, b, e].clamp_min(eps))
        losses.append(-log_event)
    if (~event_mask).any():
        i = idx[~event_mask]
        b = tb[~event_mask]
        log_cens = cum_log_surv_incl[i, b]
        losses.append(-log_cens)
    return torch.cat(losses).mean()


def train_ssl(model: SSLPretrainModel, train_loader: DataLoader, val_loader: Optional[DataLoader], cfg: Config, out_path: Path) -> SSLPretrainModel:
    device = cfg.device
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and device.startswith('cuda')))
    best = float('inf'); bad = 0
    out_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, cfg.ssl_epochs + 1):
        model.train(); losses = []
        for batch in tqdm(train_loader, desc=f'SSL epoch {epoch}'):
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.mixed_precision and device.startswith('cuda'))):
                loss = ssl_batch_loss(model, batch, cfg, device)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            losses.append(float(loss.detach().cpu()))
        val_loss = np.mean(losses)
        if val_loader is not None:
            model.eval(); vl = []
            with torch.no_grad():
                for batch in val_loader:
                    loss = ssl_batch_loss(model, batch, cfg, device)
                    vl.append(float(loss.detach().cpu()))
            if vl:
                val_loss = float(np.mean(vl))
        logging.info("SSL epoch %d train %.4f val %.4f", epoch, float(np.mean(losses)), val_loss)
        if val_loss < best:
            best = val_loss; bad = 0
            torch.save(model.state_dict(), out_path)
        else:
            bad += 1
            if bad >= cfg.early_stop_patience:
                break
    model.load_state_dict(torch.load(out_path, map_location=device))
    return model


def train_survival(model: SurvivalTransformer, train_loader: DataLoader, val_loader: DataLoader, cfg: Config, out_path: Path) -> SurvivalTransformer:
    device = cfg.device
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and device.startswith('cuda')))
    best = float('inf'); bad = 0
    out_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, cfg.finetune_epochs + 1):
        model.train(); losses = []
        for batch in tqdm(train_loader, desc=f'Survival epoch {epoch}'):
            opt.zero_grad(set_to_none=True)
            x_in = batch['x_in'].to(device); static = batch['static'].to(device)
            et = batch['event_type'].to(device); tb = batch['time_bin'].to(device)
            with torch.cuda.amp.autocast(enabled=(cfg.mixed_precision and device.startswith('cuda'))):
                logits = model(x_in, static)
                loss = discrete_survival_nll(logits, et, tb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            losses.append(float(loss.detach().cpu()))
        model.eval(); vl = []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch['x_in'].to(device), batch['static'].to(device))
                loss = discrete_survival_nll(logits, batch['event_type'].to(device), batch['time_bin'].to(device))
                vl.append(float(loss.detach().cpu()))
        val_loss = float(np.mean(vl)) if vl else float(np.mean(losses))
        logging.info("Survival epoch %d train %.4f val %.4f", epoch, float(np.mean(losses)), val_loss)
        if val_loss < best:
            best = val_loss; bad = 0
            torch.save(model.state_dict(), out_path)
        else:
            bad += 1
            if bad >= cfg.early_stop_patience:
                break
    model.load_state_dict(torch.load(out_path, map_location=device))
    return model


def predict_survival(model: SurvivalTransformer, ds: AKILandmarkDataset, cfg: Config, batch_size: Optional[int] = None) -> np.ndarray:
    workers = int(getattr(cfg, 'dataloader_workers', 0))
    loader = DataLoader(ds, batch_size=batch_size or cfg.batch_size, shuffle=False, num_workers=workers, pin_memory=cfg.device.startswith('cuda'))
    device = cfg.device
    model.to(device); model.eval()
    all_cif = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='predict CIF'):
            logits = model(batch['x_in'].to(device), batch['static'].to(device))
            probs = torch.softmax(logits, dim=-1)
            p0 = probs[:, :, 0]
            surv_before = torch.cumprod(torch.cat([torch.ones((p0.size(0), 1), device=device), p0[:, :-1]], dim=1), dim=1)
            event_prob_aki = surv_before * probs[:, :, 1]
            cif = torch.cumsum(event_prob_aki, dim=1)
            all_cif.append(cif.detach().cpu().numpy())
    return np.vstack(all_cif) if all_cif else np.zeros((0, cfg.n_surv_bins), dtype=np.float32)


## 16. Splits, discrimination, calibration, Brier score, and DCA

MIMIC is split by `anchor_year_group` when available; eICU uses hospital-level grouped splitting. Evaluation reports time-point AUC, AUPRC, Brier score, calibration slope, calibration plots, and decision-curve analysis.


In [ ]:
# -----------------------------------------------------------------------------
# Splitting, metrics, calibration, DCA
# -----------------------------------------------------------------------------
def split_landmarks(df: pd.DataFrame, dataset: str, cfg: Config) -> Dict[str, np.ndarray]:
    df = df.reset_index(drop=True)
    if dataset == 'mimic' and 'anchor_year_group' in df.columns:
        groups_sorted = sorted(df['anchor_year_group'].dropna().astype(str).unique().tolist())
        if len(groups_sorted) >= 3:
            test_group = groups_sorted[-1]
            val_group = groups_sorted[-2]
            train_idx = df.index[~df['anchor_year_group'].astype(str).isin([val_group, test_group])].to_numpy()
            val_idx = df.index[df['anchor_year_group'].astype(str).eq(val_group)].to_numpy()
            test_idx = df.index[df['anchor_year_group'].astype(str).eq(test_group)].to_numpy()
            if len(train_idx) > 0 and len(val_idx) > 0 and len(test_idx) > 0:
                return {'train': train_idx, 'val': val_idx, 'test': test_idx}
    group_col = 'hospital_id' if dataset == 'eicu' else 'stay_id'
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=cfg.seed)
    train_idx, tmp_idx = next(splitter.split(df, groups=df[group_col]))
    tmp = df.iloc[tmp_idx].reset_index(drop=True)
    splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=cfg.seed + 1)
    val_rel, test_rel = next(splitter2.split(tmp, groups=tmp[group_col]))
    val_idx = tmp_idx[val_rel]
    test_idx = tmp_idx[test_rel]
    return {'train': train_idx, 'val': val_idx, 'test': test_idx}



def validate_split_integrity(df: pd.DataFrame, splits: Dict[str, np.ndarray], dataset: str, cfg: Config) -> None:
    """Fail on patient/stay overlap and warn on clinically fragile event counts."""
    split_sets = {name: set(map(int, idx)) for name, idx in splits.items()}
    names = list(split_sets)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            overlap = split_sets[a] & split_sets[b]
            if overlap:
                raise RuntimeError(f'{dataset} split index overlap between {a} and {b}: {len(overlap)} rows')
    for group_col in ['subject_id', 'stay_id']:
        if group_col not in df.columns:
            continue
        group_sets = {name: set(df.iloc[list(idx)][group_col].astype(str)) for name, idx in splits.items()}
        for i, a in enumerate(names):
            for b in names[i + 1:]:
                overlap = group_sets[a] & group_sets[b]
                if overlap:
                    raise RuntimeError(f'{dataset} {group_col} leakage between {a} and {b}: {len(overlap)} shared values')
    h = int(getattr(cfg, 'shap_horizon_hours', 72))
    for name, idx in splits.items():
        part = df.iloc[list(idx)]
        events = int(((part['event_type'].values == 1) & (part['time_to_event_hours'].values <= h)).sum()) if len(part) else 0
        if events < int(getattr(cfg, 'min_split_events_warn', 25)):
            logging.warning('%s %s split has only %s AKI events by %sh; estimates may be unstable.', dataset, name, events, h)


def binary_label_at_horizon(meta: pd.DataFrame, horizon: int) -> Tuple[np.ndarray, np.ndarray]:
    y = ((meta['event_type'].values == 1) & (meta['time_to_event_hours'].values <= horizon)).astype(int)
    # Competing events are clinically meaningful non-AKI by that horizon; censoring before horizon is rare here.
    include = np.ones(len(meta), dtype=bool)
    return y, include


def calibration_intercept_slope(y: np.ndarray, p: np.ndarray) -> Tuple[float, float]:
    if len(np.unique(y)) < 2 or len(y) == 0:
        return np.nan, np.nan
    eps = 1e-6
    logit = np.log(np.clip(p, eps, 1 - eps) / np.clip(1 - p, eps, 1 - eps)).reshape(-1, 1)
    for kwargs in (
        {'penalty': None, 'solver': 'lbfgs', 'max_iter': 1000},
        {'penalty': 'none', 'solver': 'lbfgs', 'max_iter': 1000},
        {'penalty': 'l2', 'C': 1e6, 'solver': 'lbfgs', 'max_iter': 1000},
    ):
        try:
            lr = LogisticRegression(**kwargs)
            lr.fit(logit, y)
            return float(lr.intercept_[0]), float(lr.coef_[0, 0])
        except Exception:
            continue
    return np.nan, np.nan


def expected_calibration_error(y: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    if len(y) == 0:
        return np.nan
    df = pd.DataFrame({'y': y, 'p': np.clip(p, 0, 1)})
    try:
        df['bin'] = pd.qcut(df['p'], q=min(n_bins, max(2, df['p'].nunique())), duplicates='drop')
    except Exception:
        return np.nan
    cal = df.groupby('bin', observed=True).agg(pred=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size'))
    return float((cal['n'] / cal['n'].sum() * (cal['obs'] - cal['pred']).abs()).sum())


def _safe_binary_metrics(y: np.ndarray, p: np.ndarray) -> Dict[str, float]:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    out = {
        'observed': float(np.mean(y)) if len(y) else np.nan,
        'mean_pred': float(np.mean(p)) if len(p) else np.nan,
        'brier': float(brier_score_loss(y, np.clip(p, 0, 1))) if len(y) else np.nan,
        'auc': np.nan,
        'auprc': np.nan,
        'ece': expected_calibration_error(y, p),
    }
    if len(np.unique(y)) > 1:
        out['auc'] = float(roc_auc_score(y, p))
        out['auprc'] = float(average_precision_score(y, p))
    return out


def cluster_bootstrap_metric_cis(y: np.ndarray, p: np.ndarray, clusters: Sequence[Any], cfg: Config) -> Dict[str, Any]:
    metrics = ['auc', 'auprc', 'brier', 'observed', 'mean_pred', 'ece']
    out: Dict[str, Any] = {f'{m}_ci_low': np.nan for m in metrics}
    out.update({f'{m}_ci_high': np.nan for m in metrics})
    out.update({'bootstrap_repeats_requested': int(getattr(cfg, 'bootstrap_repeats', 0)), 'bootstrap_repeats_completed': 0, 'bootstrap_sampled_n': int(len(y)), 'bootstrap_cluster_n': 0})
    repeats = int(getattr(cfg, 'bootstrap_repeats', 0))
    if repeats <= 0 or len(y) == 0:
        return out
    rng = np.random.default_rng(int(getattr(cfg, 'seed', 42)) + 991)
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    clusters = pd.Series(clusters).fillna('missing').astype(str).to_numpy()
    max_rows = int(getattr(cfg, 'bootstrap_max_rows', 50_000))
    if len(y) > max_rows:
        idx = rng.choice(np.arange(len(y)), size=max_rows, replace=False)
        y = y[idx]; p = p[idx]; clusters = clusters[idx]
        out['bootstrap_sampled_n'] = int(max_rows)
    keys, inv = np.unique(clusters, return_inverse=True)
    out['bootstrap_cluster_n'] = int(len(keys))
    if len(keys) < 2:
        return out
    cluster_indices = [np.where(inv == i)[0] for i in range(len(keys))]
    values: Dict[str, List[float]] = defaultdict(list)
    for _ in range(repeats):
        sampled = rng.integers(0, len(keys), size=len(keys))
        idx = np.concatenate([cluster_indices[j] for j in sampled])
        if len(idx) == 0 or len(np.unique(y[idx])) < 2:
            continue
        met = _safe_binary_metrics(y[idx], p[idx])
        for m in metrics:
            if np.isfinite(met.get(m, np.nan)):
                values[m].append(float(met[m]))
    completed = max((len(v) for v in values.values()), default=0)
    out['bootstrap_repeats_completed'] = int(completed)
    alpha = float(getattr(cfg, 'bootstrap_ci_alpha', 0.05))
    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    for m, vals in values.items():
        if vals:
            out[f'{m}_ci_low'] = float(np.percentile(vals, lo_q))
            out[f'{m}_ci_high'] = float(np.percentile(vals, hi_q))
    return out


def evaluate_thresholds(meta: pd.DataFrame, cif: np.ndarray, cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    rows = []
    thresholds = list(getattr(cfg, 'risk_thresholds', (0.05, 0.10, 0.20, 0.30)))
    for h in cfg.horizons:
        bin_idx = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
        p = cif[:, bin_idx]
        y, include = binary_label_at_horizon(meta, h)
        y = y[include]; p = p[include]
        for threshold in thresholds:
            pred = p >= float(threshold)
            tp = int(np.sum(pred & (y == 1)))
            fp = int(np.sum(pred & (y == 0)))
            tn = int(np.sum((~pred) & (y == 0)))
            fn = int(np.sum((~pred) & (y == 1)))
            rows.append({
                'prefix': prefix,
                'horizon_hours': h,
                'threshold': float(threshold),
                'n': int(len(y)),
                'events': int(y.sum()),
                'alerts': int(pred.sum()),
                'alert_rate': float(pred.mean()) if len(pred) else np.nan,
                'sensitivity': float(tp / max(tp + fn, 1)),
                'specificity': float(tn / max(tn + fp, 1)),
                'ppv': float(tp / max(tp + fp, 1)),
                'npv': float(tn / max(tn + fn, 1)),
                'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_threshold_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_threshold_metrics.csv', index=False)
    return out


def evaluate_subgroups(meta: pd.DataFrame, cif: np.ndarray, cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    """Write subgroup discrimination/calibration diagnostics for clinical review."""
    if meta is None or len(meta) == 0 or cif is None or len(cif) == 0:
        return pd.DataFrame()
    meta = meta.reset_index(drop=True).copy()
    groupers: Dict[str, pd.Series] = {}
    if 'sex_female' in meta.columns:
        sex = pd.to_numeric(meta['sex_female'], errors='coerce')
        groupers['sex'] = pd.Series(np.where(sex >= 0.5, 'female', 'male'), index=meta.index)
    if 'age' in meta.columns:
        age = pd.to_numeric(meta['age'], errors='coerce')
        groupers['age_band'] = pd.cut(age, bins=[0, 65, 80, 200], labels=['<65', '65-79', '>=80'], right=False).astype('object')
    if 'race_ethnicity' in meta.columns:
        groupers['race_ethnicity'] = meta['race_ethnicity'].fillna('unknown').astype(str)
    for col in ['ckd', 'diabetes', 'chf', 'liver_disease', 'sepsis_dx', 'medical_icu', 'surgical_icu', 'baseline_source_prior', 'baseline_source_admission_fallback']:
        if col in meta.columns:
            v = pd.to_numeric(meta[col], errors='coerce').fillna(0)
            groupers[col] = pd.Series(np.where(v >= 0.5, 'yes', 'no'), index=meta.index)
    if 'baseline_source' in meta.columns:
        groupers['baseline_source'] = meta['baseline_source'].fillna('missing').astype(str)
    rows = []
    min_n = int(getattr(cfg, 'min_subgroup_n', 100))
    for h in cfg.horizons:
        bin_idx = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
        p_all = cif[:, bin_idx]
        y_all, include = binary_label_at_horizon(meta, h)
        for group_name, values in groupers.items():
            vals = values.astype('object').where(values.notna(), 'missing').astype(str).values
            for level in sorted(set(vals)):
                m = (vals == level) & include
                if int(m.sum()) < min_n:
                    continue
                y = y_all[m]
                p = p_all[m]
                row = {'prefix': prefix, 'horizon_hours': h, 'subgroup': group_name, 'level': level, 'n': int(len(y)), 'events': int(y.sum())}
                row.update(_safe_binary_metrics(y, p))
                intercept, slope = calibration_intercept_slope(y, p)
                row['calibration_intercept'] = intercept
                row['calibration_slope'] = slope
                rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_subgroup_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_subgroup_metrics.csv', index=False)
    return out



def evaluate_stay_level_predictions(meta: pd.DataFrame, cif: np.ndarray, cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    """Evaluate only the first eligible landmark per stay to reduce repeated-row optimism."""
    if meta is None or len(meta) == 0 or cif is None or len(cif) == 0 or 'stay_id' not in meta.columns:
        return pd.DataFrame()
    m = meta.reset_index(drop=True).copy()
    rows = []
    order = np.argsort(pd.to_numeric(m.get('landmark_hour', pd.Series(np.arange(len(m)))), errors='coerce').fillna(np.inf).to_numpy())
    m = m.iloc[order].reset_index(drop=True)
    cif_sorted = cif[order]
    first_idx = m.groupby('stay_id', sort=False).head(1).index.to_numpy()
    m_first = m.iloc[first_idx].reset_index(drop=True)
    cif_first = cif_sorted[first_idx]
    for h in cfg.horizons:
        bin_idx = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
        p = cif_first[:, bin_idx]
        y, include = binary_label_at_horizon(m_first, h)
        y = y[include]; p = p[include]
        row = {'prefix': prefix, 'horizon_hours': h, 'aggregation': 'first_eligible_landmark_per_stay', 'n': int(len(y)), 'events': int(y.sum())}
        row.update(_safe_binary_metrics(y, p))
        intercept, slope = calibration_intercept_slope(y, p)
        row['calibration_intercept'] = intercept
        row['calibration_slope'] = slope
        rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_stay_level_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_stay_level_metrics.csv', index=False)
    return out


def evaluate_predictions(meta: pd.DataFrame, cif: np.ndarray, cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    rows = []
    fig_dir = dirs['figures'] / prefix
    fig_dir.mkdir(parents=True, exist_ok=True)
    meta_reset = meta.reset_index(drop=True).copy()
    for h in cfg.horizons:
        bin_idx = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
        p_all = cif[:, bin_idx]
        y_all, include = binary_label_at_horizon(meta_reset, h)
        y = y_all[include]
        p = p_all[include]
        meta_eval = meta_reset.loc[include].reset_index(drop=True)
        row = {'prefix': prefix, 'horizon_hours': h, 'n': int(len(y)), 'events': int(y.sum())}
        row.update(_safe_binary_metrics(y, p))
        intercept, slope = calibration_intercept_slope(y, p)
        row['calibration_intercept'] = intercept
        row['calibration_slope'] = slope
        cluster_col = 'stay_id' if 'stay_id' in meta_eval.columns else ('hospital_id' if 'hospital_id' in meta_eval.columns else None)
        if cluster_col is not None:
            row['bootstrap_cluster_col'] = cluster_col
            row.update(cluster_bootstrap_metric_cis(y, p, meta_eval[cluster_col].values, cfg))
        else:
            row['bootstrap_cluster_col'] = 'row'
            row.update(cluster_bootstrap_metric_cis(y, p, np.arange(len(y)), cfg))
        rows.append(row)
        plot_calibration(y, p, fig_dir / f'calibration_{h}h.png', title=f'{prefix} calibration {h}h')
        plot_dca(y, p, fig_dir / f'dca_{h}h.png', title=f'{prefix} DCA {h}h')
    out = pd.DataFrame(rows)
    save_parquet(out, dirs['metrics'] / f'{prefix}_metrics.parquet')
    out.to_csv(dirs['metrics'] / f'{prefix}_metrics.csv', index=False)
    evaluate_thresholds(meta_reset, cif, cfg, prefix, dirs)
    evaluate_subgroups(meta_reset, cif, cfg, prefix, dirs)
    evaluate_stay_level_predictions(meta_reset, cif, cfg, prefix, dirs)
    return out

def plot_calibration(y: np.ndarray, p: np.ndarray, path: Path, title: str = '') -> None:
    if len(y) == 0:
        return
    df = pd.DataFrame({'y': y, 'p': p})
    try:
        df['bin'] = pd.qcut(df['p'], q=min(10, max(2, df['p'].nunique())), duplicates='drop')
    except Exception:
        return
    cal = df.groupby('bin', observed=True).agg(pred=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size')).reset_index()
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.scatter(cal['pred'], cal['obs'], s=np.maximum(20, cal['n'] / max(cal['n'].max(), 1) * 200))
    plt.xlabel('Predicted risk')
    plt.ylabel('Observed risk')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_dca(y: np.ndarray, p: np.ndarray, path: Path, title: str = '') -> None:
    if len(y) == 0:
        return
    thresholds = np.linspace(0.01, 0.50, 50)
    n = len(y)
    prevalence = y.mean()
    nb_model = []
    nb_all = []
    for pt in thresholds:
        pred = p >= pt
        tp = np.sum(pred & (y == 1))
        fp = np.sum(pred & (y == 0))
        nb = tp / n - fp / n * (pt / (1 - pt))
        nb_model.append(nb)
        nb_all.append(prevalence - (1 - prevalence) * (pt / (1 - pt)))
    plt.figure(figsize=(6, 4))
    plt.plot(thresholds, nb_model, label='model')
    plt.plot(thresholds, nb_all, label='treat all', linestyle='--')
    plt.axhline(0, label='treat none', linestyle=':')
    plt.xlabel('Threshold probability')
    plt.ylabel('Net benefit')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


## 17. Engineered baselines, SHAP, and PSM sensitivity analyses

This section creates interpretable trajectory features, trains a Random Forest baseline, generates SHAP summary/dependence/waterfall plots, and runs two PSM analyses: source-vs-target database transport balance and nephrotoxin-exposure sensitivity.


In [ ]:
# -----------------------------------------------------------------------------
# Feature engineering, baseline tree, SHAP, and PSM
# -----------------------------------------------------------------------------
def slope_from_series(values: np.ndarray, mask: np.ndarray, hours: np.ndarray) -> float:
    ok = (mask > 0) & np.isfinite(values)
    if ok.sum() < 2:
        return np.nan
    xh = hours[ok]
    y = values[ok]
    if np.std(xh) < 1e-6:
        return np.nan
    return float(np.polyfit(xh, y, deg=1)[0])



def make_simple_imputer(strategy: str = 'median') -> SimpleImputer:
    """Create an imputer that preserves column count across scikit-learn versions."""
    try:
        return SimpleImputer(strategy=strategy, keep_empty_features=True)
    except TypeError:
        # Older scikit-learn releases lack keep_empty_features. Constant imputation
        # preserves all-empty columns and avoids SHAP feature-name shape mismatches.
        return SimpleImputer(strategy='constant', fill_value=0.0)

def build_feature_table(landmarks: pd.DataFrame, cfg: Config, dirs: Dict[str, Path], name: str) -> pd.DataFrame:
    out_path = dirs['features'] / f'{name}_features.parquet'
    if out_path.exists() and not cfg.overwrite:
        return read_parquet(out_path)
    logging.info("Building engineered landmark feature table: %s", name)
    tmp_path = out_path.with_suffix('.tmp.parquet')
    landmarks_reset = landmarks.reset_index(drop=True)
    rows: List[Dict[str, Any]] = []
    flush_every = int(max(1000, min(getattr(cfg, 'feature_chunk_rows', 25_000), 100_000)))
    hours = np.arange(-cfg.n_history_steps + 1, 1, dtype=float) * cfg.grid_hours
    windows = {6: int(math.ceil(6 / cfg.grid_hours)), 12: int(math.ceil(12 / cfg.grid_hours)), 24: int(math.ceil(24 / cfg.grid_hours))}

    def _flush(sink: ParquetSink) -> None:
        nonlocal rows
        if rows:
            sink.write(pd.DataFrame(rows))
            rows = []
            gc.collect()

    with ParquetSink(tmp_path) as sink:
        for i, row in tqdm(landmarks_reset.iterrows(), total=len(landmarks_reset), desc=f'features {name}'):
            x, m, c = get_history_slice(row, cfg)
            feat: Dict[str, Any] = {
                'row_id': int(i),
                'dataset': row['dataset'],
                'stay_id': row['stay_id'],
                'subject_id': row.get('subject_id', row.get('stay_id', 'unknown')),
                'hospital_id': row['hospital_id'],
                'race_ethnicity': row.get('race_ethnicity', 'unknown'),
                'anchor_year_group': row.get('anchor_year_group', 'unknown'),
                'baseline_source': row.get('baseline_source', 'missing'),
                'landmark_hour': row['landmark_hour'],
                'event_type': row['event_type'],
                'time_to_event_hours': row['time_to_event_hours'],
                'time_bin': row['time_bin'],
            }
            for col in STATIC_COLS:
                feat[col] = row.get(col, 0.0)
            for var in CONTINUOUS_VARS:
                j = VAR_INDEX[var]
                obs_full_history = (m[:, j] > 0) & np.isfinite(x[:, j])

                feat[f'{var}_last'] = np.nan
                feat[f'{var}_min24'] = np.nan
                feat[f'{var}_max24'] = np.nan
                feat[f'{var}_obs_count24'] = 0.0

                if obs_full_history.any():
                    feat[f'{var}_last'] = float(x[np.where(obs_full_history)[0][-1], j])

                obs_last24h = obs_full_history[-windows[24]:]
                if obs_last24h.any():
                    values_last24h = np.where(obs_last24h, x[-windows[24]:, j], np.nan)
                    feat[f'{var}_min24'] = float(np.nanmin(values_last24h))
                    feat[f'{var}_max24'] = float(np.nanmax(values_last24h))
                    feat[f'{var}_obs_count24'] = float(obs_last24h.sum())
            for w_h, w in windows.items():
                j = VAR_INDEX['creatinine']
                feat[f'creatinine_slope_{w_h}h'] = slope_from_series(x[-w:, j], m[-w:, j], hours[-w:])
                j_u = VAR_INDEX['urine_output']
                uo_obs = (m[-w:, j_u] > 0) & np.isfinite(x[-w:, j_u])
                weight = float(row.get('weight_kg', np.nan)) if 'weight_kg' in row else np.nan
                if not np.isfinite(weight):
                    jw = VAR_INDEX['weight_kg']; weight_vals = x[:, jw][m[:, jw] > 0]
                    weight = float(weight_vals[-1]) if len(weight_vals) else 75.0
                feat[f'urine_rate_{w_h}h_mlkgph'] = float(np.nansum(np.where(uo_obs, x[-w:, j_u], 0.0)) / max(weight * w_h, 1e-6)) if uo_obs.any() else np.nan
                j_map = VAR_INDEX['map']
                map_vals = x[-w:, j_map]
                map_obs = (m[-w:, j_map] > 0) & np.isfinite(map_vals)
                feat[f'map_deficit_area_{w_h}h'] = float(np.nansum(np.maximum(65.0 - map_vals[map_obs], 0.0)) * cfg.grid_hours) if map_obs.any() else np.nan
                for var in ['vasopressor', 'diuretic', 'nephrotoxin', 'antibiotic', 'mechanical_ventilation']:
                    jv = VAR_INDEX[var]
                    feat[f'{var}_any_{w_h}h'] = float(np.nanmax(x[-w:, jv]) > 0.5)
                    feat[f'{var}_burden_{w_h}h'] = float(np.nansum(x[-w:, jv]) * cfg.grid_hours)
                j_nf = VAR_INDEX['net_fluid']
                nf_obs = (m[-w:, j_nf] > 0) & np.isfinite(x[-w:, j_nf])
                feat[f'net_fluid_sum_{w_h}h'] = float(np.nansum(x[-w:, j_nf][nf_obs])) if nf_obs.any() else np.nan
            for var in ['creatinine', 'bun', 'lactate', 'ph', 'pao2', 'paco2', 'urine_output']:
                j = VAR_INDEX[var]
                feat[f'{var}_measured_last6h'] = float((m[-windows[6]:, j] > 0).any())
                feat[f'{var}_obs_count_history'] = float((m[:, j] > 0).sum())
            feat['shock_index_last'] = feat.get('heart_rate_last', np.nan) / max(feat.get('map_last', np.nan), 1e-6) if np.isfinite(feat.get('heart_rate_last', np.nan)) and np.isfinite(feat.get('map_last', np.nan)) else np.nan
            feat['vent_acidosis_interaction'] = feat.get('mechanical_ventilation_any_24h', 0.0) * float(feat.get('ph_last', np.nan) < 7.30) if np.isfinite(feat.get('ph_last', np.nan)) else np.nan
            feat['sepsis_low_map_interaction'] = feat.get('sepsis_dx', 0.0) * float(feat.get('map_min24', np.nan) < 65) if np.isfinite(feat.get('map_min24', np.nan)) else np.nan
            rows.append(feat)
            if len(rows) >= flush_every:
                _flush(sink)
        _flush(sink)
    if _parquet_row_count(tmp_path) <= 0:
        raise RuntimeError(f'Feature construction for {name} produced no rows: {tmp_path}')
    tmp_path.replace(out_path)
    return read_parquet(out_path)

def prepare_binary_features(feat: pd.DataFrame, horizon: int) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    y = ((feat['event_type'].values == 1) & (feat['time_to_event_hours'].values <= horizon)).astype(int)
    drop = {'row_id', 'dataset', 'stay_id', 'hospital_id', 'race_ethnicity', 'event_type', 'time_to_event_hours', 'time_bin'}
    cols = [c for c in feat.columns if c not in drop and pd.api.types.is_numeric_dtype(feat[c])]
    X = feat[cols].replace([np.inf, -np.inf], np.nan)
    return X, y, cols



def _binary_rf_metrics_from_columns(train_feat: pd.DataFrame, test_feat: pd.DataFrame, cols: List[str], horizon: int, cfg: Config) -> Dict[str, Any]:
    y_train = ((train_feat['event_type'].values == 1) & (train_feat['time_to_event_hours'].values <= horizon)).astype(int)
    y_test = ((test_feat['event_type'].values == 1) & (test_feat['time_to_event_hours'].values <= horizon)).astype(int)
    if len(cols) == 0 or len(np.unique(y_train)) < 2 or len(test_feat) == 0:
        return {'auc': np.nan, 'auprc': np.nan, 'brier': np.nan, 'n_features': int(len(cols)), 'n_test': int(len(y_test)), 'events_test': int(y_test.sum()) if len(y_test) else 0}
    X_train = train_feat[cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_feat[cols].replace([np.inf, -np.inf], np.nan)
    pipe = Pipeline([
        ('imputer', make_simple_imputer('median')),
        ('rf', RandomForestClassifier(n_estimators=max(100, min(int(getattr(cfg, 'rf_n_estimators', 300)), 300)), min_samples_leaf=25, max_features='sqrt', n_jobs=max(1, cfg.n_jobs - 1), class_weight='balanced_subsample', random_state=cfg.seed))
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)
    p = proba[:, 1] if proba.shape[1] > 1 else np.zeros(len(X_test), dtype=float)
    met = _safe_binary_metrics(y_test, p)
    met.update({'n_features': int(len(cols)), 'n_test': int(len(y_test)), 'events_test': int(y_test.sum()) if len(y_test) else 0})
    return met


def run_measurement_process_bias_audit(train_feat: pd.DataFrame, test_feat: pd.DataFrame, cfg: Config, dirs: Dict[str, Path], prefix: str) -> pd.DataFrame:
    """Quantify how predictive measurement frequency alone is; high AUC signals EHR process bias."""
    meas_cols = [c for c in train_feat.columns if ('measured_' in c or 'obs_count' in c) and c in test_feat.columns and pd.api.types.is_numeric_dtype(train_feat[c])]
    met = _binary_rf_metrics_from_columns(train_feat, test_feat, meas_cols, cfg.shap_horizon_hours, cfg)
    met.update({'prefix': prefix, 'horizon_hours': cfg.shap_horizon_hours, 'audit': 'measurement_process_only', 'warning': bool(np.isfinite(met.get('auc', np.nan)) and met['auc'] >= float(cfg.measurement_bias_auc_warn))})
    out = pd.DataFrame([met])
    out.to_csv(dirs['metrics'] / f'{prefix}_measurement_process_bias_audit.csv', index=False)
    return out


def run_treatment_feature_ablation(train_feat: pd.DataFrame, test_feat: pd.DataFrame, cfg: Config, dirs: Dict[str, Path], prefix: str) -> pd.DataFrame:
    """Compare all engineered features with treatment/documentation features removed."""
    X_train, y_train, cols = prepare_binary_features(train_feat, cfg.shap_horizon_hours)
    treatment_patterns = ['vasopressor', 'diuretic', 'nephrotoxin', 'antibiotic', 'mechanical_ventilation', 'rrt']
    reduced_cols = [c for c in cols if not any(p in c for p in treatment_patterns)]
    all_met = _binary_rf_metrics_from_columns(train_feat, test_feat, cols, cfg.shap_horizon_hours, cfg)
    red_met = _binary_rf_metrics_from_columns(train_feat, test_feat, reduced_cols, cfg.shap_horizon_hours, cfg)
    delta_auc = all_met.get('auc', np.nan) - red_met.get('auc', np.nan) if np.isfinite(all_met.get('auc', np.nan)) and np.isfinite(red_met.get('auc', np.nan)) else np.nan
    out = pd.DataFrame([
        {'prefix': prefix, 'horizon_hours': cfg.shap_horizon_hours, 'model': 'all_engineered_features', **all_met},
        {'prefix': prefix, 'horizon_hours': cfg.shap_horizon_hours, 'model': 'without_treatment_documentation_features', **red_met},
        {'prefix': prefix, 'horizon_hours': cfg.shap_horizon_hours, 'model': 'delta_all_minus_reduced', 'auc': delta_auc, 'warning': bool(np.isfinite(delta_auc) and abs(delta_auc) >= float(cfg.treatment_ablation_auc_delta_warn))},
    ])
    out.to_csv(dirs['metrics'] / f'{prefix}_treatment_feature_ablation.csv', index=False)
    return out


def run_baseline_source_sensitivity(meta: pd.DataFrame, cif: Optional[np.ndarray], cfg: Config, dirs: Dict[str, Path], prefix: str) -> pd.DataFrame:
    """Audit whether baseline creatinine source changes calibration/performance materially."""
    if meta is None or len(meta) == 0 or cif is None or len(cif) == 0 or 'baseline_source' not in meta.columns:
        return pd.DataFrame()
    rows = []
    m = meta.reset_index(drop=True).copy()
    for h in cfg.horizons:
        bin_idx = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
        p_all = cif[:, bin_idx]
        y_all, include = binary_label_at_horizon(m, h)
        groups = m['baseline_source'].fillna('missing').astype(str).values
        for source in sorted(set(groups)):
            sel = (groups == source) & include
            if int(sel.sum()) < int(getattr(cfg, 'min_subgroup_n', 100)):
                continue
            y = y_all[sel]
            p = p_all[sel]
            met = _safe_binary_metrics(y, p)
            intercept, slope = calibration_intercept_slope(y, p)
            rows.append({'prefix': prefix, 'horizon_hours': h, 'baseline_source': source, 'n': int(len(y)), 'events': int(y.sum()), **met, 'calibration_intercept': intercept, 'calibration_slope': slope})
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(dirs['metrics'] / f'{prefix}_baseline_source_sensitivity.csv', index=False)
    return out


def train_tree_shap(
    train_feat: pd.DataFrame,
    test_feat: pd.DataFrame,
    cfg: Config,
    dirs: Dict[str, Path],
    prefix: str,
) -> pd.DataFrame:
    out_dir = dirs['figures'] / prefix / 'shap'
    out_dir.mkdir(parents=True, exist_ok=True)
    X_train, y_train, cols = prepare_binary_features(train_feat, cfg.shap_horizon_hours)
    X_test, y_test, _ = prepare_binary_features(test_feat, cfg.shap_horizon_hours)
    if X_train.shape[1] == 0 or len(np.unique(y_train)) < 2:
        logging.warning('Skipping RF/SHAP for %s: insufficient features or only one training class.', prefix)
        metrics = pd.DataFrame([{
            'prefix': prefix, 'horizon_hours': cfg.shap_horizon_hours,
            'n_train': int(len(y_train)), 'n_test': int(len(y_test)),
            'events_test': int(y_test.sum()) if len(y_test) else 0,
            'auc': np.nan, 'auprc': np.nan, 'brier': np.nan,
        }])
        metrics.to_csv(dirs['metrics'] / f'{prefix}_rf_shap_model_metrics.csv', index=False)
        return metrics
    pipe = Pipeline([
        ('imputer', make_simple_imputer('median')),
        ('rf', RandomForestClassifier(
            n_estimators=int(getattr(cfg, 'rf_n_estimators', 300)),
            min_samples_leaf=25,
            max_features='sqrt',
            n_jobs=max(1, cfg.n_jobs - 1),
            class_weight='balanced_subsample',
            random_state=cfg.seed,
        ))
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)
    p = proba[:, 1] if proba.shape[1] > 1 else np.zeros(len(X_test), dtype=float)
    rows = [{
        'prefix': prefix,
        'horizon_hours': cfg.shap_horizon_hours,
        'n_train': int(len(y_train)),
        'n_test': int(len(y_test)),
        'events_test': int(y_test.sum()),
        'auc': float(roc_auc_score(y_test, p)) if len(np.unique(y_test)) > 1 else np.nan,
        'auprc': float(average_precision_score(y_test, p)) if len(np.unique(y_test)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_test, np.clip(p, 0, 1))),
    }]
    metrics = pd.DataFrame(rows)
    metrics.to_csv(dirs['metrics'] / f'{prefix}_rf_shap_model_metrics.csv', index=False)
    run_measurement_process_bias_audit(train_feat, test_feat, cfg, dirs, prefix)
    run_treatment_feature_ablation(train_feat, test_feat, cfg, dirs, prefix)
    with open(dirs['models'] / f'{prefix}_rf_model.pkl', 'wb') as f:
        pickle.dump({'pipeline': pipe, 'columns': cols}, f)

    # SHAP explanation on sampled test rows.
    try:
        import shap
        n = min(cfg.shap_sample_size, len(X_test))
        sample_idx = np.random.default_rng(cfg.seed).choice(np.arange(len(X_test)), size=n, replace=False)
        X_sample_raw = X_test.iloc[sample_idx]
        X_sample = pd.DataFrame(pipe.named_steps['imputer'].transform(X_sample_raw), columns=cols)
        rf = pipe.named_steps['rf']
        explainer = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X_sample, check_additivity=False)
        if isinstance(shap_values, list):
            sv = shap_values[1]
        elif getattr(shap_values, 'ndim', 0) == 3:
            sv = shap_values[:, :, 1]
        else:
            sv = shap_values
        plt.figure()
        shap.summary_plot(sv, X_sample, show=False, max_display=30)
        plt.tight_layout()
        plt.savefig(out_dir / 'shap_summary_dot.png', dpi=220, bbox_inches='tight')
        plt.close()
        plt.figure()
        shap.summary_plot(sv, X_sample, show=False, plot_type='bar', max_display=30)
        plt.tight_layout()
        plt.savefig(out_dir / 'shap_summary_bar.png', dpi=220, bbox_inches='tight')
        plt.close()
        mean_abs = np.abs(sv).mean(axis=0)
        top_features = [cols[i] for i in np.argsort(mean_abs)[::-1][:5]]
        pd.DataFrame({'feature': cols, 'mean_abs_shap': mean_abs}).sort_values('mean_abs_shap', ascending=False).to_csv(out_dir / 'shap_feature_importance.csv', index=False)
        for feat_name in top_features[:3]:
            try:
                plt.figure()
                shap.dependence_plot(feat_name, sv, X_sample, show=False)
                plt.tight_layout()
                plt.savefig(out_dir / f'shap_dependence_{feat_name[:80].replace("/", "_")}.png', dpi=220, bbox_inches='tight')
                plt.close()
            except Exception as exc:
                logging.warning("SHAP dependence failed for %s: %s", feat_name, exc)
        # Local explanation for the highest predicted-risk sample.
        hi_rel = int(np.argmax(p[sample_idx]))
        try:
            exp = shap.Explanation(values=sv[hi_rel], base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value, data=X_sample.iloc[hi_rel].values, feature_names=cols)
            shap.plots.waterfall(exp, show=False, max_display=20)
            plt.tight_layout()
            plt.savefig(out_dir / 'shap_waterfall_high_risk.png', dpi=220, bbox_inches='tight')
            plt.close()
        except Exception as exc:
            logging.warning("SHAP waterfall failed: %s", exc)
    except Exception as exc:
        logging.exception("SHAP failed for %s: %s", prefix, exc)
    return metrics


def standardized_mean_differences(df: pd.DataFrame, treatment: np.ndarray, cols: List[str]) -> pd.DataFrame:
    rows = []
    for c in cols:
        x = pd.to_numeric(df[c], errors='coerce')
        xt = x[treatment == 1]
        xc = x[treatment == 0]
        mt, mc = xt.mean(), xc.mean()
        vt, vc = xt.var(), xc.var()
        pooled = math.sqrt(max((vt + vc) / 2.0, 1e-12)) if np.isfinite(vt) and np.isfinite(vc) else np.nan
        smd = (mt - mc) / pooled if pooled and np.isfinite(pooled) else np.nan
        rows.append({'feature': c, 'mean_treated': mt, 'mean_control': mc, 'smd': smd})
    return pd.DataFrame(rows)


def propensity_match(df: pd.DataFrame, treat_col: str, covariates: List[str], cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    d = df.copy().replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=[treat_col]).copy()
    d[treat_col] = d[treat_col].astype(int)
    covariates = [c for c in covariates if c in d.columns and pd.api.types.is_numeric_dtype(d[c])]
    X = d[covariates]
    y = d[treat_col].values
    if len(np.unique(y)) < 2 or len(covariates) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    pipe = Pipeline([
        ('imputer', make_simple_imputer('median')),
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2000, class_weight='balanced')),
    ])
    pipe.fit(X, y)
    ps = pipe.predict_proba(X)[:, 1]
    logit_ps = np.log(np.clip(ps, 1e-6, 1 - 1e-6) / np.clip(1 - ps, 1e-6, 1 - 1e-6))
    d['propensity'] = ps
    d['logit_propensity'] = logit_ps
    sd_logit = np.nanstd(logit_ps)
    caliper = cfg.psm_caliper_sd * sd_logit
    treated = d.loc[d[treat_col] == 1].copy()
    control = d.loc[d[treat_col] == 0].copy()
    if len(treated) == 0 or len(control) == 0:
        return pd.DataFrame(), standardized_mean_differences(d, y, covariates), pd.DataFrame()
    nn = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(control[['logit_propensity']].values)
    dist, ind = nn.kneighbors(treated[['logit_propensity']].values)
    pairs = []
    used_controls = set()
    for t_idx, distance, cpos in zip(treated.index, dist[:, 0], ind[:, 0]):
        c_idx = control.index[int(cpos)]
        if distance <= caliper and c_idx not in used_controls:
            pairs.append((t_idx, c_idx, float(distance)))
            used_controls.add(c_idx)
    if not pairs:
        return pd.DataFrame(), standardized_mean_differences(d, y, covariates), pd.DataFrame()
    pair_df = pd.DataFrame(pairs, columns=['treated_index', 'control_index', 'distance'])
    matched = pd.concat([d.loc[pair_df['treated_index']], d.loc[pair_df['control_index']]], ignore_index=False).copy()
    before = standardized_mean_differences(d, y, covariates)
    after = standardized_mean_differences(matched, matched[treat_col].values, covariates)
    balance = before.merge(after[['feature', 'smd']], on='feature', how='left', suffixes=('_before', '_after'))
    return matched, balance, pair_df


def run_psm_sensitivity(
    source_feat: pd.DataFrame,
    target_feat: pd.DataFrame,
    target_pred: Optional[pd.DataFrame],
    cfg: Config,
    dirs: Dict[str, Path],
    prefix: str,
) -> None:
    psm_dir = dirs['psm'] / prefix
    psm_dir.mkdir(parents=True, exist_ok=True)
    # Stay-level first landmark matching between source and target databases.
    src = source_feat.sort_values('landmark_hour').groupby('stay_id', as_index=False).head(1).copy()
    tgt = target_feat.sort_values('landmark_hour').groupby('stay_id', as_index=False).head(1).copy()
    src['target_dataset'] = 0
    tgt['target_dataset'] = 1
    combo = pd.concat([src, tgt], ignore_index=True, sort=False)
    covars = [c for c in combo.columns if c not in {'row_id', 'dataset', 'stay_id', 'hospital_id', 'race_ethnicity', 'event_type', 'time_to_event_hours', 'time_bin', 'target_dataset'} and pd.api.types.is_numeric_dtype(combo[c])]
    matched, balance, pairs = propensity_match(combo, 'target_dataset', covars, cfg)
    if not matched.empty:
        matched.to_csv(psm_dir / 'dataset_transport_matched_stays.csv', index=False)
        pairs.to_csv(psm_dir / 'dataset_transport_pairs.csv', index=False)
    balance.to_csv(psm_dir / 'dataset_transport_balance.csv', index=False)

    # Exposure PSM: nephrotoxin exposure in the preceding 24h, if available.
    exposure_col = 'nephrotoxin_any_24h'
    if exposure_col in target_feat.columns:
        exp_df = target_feat.copy()
        exp_df['exposed'] = (exp_df[exposure_col].fillna(0) > 0.5).astype(int)
        exp_covars = [c for c in covars if c in exp_df.columns and c != exposure_col]
        matched_exp, bal_exp, pairs_exp = propensity_match(exp_df, 'exposed', exp_covars, cfg)
        if not matched_exp.empty:
            matched_exp.to_csv(psm_dir / 'nephrotoxin_matched_landmarks.csv', index=False)
            pairs_exp.to_csv(psm_dir / 'nephrotoxin_pairs.csv', index=False)
        bal_exp.to_csv(psm_dir / 'nephrotoxin_balance.csv', index=False)


## 18. Experiment orchestration

The orchestration layer runs the full pipeline. It supports strict external validation and pooled-unlabeled SSL adaptation as distinct experiments:

- `strict_mimic_to_eicu`
- `strict_eicu_to_mimic`
- `pooled_ssl_mimic_to_eicu`
- `pooled_ssl_eicu_to_mimic`

The pooled SSL settings expose target-domain unlabeled trajectories during pretraining only; they should be reported as adaptation experiments rather than strict external validation.


In [ ]:
# -----------------------------------------------------------------------------
# Experiment orchestration
# -----------------------------------------------------------------------------
def make_loader(ds: Dataset, shuffle: bool, cfg: Config) -> DataLoader:
    workers = int(getattr(cfg, 'dataloader_workers', 0))
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle, num_workers=workers, pin_memory=cfg.device.startswith('cuda'), drop_last=False)


def train_one_experiment(
    name: str,
    source_name: str,
    target_name: str,
    source_lm: pd.DataFrame,
    target_lm: pd.DataFrame,
    cfg: Config,
    dirs: Dict[str, Path],
    pooled_ssl: bool = False,
) -> None:
    exp_dir = dirs['models'] / name
    exp_dir.mkdir(parents=True, exist_ok=True)
    logging.info("Experiment %s | source=%s target=%s pooled_ssl=%s", name, source_name, target_name, pooled_ssl)
    splits = split_landmarks(source_lm, source_name, cfg)
    validate_split_integrity(source_lm, splits, source_name, cfg)
    src_train = source_lm.iloc[splits['train']].reset_index(drop=True)
    src_val = source_lm.iloc[splits['val']].reset_index(drop=True)
    src_test = source_lm.iloc[splits['test']].reset_index(drop=True)
    # Target is never supervised in strict transport. For reporting, evaluate all eligible target landmarks.
    tgt_all = target_lm.reset_index(drop=True)

    scaler = compute_scaler(src_train, cfg, exp_dir / 'scaler.json')
    train_ds = AKILandmarkDataset(src_train, cfg, scaler, augment=True)
    val_ds = AKILandmarkDataset(src_val, cfg, scaler, augment=False)
    test_ds = AKILandmarkDataset(src_test, cfg, scaler, augment=False)
    target_ds = AKILandmarkDataset(tgt_all, cfg, scaler, augment=False)

    ssl_datasets: List[Dataset] = [train_ds]
    if pooled_ssl:
        # Unlabeled target-domain exposure is explicitly named in the experiment name.
        ssl_datasets.append(target_ds)
    ssl_train_ds = ConcatDataset(ssl_datasets) if len(ssl_datasets) > 1 else train_ds
    ssl_model = SSLPretrainModel(input_dim=N_VARS * 4, static_dim=len(STATIC_COLS), cfg=cfg)
    ssl_path = exp_dir / 'ssl_pretrained.pt'
    ssl_model = train_ssl(ssl_model, make_loader(ssl_train_ds, shuffle=True, cfg=cfg), make_loader(val_ds, shuffle=False, cfg=cfg), cfg, ssl_path)

    surv_model = SurvivalTransformer(ssl_model.encoder, cfg, n_causes=3)
    surv_path = exp_dir / 'survival_transformer.pt'
    surv_model = train_survival(surv_model, make_loader(train_ds, True, cfg), make_loader(val_ds, False, cfg), cfg, surv_path)

    for split_name, meta, ds in [('internal_test', src_test, test_ds), ('external', tgt_all, target_ds)]:
        cif = predict_survival(surv_model, ds, cfg)
        pred = meta.reset_index(drop=True).copy()
        for b in range(cfg.n_surv_bins):
            pred[f'cif_bin_{b+1}'] = cif[:, b]
        for h in cfg.horizons:
            bi = int(min(cfg.n_surv_bins - 1, math.ceil(h / cfg.landmark_step_hours) - 1))
            pred[f'risk_{h}h'] = cif[:, bi]
        pred_path = dirs['predictions'] / f'{name}_{split_name}_predictions.parquet'
        save_parquet(pred, pred_path)
        evaluate_predictions(pred, cif, cfg, f'{name}_{split_name}', dirs)
        run_baseline_source_sensitivity(pred, cif, cfg, dirs, f'{name}_{split_name}')

    # Engineered baseline + SHAP: source train to target external.
    source_feat_train = build_feature_table(src_train, cfg, dirs, f'{name}_source_train')
    target_feat_all = build_feature_table(tgt_all, cfg, dirs, f'{name}_target_external')
    train_tree_shap(source_feat_train, target_feat_all, cfg, dirs, f'{name}_external')
    target_pred = read_parquet(dirs['predictions'] / f'{name}_external_predictions.parquet')
    run_psm_sensitivity(source_feat_train, target_feat_all, target_pred, cfg, dirs, f'{name}_external')



def _parquet_row_count(path: Path) -> int:
    """Return parquet row count using metadata; return -1 if the file is unreadable."""
    path = Path(path)
    if not path.exists():
        return 0
    try:
        return int(pq.ParquetFile(path).metadata.num_rows)
    except Exception as exc:
        logging.warning("Could not read parquet metadata for %s: %s", path, exc)
        return -1


def _unlink_quietly(path: Path) -> None:
    path = Path(path)
    try:
        if path.exists():
            path.unlink()
    except Exception as exc:
        logging.warning("Could not remove stale cache file %s: %s", path, exc)




def _clinical_cache_version_path(dataset: str, dirs: Dict[str, Path]) -> Path:
    return dirs['cache'] / f'{dataset}_clinical_pipeline.version'


def _clinical_cache_version_status(dataset: str, dirs: Dict[str, Path]) -> Tuple[bool, str]:
    path = _clinical_cache_version_path(dataset, dirs)
    if not path.exists():
        return False, 'missing strict clinical cache-version marker'
    found = path.read_text().strip()
    if found != CLINICAL_PIPELINE_VERSION:
        return False, f'clinical cache version mismatch: found={found!r}, expected={CLINICAL_PIPELINE_VERSION!r}'
    return True, f'clinical cache version OK: {found}'

def _dataset_parquet_cache_paths(dataset: str, dirs: Dict[str, Path]) -> List[Path]:
    if dataset == 'mimic':
        return [
            dirs['cohort'] / 'mimic_cohort_base.parquet',
            dirs['cohort'] / 'mimic_baseline_weight.parquet',
            dirs['cohort'] / 'mimic_cohort_final.parquet',
            dirs['events'] / 'mimic_binned_events_raw.parquet',
            dirs['events'] / 'mimic_binned_events.parquet',
            dirs['landmarks'] / 'mimic_landmarks.parquet',
            dirs['landmarks'] / 'mimic_stay_summary.parquet',
        ]
    if dataset == 'eicu':
        return [
            dirs['cohort'] / 'eicu_cohort_base.parquet',
            dirs['cohort'] / 'eicu_baseline.parquet',
            dirs['cohort'] / 'eicu_cohort_final.parquet',
            dirs['events'] / 'eicu_binned_events_raw.parquet',
            dirs['events'] / 'eicu_binned_events.parquet',
            dirs['landmarks'] / 'eicu_landmarks.parquet',
            dirs['landmarks'] / 'eicu_stay_summary.parquet',
        ]
    raise ValueError(f"Unknown dataset: {dataset}")


def _purge_empty_or_corrupt_dataset_cache(dataset: str, dirs: Dict[str, Path]) -> None:
    """Remove stale parquet artifacts and known-incompatible cache files.

    This is stricter than a simple row-count check. The failure reported by the
    user showed that a large nonempty MIMIC event parquet can still be unusable
    if it was produced with old identifier joins and has no usable creatinine
    rows for ICU stays.
    """
    version_ok, version_reason = _clinical_cache_version_status(dataset, dirs)
    if not version_ok:
        marker_missing = version_reason.startswith('missing strict clinical cache-version marker')
        event_ok, event_reason = _event_file_status(dataset, dirs)
        final_cohort = dirs['cohort'] / f'{dataset}_cohort_final.parquet'
        landmarks_path = dirs['landmarks'] / f'{dataset}_landmarks.parquet'
        cohort_ok = _parquet_row_count(final_cohort) > 0
        landmarks_ok = _parquet_row_count(landmarks_path) > 0
        if marker_missing and event_ok and cohort_ok and landmarks_ok:
            logging.warning(
                "Missing %s cache marker but core cohort/event/landmark files are valid (%s); writing current marker instead of purging.",
                dataset, event_reason,
            )
            _clinical_cache_version_path(dataset, dirs).write_text(CLINICAL_PIPELINE_VERSION)
        else:
            logging.warning("Removing cached %s artifacts because strict clinical phenotype settings changed or cache is incomplete: %s; event_status=%s", dataset, version_reason, event_reason)
            for path in _dataset_parquet_cache_paths(dataset, dirs):
                _unlink_quietly(path)
            _unlink_quietly(_clinical_cache_version_path(dataset, dirs))
            _remove_landmark_outputs(dataset, dirs)
    for path in _dataset_parquet_cache_paths(dataset, dirs):
        if path.exists():
            n = _parquet_row_count(path)
            if n <= 0:
                logging.warning("Removing empty/corrupt cached %s file with %s rows: %s", dataset, n, path)
                _unlink_quietly(path)
    if dataset == 'mimic':
        version_path = dirs['cache'] / 'mimic_item_maps.version'
        if (dirs['cache'] / 'mimic_item_maps.json').exists() and (not version_path.exists() or version_path.read_text().strip() != MIMIC_ITEM_MAP_VERSION):
            logging.warning("Removing stale MIMIC item-map cache; it predates %s", MIMIC_ITEM_MAP_VERSION)
            for path in _dataset_nonparquet_cache_paths('mimic', dirs):
                _unlink_quietly(path)
    ok, reason = _event_file_status(dataset, dirs)
    event_path = dirs['events'] / f'{dataset}_binned_events.parquet'
    if event_path.exists() and not ok:
        logging.warning("Removing invalid cached %s event table: %s", dataset, reason)
        _unlink_quietly(event_path)
        _unlink_quietly(dirs['events'] / f'{dataset}_binned_events_raw.parquet')
        _remove_landmark_outputs(dataset, dirs)


def _dataset_nonparquet_cache_paths(dataset: str, dirs: Dict[str, Path]) -> List[Path]:
    if dataset == 'mimic':
        return [dirs['cache'] / 'mimic_item_maps.json', dirs['cache'] / 'mimic_item_maps.version']
    return []


def _event_file_status(dataset: str, dirs: Dict[str, Path]) -> Tuple[bool, str]:
    path = dirs['events'] / f'{dataset}_binned_events.parquet'
    if not path.exists():
        return False, 'missing binned-event parquet'
    n = _parquet_row_count(path)
    if n <= 0:
        return False, f'empty or unreadable binned-event parquet; row_count={n}'
    try:
        ev = pd.read_parquet(path, columns=['stay_id', 'variable'])
        vars_present = set(ev['variable'].dropna().astype(str).unique().tolist())
        missing = {'creatinine', 'urine_output'} - vars_present
        if missing:
            return False, f'binned-event parquet lacks critical variables {sorted(missing)}'
        scr = set(ev.loc[ev['variable'].eq('creatinine'), 'stay_id'].astype(str).unique())
        uo = set(ev.loc[ev['variable'].eq('urine_output'), 'stay_id'].astype(str).unique())
        if len(scr) == 0 or len(uo) == 0:
            return False, f'critical variable stay counts are zero: creatinine={len(scr)}, urine_output={len(uo)}'
        return True, f'usable; rows={n}; creatinine_stays={len(scr)}; urine_output_stays={len(uo)}'
    except Exception as exc:
        return False, f'could not inspect critical variables in binned events: {exc}'

def _remove_landmark_outputs(dataset: str, dirs: Dict[str, Path]) -> None:
    for suffix in ['landmarks', 'stay_summary']:
        _unlink_quietly(dirs['landmarks'] / f'{dataset}_{suffix}.parquet')


def _landmark_file_status(dataset: str, dirs: Dict[str, Path]) -> Tuple[bool, str]:
    """Check clinical version, row count, and whether saved grid_path files are still present."""
    version_ok, version_reason = _clinical_cache_version_status(dataset, dirs)
    if not version_ok:
        return False, version_reason
    path = dirs['landmarks'] / f'{dataset}_landmarks.parquet'
    if not path.exists():
        return False, 'missing landmark parquet'
    n = _parquet_row_count(path)
    if n <= 0:
        return False, f'empty or unreadable landmark parquet; row_count={n}'
    try:
        pf = pq.ParquetFile(path)
        cols = set(pf.schema.names)
        if 'grid_path' not in cols:
            return False, 'landmark parquet lacks grid_path column'
        head = pf.read_row_group(0, columns=['grid_path']).to_pandas().head(20)
        grid_paths = [Path(str(p)) for p in head['grid_path'].dropna().tolist()]
        if not grid_paths:
            return False, 'no non-missing grid_path values in first parquet row group'
        missing = [str(p) for p in grid_paths if not p.exists()]
        if missing:
            return False, f'cached grid files are missing, first missing={missing[0]}'
    except Exception as exc:
        return False, f'could not validate landmark grid paths: {exc}'
    return True, f'usable; rows={n}'


def _diagnose_dataset(dataset: str, dirs: Dict[str, Path]) -> str:
    pieces = []
    for path in _dataset_parquet_cache_paths(dataset, dirs):
        if path.exists():
            pieces.append(f"{path.name}: rows={_parquet_row_count(path)}")
        else:
            pieces.append(f"{path.name}: missing")
    ok_events, event_reason = _event_file_status(dataset, dirs)
    pieces.append(f"event_status={event_reason}")
    excl = dirs['landmarks'] / f'{dataset}_exclusions.json'
    if excl.exists():
        try:
            with open(excl, 'r') as f:
                pieces.append(f"exclusions={json.load(f)}")
        except Exception as exc:
            pieces.append(f"exclusions unreadable: {exc}")
    return '; '.join(pieces)


def _assert_landmarks_nonempty(dataset: str, lm: pd.DataFrame, dirs: Dict[str, Path]) -> None:
    if lm is None or lm.empty:
        raise RuntimeError(
            f"{dataset} landmark dataset is empty after automatic cache repair/rebuild. "
            f"Diagnostics: {_diagnose_dataset(dataset, dirs)}. "
            "This usually means the cohort was completely excluded by baseline creatinine, "
            "urine-output availability, severe AKI/RRT before 6h, or an item/table mapping mismatch."
        )


def build_dataset_data(dataset: str, cfg: Config, dirs: Dict[str, Path], force_landmarks: bool = False) -> pd.DataFrame:
    """Build one database at a time, preserving the other database's completed caches.

    The first pass reuses nonempty cached cohort/event files. If that still yields an empty
    landmark table, the function performs one dataset-specific full cache rebuild. This is
    intentional: interrupted or earlier buggy runs can leave nonempty-but-incompatible caches.
    """
    dataset = dataset.lower()

    def _build_once(local_cfg: Config) -> pd.DataFrame:
        if dataset == 'mimic':
            mimic_base = build_mimic_cohort(local_cfg, dirs)
            mimic_maps = build_mimic_item_maps(local_cfg, dirs)
            mimic_base_w = extract_mimic_baseline_and_weight(local_cfg, dirs, mimic_base, mimic_maps)
            mimic_cohort = finalize_cohort('mimic', local_cfg, dirs, mimic_base, mimic_base_w)
            mimic_events = extract_mimic_events(local_cfg, dirs, mimic_cohort, mimic_maps)
            return build_grids_and_landmarks('mimic', local_cfg, dirs, mimic_cohort, mimic_events)
        if dataset == 'eicu':
            eicu_base = build_eicu_cohort(local_cfg, dirs)
            eicu_baseline = extract_eicu_baseline(local_cfg, dirs, eicu_base)
            eicu_cohort = finalize_cohort('eicu', local_cfg, dirs, eicu_base, eicu_baseline)
            eicu_events = extract_eicu_events(local_cfg, dirs, eicu_cohort)
            return build_grids_and_landmarks('eicu', local_cfg, dirs, eicu_cohort, eicu_events)
        raise ValueError(f"Unknown dataset: {dataset}")

    if cfg.overwrite:
        force_landmarks = True
    else:
        # Important fix for interrupted notebook runs: old empty parquet files must not be reused.
        _purge_empty_or_corrupt_dataset_cache(dataset, dirs)

    if force_landmarks:
        _remove_landmark_outputs(dataset, dirs)

    lm = _build_once(cfg)

    if (lm is None or lm.empty) and not cfg.overwrite:
        logging.warning(
            "%s landmarks are still empty after cache-aware rebuild. Performing one full %s cache rebuild.",
            dataset, dataset,
        )
        for path in _dataset_parquet_cache_paths(dataset, dirs) + _dataset_nonparquet_cache_paths(dataset, dirs):
            _unlink_quietly(path)
        _remove_landmark_outputs(dataset, dirs)
        retry_cfg = Config(**asdict(cfg))
        retry_cfg.overwrite = True
        lm = _build_once(retry_cfg)

    _assert_landmarks_nonempty(dataset, lm, dirs)
    logging.info("%s landmarks ready: %s rows from %s stays", dataset, len(lm), lm['stay_id'].nunique())
    return lm

def build_all_data(cfg: Config, dirs: Dict[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    mimic_lm = build_dataset_data('mimic', cfg, dirs)
    eicu_lm = build_dataset_data('eicu', cfg, dirs)
    save_parquet(pd.concat([mimic_lm, eicu_lm], ignore_index=True, sort=False), dirs['landmarks'] / 'combined_landmarks.parquet')
    return mimic_lm, eicu_lm


def load_or_build_landmarks(cfg: Config, dirs: Dict[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load usable landmarks, or rebuild only the invalid/missing dataset.

    This fixes the failure mode where one database has a nonempty landmark file while the
    other has a zero-row landmark file left by an interrupted or earlier buggy run. The
    function now validates cached landmark row counts and grid files before training.
    """
    landmarks: Dict[str, Optional[pd.DataFrame]] = {'mimic': None, 'eicu': None}

    for dataset in ['mimic', 'eicu']:
        ok, reason = _landmark_file_status(dataset, dirs)
        path = dirs['landmarks'] / f'{dataset}_landmarks.parquet'
        if ok and not cfg.overwrite:
            logging.info("Using cached %s landmarks: %s", dataset, reason)
            landmarks[dataset] = read_parquet(path)
        else:
            logging.warning("%s landmarks need rebuilding: %s", dataset, reason)
            landmarks[dataset] = build_dataset_data(dataset, cfg, dirs, force_landmarks=True)

    mimic_lm = landmarks['mimic']
    eicu_lm = landmarks['eicu']
    _assert_landmarks_nonempty('mimic', mimic_lm, dirs)
    _assert_landmarks_nonempty('eicu', eicu_lm, dirs)
    save_parquet(pd.concat([mimic_lm, eicu_lm], ignore_index=True, sort=False), dirs['landmarks'] / 'combined_landmarks.parquet')
    logging.info("Landmark summary: MIMIC=%s rows/%s stays; eICU=%s rows/%s stays",
                 len(mimic_lm), mimic_lm['stay_id'].nunique(), len(eicu_lm), eicu_lm['stay_id'].nunique())
    return mimic_lm, eicu_lm


def _metric_csv_candidates(dirs: Dict[str, Path]) -> List[Path]:
    metrics_dir = Path(dirs['metrics'])
    if not metrics_dir.exists():
        return []
    paths: List[Path] = []
    for p in sorted(metrics_dir.glob('*.csv')):
        name = p.name
        if name.endswith('_metrics.csv') and not any(x in name for x in ['threshold', 'subgroup', 'baseline_source']):
            paths.append(p)
        elif name.endswith('_rf_shap_model_metrics.csv') or name.endswith('_stay_level_metrics.csv'):
            paths.append(p)
    return sorted(set(paths))


def write_honest_model_selection_report(cfg: Config, dirs: Dict[str, Path], experiment: str = 'unknown') -> Dict[str, Any]:
    """Summarize all candidate results without turning exploratory tuning into a hidden claim."""
    reports = dirs['reports']
    reports.mkdir(parents=True, exist_ok=True)
    rows = []
    for path in _metric_csv_candidates(dirs):
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            logging.warning('Could not read metric candidate %s: %s', path, exc)
            continue
        if df.empty:
            continue
        df = df.copy()
        df['source_file'] = path.name
        df['metric_family'] = 'stay_level' if path.name.endswith('_stay_level_metrics.csv') else ('rf_shap' if path.name.endswith('_rf_shap_model_metrics.csv') else 'landmark_row')
        rows.append(df)
    if rows:
        all_metrics = pd.concat(rows, ignore_index=True, sort=False)
    else:
        all_metrics = pd.DataFrame()
    all_path = reports / 'all_candidate_metrics_for_selection.csv'
    all_metrics.to_csv(all_path, index=False)
    target_h = int(getattr(cfg, 'primary_claim_horizon_hours', getattr(cfg, 'target_operating_horizon_hours', 72)))
    selected: Dict[str, Any] = {}
    flags: List[str] = []
    if all_metrics.empty or 'auc' not in all_metrics.columns:
        status = 'NO_RESULTS_AVAILABLE'
        flags.append('No candidate metric CSVs were available. Run Stage 5 before making performance claims.')
        ranked = pd.DataFrame()
    else:
        m = all_metrics.copy()
        m['auc'] = pd.to_numeric(m['auc'], errors='coerce')
        m['auprc'] = pd.to_numeric(m.get('auprc', np.nan), errors='coerce')
        m['ece'] = pd.to_numeric(m.get('ece', np.nan), errors='coerce')
        m['horizon_hours'] = pd.to_numeric(m.get('horizon_hours', np.nan), errors='coerce')
        external = m.loc[m.get('prefix', pd.Series('', index=m.index)).fillna('').astype(str).str.contains('external', case=False, regex=False)].copy()
        external = external.loc[external['horizon_hours'].eq(target_h)]
        ranked = external.sort_values(['auc', 'auprc', 'ece'], ascending=[False, False, True], na_position='last')
        if ranked.empty:
            status = 'NO_EXTERNAL_RESULT_AT_PRIMARY_HORIZON'
            flags.append(f'No external metric at the prespecified {target_h}h horizon was available.')
        else:
            best = ranked.iloc[0]
            selected = {k: (None if pd.isna(v) else v) for k, v in best.to_dict().items()}
            auc = float(best.get('auc')) if pd.notna(best.get('auc')) else np.nan
            events = pd.to_numeric(pd.Series([best.get('events')]), errors='coerce').iloc[0]
            n = pd.to_numeric(pd.Series([best.get('n')]), errors='coerce').iloc[0]
            ece = pd.to_numeric(pd.Series([best.get('ece')]), errors='coerce').iloc[0]
            if not np.isfinite(auc):
                status = 'NO_VALID_AUROC'
                flags.append('Best-ranked row does not contain a finite AUROC.')
            elif auc >= float(getattr(cfg, 'minimum_external_auc_for_strong_claim', 0.80)):
                status = 'PUBLICATION_GRADE_DISCRIMINATION_CANDIDATE_NOT_DEPLOYMENT_READY'
            else:
                status = 'NO_AUROC_0_80_EXTERNAL_CANDIDATE_FOUND'
                flags.append(f'Best external AUROC at {target_h}h is {auc:.3f}, below the aspirational 0.80 threshold.')
            if np.isfinite(events) and events < int(getattr(cfg, 'minimum_events_for_performance_claim', 25)):
                flags.append(f'Selected row has only {int(events)} events; do not make a strong performance claim.')
            if np.isfinite(n) and n < 100:
                flags.append(f'Selected row has n={int(n)} rows; estimates are unstable.')
            if np.isfinite(ece) and ece > float(getattr(cfg, 'deployment_max_ece', 0.05)):
                flags.append(f'Selected row ECE={float(ece):.3f}; calibration is not deployment-grade.')
            if str(best.get('metric_family', '')) == 'landmark_row':
                flags.append('Selected AUROC is landmark-row based; pair it with stay-level metrics because repeated landmarks are correlated.')
    ranked_path = reports / 'external_metric_ranking_primary_horizon.csv'
    if 'ranked' in locals() and not ranked.empty:
        ranked.to_csv(ranked_path, index=False)
    else:
        pd.DataFrame().to_csv(ranked_path, index=False)
    payload = {
        'pipeline_version': CLINICAL_PIPELINE_VERSION,
        'experiment': experiment,
        'primary_horizon_hours': target_h,
        'selection_metric': getattr(cfg, 'model_selection_metric', 'auc'),
        'status': status,
        'selected_best_external_row': selected,
        'flags': flags,
        'all_candidate_metrics_csv': str(all_path),
        'external_metric_ranking_csv': str(ranked_path),
        'interpretation': 'This report identifies the best observed candidate among prespecified notebook outputs. It is not causal evidence and does not satisfy the clinical deployment gate.',
    }
    with open(reports / 'model_selection_and_claims_report.json', 'w') as f:
        json.dump(payload, f, indent=2, default=str)
    md = [
        '# Model Selection and Claims Report',
        '',
        f'Pipeline version: `{CLINICAL_PIPELINE_VERSION}`',
        f'Experiment setting: `{experiment}`',
        f'Primary horizon: `{target_h}` hours',
        f'Status: **{status}**',
        '',
        '## Selected Best External Row',
    ]
    if selected:
        for key in ['prefix', 'metric_family', 'source_file', 'horizon_hours', 'n', 'events', 'auc', 'auc_ci_low', 'auc_ci_high', 'auprc', 'brier', 'ece', 'calibration_intercept', 'calibration_slope']:
            if key in selected:
                md.append(f'- `{key}`: {selected[key]}')
    else:
        md.append('- No selectable external result yet.')
    md += ['', '## Claim Guardrails']
    md += [f'- {flag}' for flag in flags] if flags else ['- No guardrail flags crossed in the selected row. This still does not imply clinical deployability.']
    md += ['', '## Files', f'- All candidate metrics: `{all_path}`', f'- External ranking: `{ranked_path}`']
    (reports / 'model_selection_and_claims_report.md').write_text('\n'.join(md) + '\n', encoding='utf-8')
    return payload

def run_experiments(cfg: Config, dirs: Dict[str, Path], experiment: str) -> None:
    mimic_lm, eicu_lm = load_or_build_landmarks(cfg, dirs)
    exp_defs = []
    if experiment in ('all', 'mimic_to_eicu'):
        exp_defs.append(('strict_mimic_to_eicu', 'mimic', 'eicu', mimic_lm, eicu_lm, False))
    if experiment in ('all', 'eicu_to_mimic'):
        exp_defs.append(('strict_eicu_to_mimic', 'eicu', 'mimic', eicu_lm, mimic_lm, False))
    if experiment in ('all', 'pooled_mimic_to_eicu'):
        exp_defs.append(('pooled_ssl_mimic_to_eicu', 'mimic', 'eicu', mimic_lm, eicu_lm, True))
    if experiment in ('all', 'pooled_eicu_to_mimic'):
        exp_defs.append(('pooled_ssl_eicu_to_mimic', 'eicu', 'mimic', eicu_lm, mimic_lm, True))
    if not exp_defs:
        raise ValueError(f"No experiment definitions matched RUN_EXPERIMENT={experiment!r}")
    for name, source_name, target_name, source_lm, target_lm, pooled_ssl in exp_defs:
        _assert_landmarks_nonempty(source_name, source_lm, dirs)
        _assert_landmarks_nonempty(target_name, target_lm, dirs)
        train_one_experiment(
            name=name,
            source_name=source_name,
            target_name=target_name,
            source_lm=source_lm,
            target_lm=target_lm,
            cfg=cfg,
            dirs=dirs,
            pooled_ssl=pooled_ssl,
        )
    write_honest_model_selection_report(cfg, dirs, experiment)
    write_clinical_deployment_gate_report(cfg, dirs, experiment)


## 18b. Cache repair and safe rerun behavior

The workflow now validates cached landmark parquet files before training. If a previous interrupted run left an empty `mimic_landmarks.parquet` or `eicu_landmarks.parquet`, the notebook automatically removes that stale file and rebuilds only the affected database instead of reusing the bad cache. This directly addresses the `Landmark datasets are empty` failure mode.

This revision includes MIMIC identifier-normalization cache repair. If a previous run produced `missing_scr_or_uo` for every MIMIC stay, keep `cfg.overwrite=False`; the notebook will remove the incompatible MIMIC item/event/landmark cache and rebuild only the affected artifacts.


## 19. Run configuration

Edit the cell below to change model/training parameters. The data paths stay fixed. This `cfg` and `dirs` are shared by every stage cell that follows.


In [ ]:

# -----------------------
# Notebook run controls
# -----------------------
# Full-study default configuration. Adjust only if you intentionally want a different
# grid size, model size, output directory, or training schedule.
cfg = Config(
    out_dir=str(DRIVE_ROOT / 'aki_ssl_survival_output_clinical_deployment_gated_v4'),
    work_dir=str(WORK_ROOT),
    grid_hours=2,
    history_hours=168,
    horizon_hours=168,
    max_icu_hours=336,
    urine_min_observed_fraction=0.80,
    baseline_admission_fallback_hours=6,
    require_acute_rise_for_scr_4=True,
    bootstrap_repeats=100,
    bootstrap_max_rows=50_000,
    bootstrap_ci_alpha=0.05,
    min_subgroup_n=100,
    risk_thresholds=(0.05, 0.10, 0.20, 0.30),
    clinical_deployment_gate=True,
    target_operating_horizon_hours=72,
    deployment_min_sensitivity=0.70,
    deployment_max_alert_rate=0.25,
    deployment_max_ece=0.05,
    deployment_required_min_n=1000,
    measurement_bias_auc_warn=0.70,
    treatment_ablation_auc_delta_warn=0.03,
    baseline_source_calibration_gap_warn=0.05,
    primary_claim_horizon_hours=72,
    minimum_external_auc_for_strong_claim=0.80,
    minimum_events_for_performance_claim=25,
    model_selection_metric='auc',
    # Conservative chunking prevents eICU long-format vitals/labs from exhausting RAM.
    chunksize=200_000,
    ssl_epochs=8,
    finetune_epochs=12,
    batch_size=128,
    dataloader_workers=0,
    rf_n_estimators=300,
    d_model=128,
    n_layers=2,
    n_heads=4,
    lr=2e-4,
    seed=42,
    deterministic=True,
    overwrite=False,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

dirs = ensure_dirs(cfg)
setup_logging(dirs['out'])
seed_all(cfg.seed, deterministic=cfg.deterministic)
require_files(cfg)

logging.info("Notebook configuration:\n%s", json.dumps(asdict(cfg), indent=2))

# Save exact run metadata for reproducibility.
with open(dirs['out'] / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)
with open(dirs['out'] / 'variables.json', 'w') as f:
    json.dump({
        'matrix_vars': MATRIX_VARS,
        'static_cols': STATIC_COLS,
        'continuous_vars': CONTINUOUS_VARS,
        'sum_vars': SUM_VARS,
        'binary_vars': BINARY_VARS,
    }, f, indent=2)
write_environment_manifest(cfg, dirs)
write_clinical_model_card(cfg, dirs)

print(f"Device: {cfg.device}")
print(f"Output directory: {dirs['out']}")


## 20. Stage 1 - MIMIC extraction (cohort -> item maps -> baseline -> events)

Extract the MIMIC-IV cohort, build item-ID maps, extract baseline creatinine/weight, finalize the cohort, and build the hourly binned events.


In [ ]:
# Stage 1: MIMIC cohort, item maps, baseline, finalized cohort, binned events.
mimic_base = build_mimic_cohort(cfg, dirs)
print(f'MIMIC base cohort: {len(mimic_base):,} stays')

mimic_maps = build_mimic_item_maps(cfg, dirs)
print(f'MIMIC item maps: { {k: len(v) for k, v in mimic_maps.items()} }')

mimic_base_w = extract_mimic_baseline_and_weight(cfg, dirs, mimic_base, mimic_maps)
print(f'Baseline sCr available: {mimic_base_w["baseline_scr"].notna().sum():,} / {len(mimic_base_w):,}')

mimic_cohort = finalize_cohort('mimic', cfg, dirs, mimic_base, mimic_base_w)
print(f'MIMIC final cohort (after baseline filter): {len(mimic_cohort):,}')

mimic_events_path = extract_mimic_events(cfg, dirs, mimic_cohort, mimic_maps)
_ev = read_parquet(mimic_events_path)
print(f'MIMIC binned events: {len(_ev):,} rows, {_ev["stay_id"].nunique():,} stays')
print(f'  variables present: {sorted(_ev["variable"].unique().tolist())}')
_crit = {'creatinine', 'urine_output'}
_found = _crit & set(_ev['variable'].unique())
_miss = _crit - _found
if _miss:
    print(f'  *** WARNING: critical variables MISSING from events: {_miss} ***')
    print(f'  *** The landmark builder will exclude all stays. Check item maps. ***')
else:
    print(f'  creatinine rows: {len(_ev[_ev["variable"]=="creatinine"]):,}')
    print(f'  urine_output rows: {len(_ev[_ev["variable"]=="urine_output"]):,}')
del _ev


## 21. Stage 2 - eICU extraction (cohort -> baseline -> events)

Build the parallel eICU cohort, extract baseline creatinine and all binned events.


In [ ]:
# Stage 2: eICU cohort, baseline, finalized cohort, binned events.
eicu_base = build_eicu_cohort(cfg, dirs)
print(f'eICU base cohort: {len(eicu_base):,} stays')

eicu_baseline = extract_eicu_baseline(cfg, dirs, eicu_base)
print(f'eICU baseline sCr available: {eicu_baseline["baseline_scr"].notna().sum():,} / {len(eicu_baseline):,}')

eicu_cohort = finalize_cohort('eicu', cfg, dirs, eicu_base, eicu_baseline)
print(f'eICU final cohort: {len(eicu_cohort):,}')

eicu_events_path = extract_eicu_events(cfg, dirs, eicu_cohort)
_ev = read_parquet(eicu_events_path)
print(f'eICU binned events: {len(_ev):,} rows, {_ev["stay_id"].nunique():,} stays')
print(f'  variables present: {sorted(_ev["variable"].unique().tolist())}')
_crit = {'creatinine', 'urine_output'}
_miss = _crit - set(_ev['variable'].unique())
if _miss:
    print(f'  *** WARNING: critical variables MISSING: {_miss} ***')
else:
    print(f'  creatinine rows: {len(_ev[_ev["variable"]=="creatinine"]):,}')
    print(f'  urine_output rows: {len(_ev[_ev["variable"]=="urine_output"]):,}')
del _ev


## 22. Stage 3 - Grids and AKI landmarks

Build the per-stay hourly matrices (value + mask + count), compute dynamic KDIGO phenotype timings, and generate the landmark dataset for each database. This is where the exclusion filters run (baseline creatinine availability, urine-output availability, severe AKI/RRT before 6 h).


In [ ]:
# Stage 3: grids + landmarks for both databases.
mimic_lm = build_dataset_data('mimic', cfg, dirs)
print(f'MIMIC landmarks: {len(mimic_lm):,} rows from {mimic_lm["stay_id"].nunique():,} stays')

eicu_lm = build_dataset_data('eicu', cfg, dirs)
print(f'eICU landmarks:  {len(eicu_lm):,} rows from {eicu_lm["stay_id"].nunique():,} stays')

save_parquet(pd.concat([mimic_lm, eicu_lm], ignore_index=True, sort=False),
             dirs['landmarks'] / 'combined_landmarks.parquet')

# Show exclusion breakdown.
for ds in ['mimic', 'eicu']:
    excl_path = dirs['landmarks'] / f'{ds}_exclusions.json'
    if excl_path.exists():
        with open(excl_path) as f:
            print(f'{ds} exclusions: {json.load(f)}')


## 23. Stage 4 - Engineered feature tables (optional inspection)

Build the tabular (non-deep) feature tables used by the random-forest / XGBoost baselines and SHAP analyses.


In [ ]:
# Stage 4: engineered feature tables for optional inspection.
mimic_features = build_feature_table(mimic_lm, cfg, dirs, 'mimic_all')
eicu_features = build_feature_table(eicu_lm, cfg, dirs, 'eicu_all')
print(f'MIMIC feature rows: {len(mimic_features):,} | cols: {mimic_features.shape[1]}')
print(f'eICU feature rows:  {len(eicu_features):,} | cols: {eicu_features.shape[1]}')


## 24. Stage 5 - SSL pretraining + survival fine-tuning + external validation

Run the experiments: MIMIC->eICU and eICU->MIMIC (strict + pooled-SSL variants). Each experiment does self-supervised pretraining, competing-risk survival fine-tuning, internal/external evaluation with calibration, engineered baselines + SHAP, and PSM sensitivity analysis. Metrics are displayed per experiment.

This is the slow stage. To run a single direction, change `RUN_EXPERIMENT` below.


In [ ]:
# -----------------------------------------------------------------------------
# CPU-feasible publication model-selection benchmark
# -----------------------------------------------------------------------------
AUDIT_ID_COLS = {
    'row_id', 'dataset', 'stay_id', 'subject_id', 'hospital_id', 'race_ethnicity',
    'anchor_year_group', 'baseline_source', 'event_type', 'time_to_event_hours', 'time_bin'
}
TREATMENT_DOC_PATTERNS = ['vasopressor', 'diuretic', 'nephrotoxin', 'antibiotic', 'mechanical_ventilation', 'rrt']
MEASUREMENT_PROCESS_PATTERNS = ['measured_', 'obs_count']


def attach_landmark_metadata_to_features(feat: pd.DataFrame, landmarks: pd.DataFrame) -> pd.DataFrame:
    """Attach audit-only metadata from landmark rows while keeping it excluded from features."""
    out = feat.copy()
    lm = landmarks.reset_index(drop=True).copy()
    if 'row_id' in out.columns and out['row_id'].notna().all():
        row_ids = pd.to_numeric(out['row_id'], errors='coerce').astype('Int64')
        valid = row_ids.notna() & (row_ids >= 0) & (row_ids < len(lm))
        if valid.all():
            meta = lm.iloc[row_ids.astype(int).to_numpy()].reset_index(drop=True)
        else:
            meta = lm.iloc[:len(out)].reset_index(drop=True)
    else:
        out['row_id'] = np.arange(len(out), dtype=int)
        meta = lm.iloc[:len(out)].reset_index(drop=True)
    for col in ['dataset', 'stay_id', 'subject_id', 'hospital_id', 'race_ethnicity', 'anchor_year_group', 'baseline_source', 'landmark_hour', 'event_type', 'time_to_event_hours', 'time_bin']:
        if col in meta.columns:
            out[col] = meta[col].values
    return out


def benchmark_feature_columns(feat: pd.DataFrame, variant: str) -> List[str]:
    cols = [c for c in feat.columns if c not in AUDIT_ID_COLS and pd.api.types.is_numeric_dtype(feat[c])]
    if variant in {'no_treatment', 'conservative'}:
        cols = [c for c in cols if not any(p in c for p in TREATMENT_DOC_PATTERNS)]
    if variant in {'no_measurement', 'conservative'}:
        cols = [c for c in cols if not any(p in c for p in MEASUREMENT_PROCESS_PATTERNS)]
    return cols


def horizon_label(feat: pd.DataFrame, horizon: int) -> np.ndarray:
    return ((feat['event_type'].values == 1) & (feat['time_to_event_hours'].values <= horizon)).astype(int)


def make_balanced_sample_weight(y: np.ndarray) -> np.ndarray:
    if len(np.unique(y)) < 2:
        return np.ones(len(y), dtype=float)
    return compute_sample_weight(class_weight='balanced', y=y).astype(float)


def fit_hgb_with_source_calibration(train_df: pd.DataFrame, val_df: pd.DataFrame, cols: List[str], horizon: int, cfg: Config):
    y_train = horizon_label(train_df, horizon)
    y_val = horizon_label(val_df, horizon)
    if len(cols) == 0 or len(np.unique(y_train)) < 2:
        return None, None
    clf = HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=0.045,
        max_iter=240,
        max_leaf_nodes=31,
        min_samples_leaf=35,
        l2_regularization=0.02,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=20,
        random_state=int(cfg.seed) + int(horizon),
    )
    clf.fit(train_df[cols].replace([np.inf, -np.inf], np.nan), y_train, sample_weight=make_balanced_sample_weight(y_train))
    calibrator = None
    if len(val_df) and len(np.unique(y_val)) == 2:
        raw_val = clf.predict_proba(val_df[cols].replace([np.inf, -np.inf], np.nan))[:, 1]
        raw_val = np.clip(raw_val, 1e-6, 1 - 1e-6)
        try:
            calibrator = LogisticRegression(solver='lbfgs', max_iter=1000)
            logit_val = np.log(raw_val / (1 - raw_val)).reshape(-1, 1)
            calibrator.fit(logit_val, y_val)
        except Exception as exc:
            logging.warning('Calibration failed at %sh: %s', horizon, exc)
            calibrator = None
    return clf, calibrator


def predict_calibrated_hgb(model, calibrator, df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    raw = model.predict_proba(df[cols].replace([np.inf, -np.inf], np.nan))[:, 1]
    raw = np.clip(raw, 1e-6, 1 - 1e-6)
    if calibrator is None:
        return raw
    logit_raw = np.log(raw / (1 - raw)).reshape(-1, 1)
    return np.clip(calibrator.predict_proba(logit_raw)[:, 1], 1e-6, 1 - 1e-6)


def _write_binary_thresholds(meta: pd.DataFrame, risk_by_horizon: Dict[int, np.ndarray], cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    rows = []
    for h, p_all in risk_by_horizon.items():
        y_all = horizon_label(meta, h)
        for threshold in getattr(cfg, 'risk_thresholds', (0.05, 0.10, 0.20, 0.30)):
            pred = p_all >= float(threshold)
            tp = int(np.sum(pred & (y_all == 1)))
            fp = int(np.sum(pred & (y_all == 0)))
            tn = int(np.sum((~pred) & (y_all == 0)))
            fn = int(np.sum((~pred) & (y_all == 1)))
            rows.append({'prefix': prefix, 'horizon_hours': h, 'threshold': float(threshold), 'n': int(len(y_all)), 'events': int(y_all.sum()), 'alerts': int(pred.sum()), 'alert_rate': float(pred.mean()) if len(pred) else np.nan, 'sensitivity': float(tp / max(tp + fn, 1)), 'specificity': float(tn / max(tn + fp, 1)), 'ppv': float(tp / max(tp + fp, 1)), 'npv': float(tn / max(tn + fn, 1)), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn})
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_threshold_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_threshold_metrics.csv', index=False)
    return out


def _write_binary_subgroups(meta: pd.DataFrame, risk_by_horizon: Dict[int, np.ndarray], cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    groupers: Dict[str, pd.Series] = {}
    if 'sex_female' in meta.columns:
        sex = pd.to_numeric(meta['sex_female'], errors='coerce')
        groupers['sex'] = pd.Series(np.where(sex >= 0.5, 'female', 'male'), index=meta.index)
    if 'age' in meta.columns:
        age = pd.to_numeric(meta['age'], errors='coerce')
        groupers['age_band'] = pd.cut(age, bins=[0, 65, 80, 200], labels=['<65', '65-79', '>=80'], right=False).astype('object')
    for col in ['race_ethnicity', 'baseline_source']:
        if col in meta.columns:
            groupers[col] = meta[col].fillna('missing').astype(str)
    for col in ['ckd', 'diabetes', 'chf', 'liver_disease', 'sepsis_dx', 'medical_icu', 'surgical_icu', 'baseline_source_prior', 'baseline_source_admission_fallback']:
        if col in meta.columns:
            v = pd.to_numeric(meta[col], errors='coerce').fillna(0)
            groupers[col] = pd.Series(np.where(v >= 0.5, 'yes', 'no'), index=meta.index)
    rows = []
    min_n = int(getattr(cfg, 'min_subgroup_n', 100))
    for h, p_all in risk_by_horizon.items():
        y_all = horizon_label(meta, h)
        for group_name, values in groupers.items():
            vals = values.astype('object').where(values.notna(), 'missing').astype(str).values
            for level in sorted(set(vals)):
                m = vals == level
                if int(m.sum()) < min_n:
                    continue
                y = y_all[m]
                p = p_all[m]
                row = {'prefix': prefix, 'horizon_hours': h, 'subgroup': group_name, 'level': level, 'n': int(len(y)), 'events': int(y.sum())}
                row.update(_safe_binary_metrics(y, p))
                intercept, slope = calibration_intercept_slope(y, p)
                row['calibration_intercept'] = intercept
                row['calibration_slope'] = slope
                rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_subgroup_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_subgroup_metrics.csv', index=False)
    return out


def _write_binary_stay_level(meta: pd.DataFrame, risk_by_horizon: Dict[int, np.ndarray], cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    if 'stay_id' not in meta.columns:
        return pd.DataFrame()
    order = np.argsort(pd.to_numeric(meta.get('landmark_hour', pd.Series(np.arange(len(meta)))), errors='coerce').fillna(np.inf).to_numpy())
    m = meta.iloc[order].reset_index(drop=True)
    first_idx = m.groupby('stay_id', sort=False).head(1).index.to_numpy()
    rows = []
    for h, p_all in risk_by_horizon.items():
        p_sorted = p_all[order]
        p = p_sorted[first_idx]
        y = horizon_label(m.iloc[first_idx].reset_index(drop=True), h)
        row = {'prefix': prefix, 'horizon_hours': h, 'aggregation': 'first_eligible_landmark_per_stay', 'n': int(len(y)), 'events': int(y.sum())}
        row.update(_safe_binary_metrics(y, p))
        intercept, slope = calibration_intercept_slope(y, p)
        row['calibration_intercept'] = intercept
        row['calibration_slope'] = slope
        rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        save_parquet(out, dirs['metrics'] / f'{prefix}_stay_level_metrics.parquet')
        out.to_csv(dirs['metrics'] / f'{prefix}_stay_level_metrics.csv', index=False)
    return out


def evaluate_binary_feature_model(meta: pd.DataFrame, risk_by_horizon: Dict[int, np.ndarray], cfg: Config, prefix: str, dirs: Dict[str, Path]) -> pd.DataFrame:
    fig_dir = dirs['figures'] / prefix
    fig_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for h, p in risk_by_horizon.items():
        y = horizon_label(meta, h)
        row = {'prefix': prefix, 'horizon_hours': int(h), 'n': int(len(y)), 'events': int(y.sum())}
        row.update(_safe_binary_metrics(y, p))
        intercept, slope = calibration_intercept_slope(y, p)
        row['calibration_intercept'] = intercept
        row['calibration_slope'] = slope
        cluster_col = 'stay_id' if 'stay_id' in meta.columns else ('hospital_id' if 'hospital_id' in meta.columns else None)
        if cluster_col is not None:
            row['bootstrap_cluster_col'] = cluster_col
            row.update(cluster_bootstrap_metric_cis(y, p, meta[cluster_col].values, cfg))
        rows.append(row)
        plot_calibration(y, p, fig_dir / f'calibration_{h}h.png', title=f'{prefix} calibration {h}h')
        plot_dca(y, p, fig_dir / f'dca_{h}h.png', title=f'{prefix} DCA {h}h')
    out = pd.DataFrame(rows)
    save_parquet(out, dirs['metrics'] / f'{prefix}_metrics.parquet')
    out.to_csv(dirs['metrics'] / f'{prefix}_metrics.csv', index=False)
    _write_binary_thresholds(meta, risk_by_horizon, cfg, prefix, dirs)
    _write_binary_subgroups(meta, risk_by_horizon, cfg, prefix, dirs)
    _write_binary_stay_level(meta, risk_by_horizon, cfg, prefix, dirs)
    return out


def run_cpu_publication_direction(name: str, source_name: str, target_name: str, source_lm: pd.DataFrame, target_lm: pd.DataFrame, source_feat: pd.DataFrame, target_feat: pd.DataFrame, cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    source_feat = attach_landmark_metadata_to_features(source_feat, source_lm)
    target_feat = attach_landmark_metadata_to_features(target_feat, target_lm)
    splits = split_landmarks(source_lm, source_name, cfg)
    validate_split_integrity(source_lm, splits, source_name, cfg)
    train_df = source_feat.iloc[splits['train']].reset_index(drop=True)
    val_df = source_feat.iloc[splits['val']].reset_index(drop=True)
    test_df = source_feat.iloc[splits['test']].reset_index(drop=True)
    external_df = target_feat.reset_index(drop=True)
    eval_splits = [('internal_test', test_df), ('external', external_df)]
    variants = {
        'full': 'all engineered features',
        'no_treatment': 'treatment/documentation features removed',
        'no_measurement': 'measurement-process features removed',
        'conservative': 'treatment/documentation and measurement-process features removed',
    }
    candidate_rows = []
    for variant, description in variants.items():
        cols = benchmark_feature_columns(train_df, variant)
        fitted: Dict[int, Tuple[Any, Any]] = {}
        for h in cfg.horizons:
            fitted[int(h)] = fit_hgb_with_source_calibration(train_df, val_df, cols, int(h), cfg)
        for split_name, eval_df in eval_splits:
            risk_by_horizon: Dict[int, np.ndarray] = {}
            for h, (model, calibrator) in fitted.items():
                if model is None:
                    risk_by_horizon[int(h)] = np.full(len(eval_df), np.nan, dtype=float)
                else:
                    risk_by_horizon[int(h)] = predict_calibrated_hgb(model, calibrator, eval_df, cols)
            prefix = f'{name}_{variant}_{split_name}'
            pred = eval_df[[c for c in ['row_id', 'dataset', 'stay_id', 'subject_id', 'hospital_id', 'race_ethnicity', 'anchor_year_group', 'baseline_source', 'landmark_hour', 'event_type', 'time_to_event_hours', 'time_bin'] if c in eval_df.columns]].copy()
            for h, p in risk_by_horizon.items():
                pred[f'risk_{h}h'] = p
            save_parquet(pred, dirs['predictions'] / f'{prefix}_predictions.parquet')
            met = evaluate_binary_feature_model(eval_df, risk_by_horizon, cfg, prefix, dirs)
            met['experiment'] = name
            met['model_family'] = 'hist_gradient_boosting_binary_landmark'
            met['feature_variant'] = variant
            met['feature_variant_description'] = description
            met['split_name'] = split_name
            met['n_features'] = int(len(cols))
            met['strict_external'] = bool(split_name == 'external')
            candidate_rows.append(met)
        del fitted
        gc.collect()
    out = pd.concat(candidate_rows, ignore_index=True, sort=False) if candidate_rows else pd.DataFrame()
    if not out.empty:
        out.to_csv(dirs['metrics'] / f'{name}_cpu_publication_candidate_metrics.csv', index=False)
    return out



def _load_xgb_classifier():
    try:
        from xgboost import XGBClassifier
        return XGBClassifier
    except Exception as exc:
        logging.warning('XGBoost is unavailable; skipping XGB benchmark: %s', exc)
        return None


def fit_xgb_with_source_calibration(train_df: pd.DataFrame, val_df: pd.DataFrame, cols: List[str], horizon: int, cfg: Config):
    XGBClassifier = _load_xgb_classifier()
    y_train = horizon_label(train_df, horizon)
    y_val = horizon_label(val_df, horizon)
    if XGBClassifier is None or len(cols) == 0 or len(np.unique(y_train)) < 2:
        return None, None
    clf = XGBClassifier(
        n_estimators=420,
        max_depth=4,
        learning_rate=0.045,
        subsample=0.85,
        colsample_bytree=0.80,
        min_child_weight=20,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device='cpu',
        n_jobs=max(1, int(getattr(cfg, 'n_jobs', 2)) - 1),
        random_state=int(cfg.seed) + 700 + int(horizon),
    )
    clf.fit(train_df[cols].replace([np.inf, -np.inf], np.nan), y_train, sample_weight=make_balanced_sample_weight(y_train), verbose=False)
    calibrator = None
    if len(val_df) and len(np.unique(y_val)) == 2:
        raw_val = clf.predict_proba(val_df[cols].replace([np.inf, -np.inf], np.nan))[:, 1]
        raw_val = np.clip(raw_val, 1e-6, 1 - 1e-6)
        try:
            calibrator = LogisticRegression(solver='lbfgs', max_iter=1000)
            logit_val = np.log(raw_val / (1 - raw_val)).reshape(-1, 1)
            calibrator.fit(logit_val, y_val)
        except Exception as exc:
            logging.warning('XGB calibration failed at %sh: %s', horizon, exc)
            calibrator = None
    return clf, calibrator


def predict_calibrated_xgb(model, calibrator, df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    raw = model.predict_proba(df[cols].replace([np.inf, -np.inf], np.nan))[:, 1]
    raw = np.clip(raw, 1e-6, 1 - 1e-6)
    if calibrator is None:
        return raw
    logit_raw = np.log(raw / (1 - raw)).reshape(-1, 1)
    return np.clip(calibrator.predict_proba(logit_raw)[:, 1], 1e-6, 1 - 1e-6)


def run_cpu_xgb_direction(name: str, source_name: str, target_name: str, source_lm: pd.DataFrame, target_lm: pd.DataFrame, source_feat: pd.DataFrame, target_feat: pd.DataFrame, cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    source_feat = attach_landmark_metadata_to_features(source_feat, source_lm)
    target_feat = attach_landmark_metadata_to_features(target_feat, target_lm)
    splits = split_landmarks(source_lm, source_name, cfg)
    validate_split_integrity(source_lm, splits, source_name, cfg)
    train_df = source_feat.iloc[splits['train']].reset_index(drop=True)
    val_df = source_feat.iloc[splits['val']].reset_index(drop=True)
    test_df = source_feat.iloc[splits['test']].reset_index(drop=True)
    external_df = target_feat.reset_index(drop=True)
    variants = {
        'full': 'all engineered features',
        'no_treatment': 'treatment/documentation features removed',
        'conservative': 'treatment/documentation and measurement-process features removed',
    }
    candidate_rows = []
    for variant, description in variants.items():
        cols = benchmark_feature_columns(train_df, variant)
        fitted: Dict[int, Tuple[Any, Any]] = {}
        for h in cfg.horizons:
            fitted[int(h)] = fit_xgb_with_source_calibration(train_df, val_df, cols, int(h), cfg)
        for split_name, eval_df in [('internal_test', test_df), ('external', external_df)]:
            risk_by_horizon: Dict[int, np.ndarray] = {}
            for h, (model, calibrator) in fitted.items():
                if model is None:
                    risk_by_horizon[int(h)] = np.full(len(eval_df), np.nan, dtype=float)
                else:
                    risk_by_horizon[int(h)] = predict_calibrated_xgb(model, calibrator, eval_df, cols)
            prefix = f'{name}_{variant}_{split_name}'
            pred = eval_df[[c for c in ['row_id', 'dataset', 'stay_id', 'subject_id', 'hospital_id', 'race_ethnicity', 'anchor_year_group', 'baseline_source', 'landmark_hour', 'event_type', 'time_to_event_hours', 'time_bin'] if c in eval_df.columns]].copy()
            for h, p in risk_by_horizon.items():
                pred[f'risk_{h}h'] = p
            save_parquet(pred, dirs['predictions'] / f'{prefix}_predictions.parquet')
            met = evaluate_binary_feature_model(eval_df, risk_by_horizon, cfg, prefix, dirs)
            met['experiment'] = name
            met['model_family'] = 'xgboost_binary_landmark'
            met['feature_variant'] = variant
            met['feature_variant_description'] = description
            met['split_name'] = split_name
            met['n_features'] = int(len(cols))
            met['strict_external'] = bool(split_name == 'external')
            candidate_rows.append(met)
        del fitted
        gc.collect()
    out = pd.concat(candidate_rows, ignore_index=True, sort=False) if candidate_rows else pd.DataFrame()
    if not out.empty:
        out.to_csv(dirs['metrics'] / f'{name}_xgb_publication_candidate_metrics.csv', index=False)
    return out


def run_cpu_xgb_publication_benchmark(mimic_lm: pd.DataFrame, eicu_lm: pd.DataFrame, mimic_features: pd.DataFrame, eicu_features: pd.DataFrame, cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    logging.info('Running CPU XGBoost strict external publication benchmark.')
    rows = []
    rows.append(run_cpu_xgb_direction('cpu_xgb_mimic_to_eicu', 'mimic', 'eicu', mimic_lm, eicu_lm, mimic_features, eicu_features, cfg, dirs))
    rows.append(run_cpu_xgb_direction('cpu_xgb_eicu_to_mimic', 'eicu', 'mimic', eicu_lm, mimic_lm, eicu_features, mimic_features, cfg, dirs))
    out = pd.concat([r for r in rows if r is not None and not r.empty], ignore_index=True, sort=False) if rows else pd.DataFrame()
    if not out.empty:
        out.to_csv(dirs['metrics'] / 'cpu_xgb_publication_all_candidate_metrics.csv', index=False)
    try:
        write_honest_model_selection_report(cfg, dirs)
        write_clinical_deployment_gate_report(cfg, dirs)
    except Exception as exc:
        logging.warning('Could not write final reports after XGB benchmark: %s', exc)
    return out

def run_cpu_publication_benchmark(mimic_lm: pd.DataFrame, eicu_lm: pd.DataFrame, mimic_features: pd.DataFrame, eicu_features: pd.DataFrame, cfg: Config, dirs: Dict[str, Path]) -> pd.DataFrame:
    logging.info('Running CPU-feasible strict external publication benchmark.')
    rows = []
    rows.append(run_cpu_publication_direction('cpu_hgb_mimic_to_eicu', 'mimic', 'eicu', mimic_lm, eicu_lm, mimic_features, eicu_features, cfg, dirs))
    rows.append(run_cpu_publication_direction('cpu_hgb_eicu_to_mimic', 'eicu', 'mimic', eicu_lm, mimic_lm, eicu_features, mimic_features, cfg, dirs))
    out = pd.concat([r for r in rows if r is not None and not r.empty], ignore_index=True, sort=False) if rows else pd.DataFrame()
    if not out.empty:
        out.to_csv(dirs['metrics'] / 'cpu_publication_all_candidate_metrics.csv', index=False)
    try:
        write_honest_model_selection_report(cfg, dirs)
        write_clinical_deployment_gate_report(cfg, dirs)
    except Exception as exc:
        logging.warning('Could not write final reports after CPU benchmark: %s', exc)
    return out


In [ ]:
# Stage 5: run all (or selected) experiments.
# Default is CPU-feasible and publication-oriented because this local machine has no CUDA GPU.
# Set AKI_RUN_EXPERIMENT=reports_only to refresh reports without retraining.
# Set AKI_RUN_EXPERIMENT=all only when you intentionally want the full SSL/survival-transformer run.
RUN_EXPERIMENT = os.environ.get('AKI_RUN_EXPERIMENT', 'cpu_publication_benchmark')

if RUN_EXPERIMENT == 'reports_only':
    write_honest_model_selection_report(cfg, dirs, RUN_EXPERIMENT)
    write_clinical_deployment_gate_report(cfg, dirs, RUN_EXPERIMENT)
elif RUN_EXPERIMENT == 'cpu_publication_benchmark':
    cpu_publication_results = run_cpu_publication_benchmark(mimic_lm, eicu_lm, mimic_features, eicu_features, cfg, dirs)
elif RUN_EXPERIMENT == 'cpu_xgb_publication_benchmark':
    cpu_xgb_publication_results = run_cpu_xgb_publication_benchmark(mimic_lm, eicu_lm, mimic_features, eicu_features, cfg, dirs)
elif RUN_EXPERIMENT in {'all', 'mimic_to_eicu', 'eicu_to_mimic', 'pooled_mimic_to_eicu', 'pooled_eicu_to_mimic'}:
    run_experiments(cfg, dirs, RUN_EXPERIMENT)
else:
    raise ValueError(f'Unknown AKI_RUN_EXPERIMENT={RUN_EXPERIMENT!r}')
print(f'Experiment stage complete: {RUN_EXPERIMENT}')

## 25. Inspect generated outputs

List the generated artifacts per subdirectory and preview available metric CSV files.


In [ ]:
# Inspect outputs.
from pathlib import Path

out = Path(cfg.out_dir)
for subdir in ['cohort', 'events', 'grids', 'landmarks', 'models', 'predictions', 'metrics', 'figures', 'features', 'psm', 'reports']:
    p = out / subdir
    if p.exists():
        n_files = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'{subdir:12s}  {n_files:6d} files  -> {p}')
    else:
        print(f'{subdir:12s}  missing -> {p}')

metrics_dir = out / 'metrics'
if metrics_dir.exists():
    csvs = sorted(metrics_dir.glob('*.csv'))
    print(f'\nMetric CSV files ({len(csvs)}):')
    for csv_path in csvs:
        print(' -', csv_path.name)
    if csvs:
        display(pd.read_csv(csvs[0]).head(15))

report_path = out / 'reports' / 'clinical_deployment_gate_report.md'
if report_path.exists():
    print('\nClinical deployment gate report:')
    print(report_path.read_text()[:2000])

selection_report_path = out / 'reports' / 'model_selection_and_claims_report.md'
if selection_report_path.exists():
    print('\nModel selection and claims report:')
    print(selection_report_path.read_text()[:2000])


## 26. Publication figures

Build polished summary figures after the experiment cells have produced metrics. This cell is intentionally no-op safe: if metrics do not exist yet, it prints what is missing instead of fabricating figures.

In [ ]:
def build_publication_figures(cfg: Config, dirs: Dict[str, Path]) -> None:
    pub_dir = dirs['figures'] / 'publication'
    pub_dir.mkdir(parents=True, exist_ok=True)
    metric_files = sorted(dirs['metrics'].glob('*_metrics.csv'))
    threshold_files = sorted(dirs['metrics'].glob('*_threshold_metrics.csv'))
    subgroup_files = sorted(dirs['metrics'].glob('*_subgroup_metrics.csv'))
    if not metric_files:
        print('No metric CSV files found yet. Run Stage 5 before building publication figures.')
        return
    metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True, sort=False)
    try:
        plt.style.use('seaborn-v0_8-whitegrid')
    except Exception:
        plt.style.use('default')
    # AUROC by horizon and experiment, with bootstrap CIs when available.
    fig, ax = plt.subplots(figsize=(10, 5.8))
    plot_df = metrics.loc[metrics['auc'].notna()].copy()
    if not plot_df.empty:
        prefixes = plot_df['prefix'].drop_duplicates().tolist()
        colors = plt.cm.tab20(np.linspace(0, 1, max(len(prefixes), 1)))
        for color, prefix in zip(colors, prefixes):
            d = plot_df.loc[plot_df['prefix'].eq(prefix)].sort_values('horizon_hours')
            ax.plot(d['horizon_hours'], d['auc'], marker='o', linewidth=2, label=prefix, color=color)
            if {'auc_ci_low', 'auc_ci_high'}.issubset(d.columns):
                lo = pd.to_numeric(d['auc_ci_low'], errors='coerce')
                hi = pd.to_numeric(d['auc_ci_high'], errors='coerce')
                if lo.notna().any() and hi.notna().any():
                    ax.fill_between(d['horizon_hours'], lo, hi, alpha=0.12, color=color)
        ax.axhline(0.80, color='black', linestyle='--', linewidth=1, label='AUROC 0.80 reference')
        ax.set_xlabel('Prediction horizon (hours)')
        ax.set_ylabel('AUROC')
        ax.set_ylim(0.45, 1.0)
        ax.set_title('External and internal discrimination by horizon')
        ax.legend(fontsize=7, ncol=2, frameon=True)
        fig.tight_layout()
        fig.savefig(pub_dir / 'figure_1_auroc_by_horizon.png', dpi=300, bbox_inches='tight')
        fig.savefig(pub_dir / 'figure_1_auroc_by_horizon.pdf', bbox_inches='tight')
    plt.close(fig)
    # Calibration panel.
    cal_cols = {'calibration_intercept', 'calibration_slope', 'ece'}
    if cal_cols.issubset(metrics.columns):
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
        for ax, col, title in zip(axes, ['calibration_intercept', 'calibration_slope', 'ece'], ['Calibration intercept', 'Calibration slope', 'Expected calibration error']):
            d = metrics.loc[metrics[col].notna()].copy()
            for prefix in d['prefix'].drop_duplicates().tolist():
                dd = d.loc[d['prefix'].eq(prefix)].sort_values('horizon_hours')
                ax.plot(dd['horizon_hours'], dd[col], marker='o', linewidth=1.8, label=prefix)
            if col == 'calibration_intercept':
                ax.axhline(0, color='black', linestyle='--', linewidth=1)
            if col == 'calibration_slope':
                ax.axhline(1, color='black', linestyle='--', linewidth=1)
            ax.set_title(title)
            ax.set_xlabel('Horizon (hours)')
        axes[0].set_ylabel('Value')
        axes[-1].legend(fontsize=6, loc='best')
        fig.tight_layout()
        fig.savefig(pub_dir / 'figure_2_calibration_summary.png', dpi=300, bbox_inches='tight')
        fig.savefig(pub_dir / 'figure_2_calibration_summary.pdf', bbox_inches='tight')
        plt.close(fig)
    if threshold_files:
        th = pd.concat([pd.read_csv(p) for p in threshold_files], ignore_index=True, sort=False)
        target_h = int(getattr(cfg, 'target_operating_horizon_hours', 72))
        d = th.loc[pd.to_numeric(th['horizon_hours'], errors='coerce').eq(target_h)].copy()
        if not d.empty:
            fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True)
            for ax, col, title in zip(axes, ['sensitivity', 'ppv', 'alert_rate'], ['Sensitivity', 'Positive predictive value', 'Alert rate']):
                for prefix in d['prefix'].drop_duplicates().tolist():
                    dd = d.loc[d['prefix'].eq(prefix)].sort_values('threshold')
                    ax.plot(dd['threshold'], dd[col], marker='o', linewidth=1.8, label=prefix)
                ax.set_title(title)
                ax.set_xlabel('Risk threshold')
                ax.set_ylim(0, 1)
            axes[0].set_ylabel('Metric')
            axes[-1].legend(fontsize=6, loc='best')
            fig.suptitle(f'Clinical operating characteristics at {target_h}h horizon', y=1.02)
            fig.tight_layout()
            fig.savefig(pub_dir / 'figure_3_threshold_operating_characteristics.png', dpi=300, bbox_inches='tight')
            fig.savefig(pub_dir / 'figure_3_threshold_operating_characteristics.pdf', bbox_inches='tight')
            plt.close(fig)
    manifest = {
        'created': time.strftime('%Y-%m-%d %H:%M:%S'),
        'metric_files': [str(p) for p in metric_files],
        'threshold_files': [str(p) for p in threshold_files],
        'subgroup_files': [str(p) for p in subgroup_files],
        'publication_figure_dir': str(pub_dir),
    }
    with open(pub_dir / 'publication_figure_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Publication figures written to {pub_dir}')

build_publication_figures(cfg, dirs)
